# EEG Alignment and Faithfulness in Decoding

Standalone research notebook for frozen **Flan-T5-large** and **BART-large** backbones. It imports no project-local code and never loads a probe checkpoint.

1. **Independent EEG training:** the SemKey Q-Merger maps raw EEG `(B, 1280, 128)` to `(B, 96, 128)`, then a learned projection maps to LLM width. Only this EEG path is trained; LLM weights remain frozen.
2. **Objective:** the original SemKey-style symmetric contrastive loss and normalized token MSE are retained. Commitment MSE is reduced only over valid text-token elements. Train and early-stopping validation use the same teacher-forced composite `0.5 CLIP + 0.5 AR + 0.7 commitment`; full-target greedy rollout CE is display-only.
3. **Controlled memories:**
   - `oracle_text` — analysis-only, frozen full-target text encoder states with their real attention mask (an oracle full-target memory)
   - `eeg` — independently trained EEG memory
   Both paths use exactly the same instruction-only decoder prefix.
4. **Canonical decoder order:** BART uses `decoder_start → BOS → instruction → lexical content → EOS`; T5 uses `decoder_start → instruction → lexical content → EOS`. The BART BOS is part of the masked prompt.
5. **Analyses:** intermediate streams use a verified tuned-lens Unembed, while every final-layer result uses official Hugging Face logits from that forward pass. Test statistics are UID-aggregated, paired/permutation-based, and bootstrap-reported. Linear probes use train-only standardization and report chance, majority-token, and shuffled-label diagnostics.

Outputs remain figures and checkpoints under `outputs/eeg_faith/{llm_key}/`; metric tables are displayed, not written as CSV/JSON. `seed=2026`.


In [ ]:
# =============================================================
# Cell 1: Imports + HF mirror + CONFIG
# =============================================================
from __future__ import annotations

import os
import time
import gc
import math
import pickle
import random
import shutil
import hashlib
import json
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from pickle import UnpicklingError
from typing import Any, Dict, Iterator, List, Literal, Optional, Sequence, Set, Tuple, Union

if not os.environ.get('HF_ENDPOINT'):
    os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print(f'HF_ENDPOINT: {os.environ.get("HF_ENDPOINT")}')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, Dataset, Sampler, TensorDataset
import matplotlib.pyplot as plt
import scipy.linalg
from scipy import stats
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA

COLOR_PRIMARY = '#0173B2'
COLOR_SECONDARY = '#DE8F05'
COLOR_ACC = '#029E73'
COLOR_PURPLE = '#CC78BC'
PLOT_DPI = 160


def apply_plot_style() -> None:
    """Apply the shared matplotlib style used by all EEG-faith figures."""
    plt.rcParams.update({
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.grid': True,
        'grid.alpha': 0.25,
        'grid.linestyle': '--',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'font.size': 11,
        'axes.titlesize': 13,
        'axes.labelsize': 11,
        'legend.frameon': False,
        'lines.linewidth': 2.0,
        'lines.markersize': 6,
    })

from transformers import (
    AutoTokenizer,
    BartForConditionalGeneration,
    BartTokenizer,
    T5ForConditionalGeneration,
)
from transformers.modeling_outputs import BaseModelOutput

try:
    from tuned_lens import TunedLens
    from tuned_lens.nn.lenses import TunedLensConfig
    from tuned_lens.nn.unembed import Unembed
    import tuned_lens.model_surgery as tuned_lens_model_surgery
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tuned-lens'])
    from tuned_lens import TunedLens
    from tuned_lens.nn.lenses import TunedLensConfig
    from tuned_lens.nn.unembed import Unembed
    import tuned_lens.model_surgery as tuned_lens_model_surgery


def _allow_torch_load_for_trusted_hf() -> None:
    """Disable the transformers torch<2.6 load gate for trusted Hub checkpoints.

    Some transformers versions refuse ``torch.load`` unless torch>=2.6. This
    notebook only loads public Flan-T5 / BART weights, so the check is skipped.
    """
    try:
        from transformers.utils import import_utils as _iu
        if hasattr(_iu, 'check_torch_load_is_safe'):
            _iu.check_torch_load_is_safe = lambda *a, **k: None  # type: ignore[assignment]
    except Exception:
        pass


_allow_torch_load_for_trusted_hf()


def _first_existing(*paths: str) -> str:
    """Return the first path that exists on disk, else the first candidate.

    Args:
        *paths: Candidate filesystem paths, tried in order.

    Returns:
        The first existing path, or ``paths[0]`` when none exist.
    """
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]


def _configure_hf_cache() -> Optional[str]:
    """Create and select a writable Hugging Face cache directory.

    Tries AutoDL tmp paths first, then ``./models``. Sets ``HF_HOME`` and the
    Hub cache env vars on the first directory that can be created.

    Returns:
        Absolute cache path, or None if every candidate failed.
    """
    candidates = [
        './autodl-tmp/models',
        '/root/autodl-tmp/models',
        './models',
    ]
    for c in candidates:
        try:
            resolved = Path(c).resolve()
            resolved.mkdir(parents=True, exist_ok=True)
            os.environ.setdefault('HF_HOME', str(resolved))
            os.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(resolved))
            os.environ.setdefault('HF_HUB_CACHE', str(resolved))
            return str(resolved)
        except Exception:
            continue
    return None


HF_CACHE = _configure_hf_cache()

CONFIG: Dict[str, Any] = {
    'data_path': _first_existing(
        './autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
        '/root/autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
        './preprocessed_data/zuco_merged_whiten_norm.df',
    ),
    'output_dir': './outputs/eeg_faith',
    'model_cache_dir': HF_CACHE,
    'llm_load_max_retries': 5,
    'llm_load_retry_backoff_s': 30,
    'in_len': 1280,
    'in_dim': 128,
    'use_channel_weights': True,
    'out_len': 96,
    'hidden_dim': 128,
    'prompt_dim': 128,
    'n_in_blocks': 6,
    'n_out_blocks': 6,
    'num_heads': 8,
    'mlp_ratio': 4,
    'dropout': 0.1,
    'prompt_drop_probs': (0.0, 0.0, 0.0),
    'use_ei': True,
    'max_target_tokens': 64,
    'input_text_len': 96,
    'w_clip': 0.5,
    'commitment_weight': 0.7,
    'w_ar': 0.5,
    'epochs': 100,
    'lr': 1e-4,
    'lr_restart_epochs': 15,
    'min_lr': 1e-9,
    'weight_decay': 0.05,
    'grad_clip': 3.0,
    'patience': 10,
    'batch_size': 72,
    'num_workers': 0,
    'seed': 2026,
    'run_smoke_only': False,
    'data_keys': ['eeg'],
    'decoder_prompt': 'Based on the following signals, translate the sentence',
    'llm_keys_to_run': ['flan_t5_large', 'bart_large'],
    'bootstrap_samples': 1000,
    'permutation_samples': 10000,
    'derangement_trials': 256,
    'frechet_dim': 64,
    'frechet_bootstrap_samples': 100,
    'probe_every': 3,
    'probe_epochs': 100,
    'probe_patience': 10,
    'probe_lr': 1e-3,
    'probe_batch_size': 256,
    'probe_grad_clip': 1.0,
    'probe_ce_margin': 5.0,
    'qualitative_n': 5,
    'lens_epochs': 100,
    'lens_patience': 10,
    'lens_lr': 1e-3,
}

LLM_CONFIGS: List[Dict[str, Any]] = [
    {
        'key': 'flan_t5_large',
        'repo_id': 'google/flan-t5-large',
        'family': 'encdec_t5',
        'source': 'hf_mirror',
        'batch_size': 72,
    },
    {
        'key': 'bart_large',
        'repo_id': 'facebook/bart-large',
        'modelscope_id': 'AI-ModelScope/bart-large',
        'family': 'encdec_bart',
        'source': 'modelscope',
        'batch_size': 72,
    },
]


def select_llm_configs(
    llm_configs: Sequence[Dict[str, Any]],
    config: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """Validate configured/requested LLM keys and preserve requested order.

    Args:
        llm_configs: Full catalog (``LLM_CONFIGS``).
        config: Must include ``epochs >= 1`` and optional ``llm_keys_to_run``.

    Returns:
        Catalog entries in the requested order (all keys if none requested).
    """
    assert int(config.get('epochs', 0)) >= 1, 'epochs must be >= 1'
    configs = list(llm_configs)
    config_keys = [str(item['key']) for item in configs]
    assert len(config_keys) == len(set(config_keys)), f'duplicate LLM config keys: {config_keys}'
    requested_raw = config.get('llm_keys_to_run')
    requested = config_keys if requested_raw is None else list(requested_raw)
    assert requested and all(isinstance(key, str) and key for key in requested)
    assert len(requested) == len(set(requested)), f'duplicate llm_keys_to_run entries: {requested}'
    unknown = sorted(set(requested) - set(config_keys))
    assert not unknown, f'unknown llm_keys_to_run entries: {unknown}'
    by_key = {str(item['key']): item for item in configs}
    return [by_key[key] for key in requested]


DATA_KEY_ALPHA: Dict[str, Optional[float]] = {
    'oracle_text': None,
    'eeg': 0.0,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
assert list(CONFIG['data_keys']) == ['eeg'], 'eeg_faith must train only an independent EEG path'
assert int(CONFIG['epochs']) >= 1, 'epochs must be >= 1'
assert len({str(item['key']) for item in LLM_CONFIGS}) == len(LLM_CONFIGS), 'duplicate LLM config keys'
assert abs(float(CONFIG['prompt_dim']) - float(CONFIG['in_dim'])) < 1e-8
assert int(CONFIG['input_text_len']) == int(CONFIG['out_len'])
apply_plot_style()
print(f'Device: {DEVICE}')
print(
    f'seed={CONFIG["seed"]}; out={CONFIG["output_dir"]}; model_cache={HF_CACHE}; '
    f'w_clip={CONFIG["w_clip"]} w_ar={CONFIG["w_ar"]} commit={CONFIG["commitment_weight"]} '
    f'epochs={CONFIG["epochs"]} patience={CONFIG["patience"]} lr={CONFIG["lr"]} '
    f'data_keys={CONFIG["data_keys"]} probe_every={CONFIG["probe_every"]} '
    f'lens_epochs={CONFIG["lens_epochs"]} lens_patience={CONFIG["lens_patience"]} '
    f'lens_lr={CONFIG["lens_lr"]} decoder_prompt={CONFIG["decoder_prompt"]!r}'
)


In [ ]:
# =============================================================
# ZuCo data pipeline: pickle load, split checks, GLIM sampler
# =============================================================

ALL_SUBJECTS = [
    'ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW',
    'ZMG', 'ZPH', 'YAC', 'YAG', 'YAK', 'YDG', 'YDR', 'YFR', 'YFS', 'YHS',
    'YIS', 'YLS', 'YMD', 'YMS', 'YRH', 'YRK', 'YRP', 'YSD', 'YSL', 'YTL',
]

PROMPT_KEYS: Dict[str, List[str]] = {
    'task': ['<UNK>'] + ['<NR>', '<TSR>'],
    'dataset': ['<UNK>'] + ['ZuCo1', 'ZuCo2'],
    'subject': ['<UNK>'] + ALL_SUBJECTS,
}


def task_to_prompt_token(task_key: str) -> str:
    """Map a ZuCo task id to the encoder prompt token.

    Args:
        task_key: Raw task string from the corpus (e.g. ``task1``, ``task3``).

    Returns:
        ``<TSR>`` for sentiment (task3), otherwise ``<NR>``.
    """
    return '<TSR>' if task_key == 'task3' else '<NR>'


def unpack_prompt_batch(
    prompt_batch: Union[List[Tuple[str, str, str]], Tuple[str, str, str], List, Tuple],
) -> Tuple[List[str], List[str], List[str]]:
    """Split a batch of ``(task, dataset, subject)`` rows into three lists.

    Args:
        prompt_batch: Either a list of 3-tuples (DataLoader collate) or a single
            3-tuple of strings.

    Returns:
        ``(tasks, datasets, subjects)`` with one string per batch row.
    """
    if isinstance(prompt_batch, list) and all(
        isinstance(row, tuple) and len(row) == 3 for row in prompt_batch
    ):
        rows = prompt_batch
        return (
            [str(row[0]) for row in rows],
            [str(row[1]) for row in rows],
            [str(row[2]) for row in rows],
        )
    if isinstance(prompt_batch, tuple) and len(prompt_batch) == 3 and all(
        isinstance(value, str) for value in prompt_batch
    ):
        first, second, third = prompt_batch
        return [str(first)], [str(second)], [str(third)]
    rows = list(prompt_batch)
    assert all(isinstance(row, (list, tuple)) and len(row) == 3 for row in rows)
    return (
        [str(row[0]) for row in rows],
        [str(row[1]) for row in rows],
        [str(row[2]) for row in rows],
    )


# --- pandas StringDtype pickle compat ---
import pandas._libs.arrays as _pd_libarrays
import pandas._libs.internals as _pd_libinternals
import pandas.core.indexes.base as _pd_indexes_base


class _CompatStringDtype:
    """Stand-in for pandas ``StringDtype`` when unpickling older ZuCo frames."""

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        """Record storage / NA metadata expected by the pickle payload.

        Args:
            *args: Optional ``(storage, na_value)`` positional fields.
            **kwargs: Named ``storage`` / ``na_value`` overrides.
        """
        self.storage = kwargs.get('storage', args[0] if args else 'python')
        self.na_value = kwargs.get('na_value', args[1] if len(args) > 1 else np.nan)
        self.kind = 'O'
        self.name = 'string'
        self.type = str


class _CompatStringArray(_pd_libarrays.NDArrayBacked):
    """Object-array stand-in for pandas ``StringArray`` during unpickle."""

    def __setstate__(self, state: Any) -> None:
        """Restore the backing ndarray from a pickle state dict or tuple.

        Args:
            state: Pickled StringArray payload.
        """
        if isinstance(state, dict):
            data = np.asarray(state.get('_ndarray', state.get('data')), dtype=object)
        elif isinstance(state, tuple) and len(state) >= 2:
            data = np.asarray(state[1], dtype=object)
        else:
            raise TypeError(f'Unsupported StringArray state: {type(state)}')
        _pd_libarrays.NDArrayBacked.__init__(self, data, np.dtype('O'))

    def __array__(self, dtype: Any = None, copy: Any = None) -> np.ndarray:
        """Return the object ndarray, optionally recast.

        Args:
            dtype: Optional NumPy dtype.
            copy: Unused; accepted for NumPy protocol compatibility.

        Returns:
            Object (or recast) ndarray of string values.
        """
        arr = np.asarray(self._ndarray, dtype=object)
        return np.asarray(arr, dtype=dtype) if dtype is not None else arr

    def __iter__(self) -> Iterator[Any]:
        """Iterate string values in the backing array."""
        return iter(self._ndarray)

    def __len__(self) -> int:
        """Number of stored string values."""
        return len(self._ndarray)


if not hasattr(_pd_indexes_base, '_eeg_faith_original_new_index'):
    current_new_index = _pd_indexes_base._new_Index
    prior_new_index = globals().get('_ORIG_NEW_INDEX')
    if getattr(current_new_index, '__name__', '') == '_new_index_compat' and callable(prior_new_index):
        current_new_index = prior_new_index
    _pd_indexes_base._eeg_faith_original_new_index = current_new_index  # type: ignore[attr-defined]
if not hasattr(_pd_libinternals, '_eeg_faith_original_unpickle_block'):
    current_unpickle_block = _pd_libinternals._unpickle_block
    prior_unpickle_block = globals().get('_ORIG_UNPICKLE_BLOCK')
    if getattr(current_unpickle_block, '__name__', '') == '_unpickle_block_compat' and callable(prior_unpickle_block):
        current_unpickle_block = prior_unpickle_block
    _pd_libinternals._eeg_faith_original_unpickle_block = current_unpickle_block  # type: ignore[attr-defined]
_ORIG_NEW_INDEX = _pd_indexes_base._eeg_faith_original_new_index  # type: ignore[attr-defined]
_ORIG_UNPICKLE_BLOCK = _pd_libinternals._eeg_faith_original_unpickle_block  # type: ignore[attr-defined]


def _new_index_compat(cls: Any, d: Dict[str, Any]) -> Any:
    """Rebuild a pandas Index, converting compat string arrays to object arrays.

    Args:
        cls: Index class being reconstructed.
        d: Pickle state dict; ``data`` may be a ``_CompatStringArray``.

    Returns:
        Index produced by the original pandas ``_new_Index``.
    """
    if isinstance(d, dict) and 'data' in d and isinstance(d['data'], _CompatStringArray):
        d = dict(d)
        d['data'] = np.asarray(d['data']._ndarray, dtype=object)
    return _ORIG_NEW_INDEX(cls, d)


def _unpickle_block_compat(values: Any, placement: Any, ndim: int) -> Any:
    """Unpickle an internals Block, flattening compat string arrays first.

    Args:
        values: Block values; may be a ``_CompatStringArray``.
        placement: Block column placement.
        ndim: Expected block ndim.

    Returns:
        Block from the original pandas ``_unpickle_block``.
    """
    if isinstance(values, _CompatStringArray) or type(values).__name__ == '_CompatStringArray':
        values = np.asarray(values._ndarray, dtype=object)
    if isinstance(values, np.ndarray) and values.ndim == 1 and int(ndim) == 2:
        values = values.reshape(1, -1)
    return _ORIG_UNPICKLE_BLOCK(values, placement, ndim)


def _patch_pandas_pickle_compat() -> None:
    """Install compatibility hooks without wrapping an earlier notebook run."""
    _pd_indexes_base._new_Index = _new_index_compat  # type: ignore[assignment]
    _pd_libinternals._unpickle_block = _unpickle_block_compat  # type: ignore[assignment]


_patch_pandas_pickle_compat()


class _CompatUnpickler(pickle.Unpickler):
    """Unpickler that remaps pandas StringArray/StringDtype to compat classes."""

    def find_class(self, module: str, name: str) -> Any:
        """Resolve a pickle class, substituting string-dtype compat types.

        Args:
            module: Declared module of the pickled class.
            name: Declared class / function name.

        Returns:
            The local compat class or the standard pickle lookup result.
        """
        if name == 'StringArray':
            return _CompatStringArray
        if name == 'StringDtype':
            return _CompatStringDtype
        if name == '_new_Index':
            return _new_index_compat
        if name == '_unpickle_block':
            return _unpickle_block_compat
        return super().find_class(module, name)


def read_dataframe_pickle(path: Union[str, Path]) -> pd.DataFrame:
    """Load a DataFrame pickle through the string-dtype compatibility unpickler.

    Args:
        path: Path to a pandas DataFrame pickle.

    Returns:
        Loaded frame with string columns cast to object dtype.
    """
    path = Path(path)
    with open(path, 'rb') as handle:
        obj = _CompatUnpickler(handle).load()
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f'Expected DataFrame, got {type(obj)}')
    for col in list(obj.columns):
        if pd.api.types.is_string_dtype(obj[col]) or str(obj[col].dtype).startswith('string'):
            obj[col] = obj[col].astype(object)
        else:
            try:
                if isinstance(obj[col].array, _CompatStringArray):
                    obj[col] = np.asarray(obj[col].array._ndarray, dtype=object)
            except Exception:
                pass
    return obj


_PICKLE_MAGIC = b'\x80'
_DATA_PATH_CANDIDATES: Tuple[str, ...] = (
    './autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
    '/root/autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
    './preprocessed_data/zuco_merged_whiten_norm.df',
    './autodl-tmp/zuco_merged_whiten_norm.df',
)


def _collect_data_candidates(requested: str) -> List[Path]:
    """Return the explicitly requested path before all fallbacks."""
    ordered: List[Path] = []
    seen: set[str] = set()
    for raw in (requested, *_DATA_PATH_CANDIDATES):
        p = Path(raw)
        key = str(p.resolve()) if p.exists() else str(p)
        if key in seen:
            continue
        seen.add(key)
        ordered.append(p)
    assert ordered and ordered[0] == Path(requested)
    return ordered


def _load_pickle_dataframe_with_fallback(requested: str) -> Tuple[pd.DataFrame, Path]:
    """Try the requested path and fallbacks until a pickle DataFrame loads.

    Args:
        requested: Preferred corpus path.

    Returns:
        ``(dataframe, resolved_path)`` for the first readable pickle.
    """
    candidates = _collect_data_candidates(requested)
    errors: List[str] = []
    for path in candidates:
        if not path.exists():
            continue
        size = path.stat().st_size
        with open(path, 'rb') as f:
            header = f.read(1)
        if header != _PICKLE_MAGIC:
            continue
        # Allow smaller files for smoke / subset testing
        print(f'Loading {path} ({size:,} bytes) ...')
        try:
            return read_dataframe_pickle(path), path
        except Exception as exc:
            errors.append(f'{path}: {exc}')
            print(f'  [WARN] Failed: {exc}')
    raise UnpicklingError(
        'No loadable ZuCo corpus pickle found.\n' + '\n'.join(errors)
    )


_INT64_MIN = -(2 ** 63)
_INT64_MAX = (2 ** 63) - 1
_FLOAT_SAFE_INTEGER_MAX = (2 ** 53) - 1


def _require_signed_int64(value: int, original: Any) -> int:
    """Return a Python int only when it lies in the signed-int64 domain."""
    integer = int(value)
    assert _INT64_MIN <= integer <= _INT64_MAX, f'text uid outside signed int64: {original!r}'
    return integer


def canonicalize_text_uid(value: Any) -> int:
    """Losslessly canonicalize an integer-like UID to a signed-int64 Python ``int``."""
    if value is None or isinstance(value, (bool, np.bool_)):
        raise AssertionError(f'invalid text uid: {value!r}')
    is_null = pd.isna(value)
    if isinstance(is_null, (bool, np.bool_)) and bool(is_null):
        raise AssertionError(f'null text uid: {value!r}')
    if isinstance(value, (int, np.integer)):
        return _require_signed_int64(int(value), value)
    if isinstance(value, (float, np.floating)):
        numeric = float(value)
        assert math.isfinite(numeric) and numeric.is_integer(), f'nonintegral text uid: {value!r}'
        assert abs(numeric) <= float(_FLOAT_SAFE_INTEGER_MAX), (
            f'float text uid exceeds exact-safe range: {value!r}'
        )
        return _require_signed_int64(int(numeric), value)
    if isinstance(value, str):
        text = unicodedata.normalize('NFKC', value).strip()
        signless = text[1:] if text[:1] in {'+', '-'} else text
        assert signless and all('0' <= char <= '9' for char in signless), f'malformed text uid: {value!r}'
        return _require_signed_int64(int(text), value)
    raise AssertionError(f'unsupported text uid type: {type(value).__name__}')


def normalize_text_for_split_check(text: Any) -> str:
    """Reject nulls, then NFKC/case-fold and collapse whitespace."""
    if text is None:
        raise AssertionError('null input text')
    is_null = pd.isna(text)
    if isinstance(is_null, (bool, np.bool_)) and bool(is_null):
        raise AssertionError('null input text')
    normalized = unicodedata.normalize('NFKC', str(text)).casefold()
    return ' '.join(normalized.split())


def canonicalize_corpus_identity_columns(merged: pd.DataFrame) -> None:
    """Canonicalize UIDs and reject null texts before any split/mapping checks."""
    assert 'text uid' in merged.columns and 'input text' in merged.columns
    canonical_uids = [canonicalize_text_uid(value) for value in merged['text uid']]
    merged['text uid'] = pd.Series(canonical_uids, index=merged.index, dtype=object)
    assert all(type(value) is int for value in merged['text uid'])
    for text in merged['input text']:
        _ = normalize_text_for_split_check(text)


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Stream a file into SHA-256 without loading it into memory."""
    assert int(chunk_size) >= 1
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(int(chunk_size))
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def validate_eeg_mask_sample(eeg: Any, mask: Any) -> None:
    """Validate one raw EEG trial `(1280,128)` and binary mask `(1280,)`."""
    eeg_arr = np.asarray(eeg)
    mask_arr = np.asarray(mask)
    assert eeg_arr.shape == (1280, 128), f'EEG shape {eeg_arr.shape}'
    assert mask_arr.shape == (1280,), f'mask shape {mask_arr.shape}'
    assert np.isfinite(eeg_arr).all(), 'non-finite EEG values'
    assert np.isfinite(mask_arr).all(), 'non-finite mask values'
    mask_values = set(np.unique(mask_arr).tolist())
    assert mask_values <= {0, 1}, f'non-binary mask values: {mask_values}'
    assert int(np.count_nonzero(mask_arr)) >= 1, 'mask has no valid sample'


def validate_all_eeg_mask_rows(merged: pd.DataFrame) -> None:
    """Stream through every corpus row without constructing a corpus-sized tensor."""
    assert len(merged) >= 1, 'empty corpus'
    for position, (row_label, eeg, mask) in enumerate(
        zip(merged.index, merged['eeg'], merged['mask'])
    ):
        try:
            validate_eeg_mask_sample(eeg, mask)
        except (AssertionError, TypeError, ValueError) as exc:
            raise AssertionError(
                f'invalid EEG/mask at position={position} index={row_label!r}: {exc}'
            ) from exc


def validate_corpus_splits(merged: pd.DataFrame) -> None:
    """Canonicalize identities, then check phases, mappings, and leakage."""
    canonicalize_corpus_identity_columns(merged)
    phases = merged['phase'].astype(str)
    expected = {'train', 'val', 'test'}
    observed = set(phases.unique().tolist())
    assert observed == expected, f'phase values must be exactly {expected}, got {observed}'
    norm_text = merged['input text'].map(normalize_text_for_split_check)
    assert bool(norm_text.str.len().gt(0).all()), 'empty normalized text'
    uid_text = pd.DataFrame({'uid': merged['text uid'], 'text': norm_text})
    inconsistent = uid_text.groupby('uid', sort=False)['text'].nunique(dropna=False)
    bad_uids = inconsistent[inconsistent.ne(1)]
    assert bad_uids.empty, f'UID maps to multiple normalized texts: {bad_uids.index[:10].tolist()}'
    uid_sets = {phase: set(merged.loc[phases.eq(phase), 'text uid'].tolist()) for phase in expected}
    text_sets = {phase: set(norm_text.loc[phases.eq(phase)].tolist()) for phase in expected}
    for left, right in (('train', 'val'), ('train', 'test'), ('val', 'test')):
        uid_overlap = uid_sets[left] & uid_sets[right]
        text_overlap = text_sets[left] & text_sets[right]
        assert not uid_overlap, f'{left}/{right} UID leakage: {list(uid_overlap)[:10]}'
        assert not text_overlap, f'{left}/{right} normalized-text leakage: {list(text_overlap)[:3]}'


def load_merged_corpus(data_path: str) -> pd.DataFrame:
    """Load and validate the ZuCo whitened corpus.

    Args:
        data_path: Preferred pickle path; fallbacks are tried on failure.

    Returns:
        Reset-index frame with provenance in ``attrs['eeg_faith_provenance']``.
    """
    df, df_path = _load_pickle_dataframe_with_fallback(data_path)
    if Path(data_path) != df_path:
        print(f'  Resolved data_path: {data_path} -> {df_path}')
    merged = df.reset_index(drop=True)
    if 'row_index' not in merged.columns:
        merged['row_index'] = np.arange(len(merged), dtype=np.int64)
    assert 'phase' in merged.columns
    required = {
        'eeg', 'mask', 'input text', 'text uid', 'phase', 'dataset', 'task', 'subject',
    }
    missing = required - set(merged.columns)
    assert not missing, f'Missing columns: {missing}'
    validate_corpus_splits(merged)
    validate_all_eeg_mask_rows(merged)
    counts = merged['phase'].astype(str).value_counts()
    phase_counts = {phase: int(counts.get(phase, 0)) for phase in ('train', 'val', 'test')}
    resolved_path = df_path.resolve()
    provenance: Dict[str, Any] = {
        'resolved_path': str(resolved_path),
        'file_size_bytes': int(resolved_path.stat().st_size),
        'sha256': sha256_file(resolved_path),
        'rows': int(len(merged)),
        'phase_counts': phase_counts,
        'required_columns': sorted(required),
        'integrity_checks': {
            'canonical_integer_uids': True,
            'non_null_nfkc_normalized_texts': True,
            'exact_phases': True,
            'uid_to_normalized_text': True,
            'cross_phase_uid_disjoint': True,
            'cross_phase_normalized_text_disjoint': True,
            'all_rows_eeg_mask_valid': True,
        },
    }
    assert provenance['rows'] == sum(phase_counts.values())
    merged.attrs['eeg_faith_provenance'] = provenance
    print(
        f'  Corpus phases: train={phase_counts["train"]}, '
        f'val={phase_counts["val"]}, test={phase_counts["test"]}; '
        f'all {len(merged):,} EEG/mask rows valid; UIDs/texts cross-phase disjoint'
    )
    return merged


def get_phase_df(merged: pd.DataFrame, phase: str) -> pd.DataFrame:
    """Return the rows of one split, reset-indexed.

    Args:
        merged: Full corpus.
        phase: ``train``, ``val``, or ``test``.

    Returns:
        Non-empty phase frame.
    """
    assert phase in ('train', 'val', 'test')
    frame = merged.loc[merged['phase'].astype(str) == phase].reset_index(drop=True)
    assert len(frame) > 0, f'Empty phase={phase}'
    return frame


class ProbeDataset(Dataset):
    """Per-row EEG/text dataset for CLIP + faithfulness analyses."""

    def __init__(self, df: pd.DataFrame) -> None:
        """Cache columns needed by the collate / encoder path.

        Args:
            df: Phase frame from ``get_phase_df``.
        """
        self.eeg = df['eeg'].tolist()
        self.mask = df['mask'].tolist()
        self.input_text = df['input text'].tolist()
        for text in self.input_text:
            _ = normalize_text_for_split_check(text)
        self.text_uid = [canonicalize_text_uid(value) for value in df['text uid']]
        self.dataset = df['dataset'].tolist()
        self.task = df['task'].tolist()
        self.subject = df['subject'].tolist()
        self.row_index = (
            [int(x) for x in df['row_index'].tolist()]
            if 'row_index' in df.columns else list(range(len(df)))
        )
        assert len(self.eeg) >= 1, 'empty ProbeDataset'

    def __len__(self) -> int:
        """Number of EEG/text rows."""
        return len(self.eeg)

    def __getitem__(self, idx: int) -> dict:
        """Return one sample dict (EEG tensor, mask, texts, prompt tokens).

        Args:
            idx: Dataset index.

        Returns:
            Collate-ready sample with ``prompt`` as ``(task, dataset, subject)``.
        """
        ds_tok = self.dataset[idx] if self.dataset[idx] in PROMPT_KEYS['dataset'] else '<UNK>'
        sub_tok = self.subject[idx] if self.subject[idx] in PROMPT_KEYS['subject'] else '<UNK>'
        text = str(self.input_text[idx])
        return {
            'eeg': torch.from_numpy(np.array(self.eeg[idx], dtype=np.float32)),
            'mask': torch.from_numpy(np.array(self.mask[idx], dtype=np.int32)),
            'input text': text,
            'target text': text,
            'text uid': int(self.text_uid[idx]),
            'prompt': (task_to_prompt_token(str(self.task[idx])), ds_tok, sub_tok),
            'row_index': int(self.row_index[idx]),
            'dataset_raw': str(self.dataset[idx]),
            'task_raw': str(self.task[idx]),
            'subject_raw': str(self.subject[idx]),
        }


def probe_collate_fn(batch: List[dict]) -> dict:
    """Stack EEG/mask/UID tensors; leave text and prompt fields as lists.

    Args:
        batch: Samples from ``ProbeDataset.__getitem__``.

    Returns:
        Batched dict for ``ProbeSystem``.
    """
    out: Dict[str, Any] = {}
    for k in batch[0].keys():
        vals = [b[k] for b in batch]
        if k == 'eeg':
            out[k] = torch.stack(vals, dim=0)
        elif k == 'mask':
            out[k] = torch.stack(vals, dim=0)
        elif k in ('text uid', 'row_index'):
            out[k] = torch.tensor(vals, dtype=torch.long)
        else:
            out[k] = vals
    return out


def _dataset_text_uids(dataset: Dataset) -> List[int]:
    """Return the per-index text UID list used by ``GLIMSampler``.

    Args:
        dataset: Must expose ``text_uid``.

    Returns:
        Signed-int64 UID for every dataset index.
    """
    n = len(dataset)
    if not hasattr(dataset, 'text_uid'):
        raise AssertionError('dataset missing text_uid')
    uids = dataset.text_uid  # type: ignore[attr-defined]
    out = [int(uids[i]) for i in range(n)]
    assert len(out) == n
    return out


class GLIMSampler(Sampler[List[int]]):
    """Exhaustive every-row-once batches with unique text UIDs per batch."""

    def __init__(
        self,
        dataset: Dataset,
        identifiers: Sequence[int],
        phase: str,
        batch_size: int,
        seed: int,
        shuffle: bool,
    ) -> None:
        """Build a UID-unique batch plan for one phase.

        Args:
            dataset: Phase dataset (length must match ``identifiers``).
            identifiers: Per-index text UIDs.
            phase: ``train``, ``val``, or ``test`` (informational).
            batch_size: Maximum rows per batch.
            seed: Epoch-0 RNG seed; later epochs add ``epoch``.
            shuffle: If True, permute groups, rows, and batch order.
        """
        assert phase in ('train', 'val', 'test')
        assert batch_size >= 1 and len(dataset) >= 1
        self.dataset = dataset
        self.identifiers = [int(uid) for uid in identifiers]
        assert len(self.identifiers) == len(dataset)
        self.phase = str(phase)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.shuffle = bool(shuffle)
        self.epoch = 0
        counts = Counter(self.identifiers)
        self.n_batches = max(max(counts.values()), math.ceil(len(dataset) / self.batch_size))
        assert self.n_batches >= 1

    def set_epoch(self, epoch: int) -> None:
        """Set the epoch used as an RNG offset.

        Args:
            epoch: Non-negative epoch index.
        """
        self.epoch = int(epoch)

    def __len__(self) -> int:
        """Number of batches produced each epoch."""
        return self.n_batches

    def _build_batches(self, epoch: int) -> List[List[int]]:
        """Pack every row once into UID-unique batches for one epoch.

        Args:
            epoch: RNG offset added to ``self.seed``.

        Returns:
            List of index-batches covering ``range(len(dataset))`` exactly once.
        """
        generator = torch.Generator().manual_seed(self.seed + int(epoch))
        groups: Dict[int, List[int]] = defaultdict(list)
        for idx, uid in enumerate(self.identifiers):
            groups[int(uid)].append(int(idx))
        group_items = list(groups.items())
        if self.shuffle:
            order = torch.randperm(len(group_items), generator=generator).tolist()
            group_items = [group_items[i] for i in order]
        group_items.sort(key=lambda item: -len(item[1]))
        batches: List[List[int]] = [[] for _ in range(self.n_batches)]
        for _uid, raw_indices in group_items:
            indices = list(raw_indices)
            if self.shuffle and len(indices) > 1:
                perm = torch.randperm(len(indices), generator=generator).tolist()
                indices = [indices[i] for i in perm]
            assert len(indices) <= self.n_batches
            if self.shuffle:
                tie_order = torch.randperm(self.n_batches, generator=generator).tolist()
                tie_rank = {batch_idx: rank for rank, batch_idx in enumerate(tie_order)}
            else:
                tie_rank = {batch_idx: batch_idx for batch_idx in range(self.n_batches)}
            available = sorted(
                range(self.n_batches), key=lambda batch_idx: (len(batches[batch_idx]), tie_rank[batch_idx]),
            )
            for sample_idx, batch_idx in zip(indices, available[:len(indices)]):
                batches[batch_idx].append(sample_idx)
        assert all(1 <= len(batch) <= self.batch_size for batch in batches), [len(b) for b in batches]
        if self.shuffle:
            for batch in batches:
                if len(batch) > 1:
                    perm = torch.randperm(len(batch), generator=generator).tolist()
                    batch[:] = [batch[i] for i in perm]
            order = torch.randperm(len(batches), generator=generator).tolist()
            batches = [batches[i] for i in order]
        flat = [idx for batch in batches for idx in batch]
        assert sorted(flat) == list(range(len(self.dataset))), 'sampler must cover every row exactly once'
        for batch in batches:
            uids = [self.identifiers[idx] for idx in batch]
            assert len(uids) == len(set(uids)), f'duplicate UID inside batch: {uids}'
        return batches

    def __iter__(self) -> Iterator[List[int]]:
        """Yield this epoch's batches, then increment ``epoch``."""
        batches = self._build_batches(self.epoch)
        assert len(batches) == len(self)
        self.epoch += 1
        yield from batches


class PhaseDataModule:
    """Pooled phase DataModule for eeg_faith."""

    def __init__(
        self,
        merged_df: pd.DataFrame,
        provenance: Dict[str, Any],
        batch_size: int = 72,
        num_workers: int = 0,
    ) -> None:
        """Store the corpus and its provenance; call ``setup`` before loaders.

        Args:
            merged_df: Output of ``load_merged_corpus``.
            provenance: ``merged.attrs['eeg_faith_provenance']``.
            batch_size: GLIM / sequential batch size.
            num_workers: DataLoader workers.
        """
        self.merged_df = merged_df
        self.provenance = dict(provenance)
        assert int(self.provenance.get('rows', -1)) == len(merged_df)
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_set: Optional[ProbeDataset] = None
        self.val_set: Optional[ProbeDataset] = None
        self.test_set: Optional[ProbeDataset] = None

    def setup(self) -> None:
        """Build ``ProbeDataset`` objects for train, val, and test."""
        train_df = get_phase_df(self.merged_df, 'train')
        val_df = get_phase_df(self.merged_df, 'val')
        test_df = get_phase_df(self.merged_df, 'test')
        print(
            f'  [pooled] Train: {len(train_df):,}  |  Val: {len(val_df):,}  '
            f'|  Test: {len(test_df):,}'
        )
        self.train_set = ProbeDataset(train_df)
        self.val_set = ProbeDataset(val_df)
        self.test_set = ProbeDataset(test_df)

    def train_dataloader(self, seed: int = 2026) -> DataLoader:
        """GLIM unique-text batches (shuffle=True)."""
        assert self.train_set is not None
        sampler = GLIMSampler(
            self.train_set,
            _dataset_text_uids(self.train_set),
            phase='train',
            batch_size=self.batch_size,
            seed=seed,
            shuffle=True,
        )
        return DataLoader(
            self.train_set, batch_sampler=sampler,
            num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
        )

    def val_dataloader(self, seed: int = 2026, sequential: bool = False) -> DataLoader:
        """GLIM unique-text batches, or sequential if ``sequential`` (overlap / probes)."""
        assert self.val_set is not None
        if sequential:
            return DataLoader(
                self.val_set, batch_size=self.batch_size, shuffle=False,
                num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
            )
        sampler = GLIMSampler(
            self.val_set,
            _dataset_text_uids(self.val_set),
            phase='val',
            batch_size=self.batch_size,
            seed=seed,
            shuffle=False,
        )
        return DataLoader(
            self.val_set, batch_sampler=sampler,
            num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
        )

    def test_dataloader(self) -> DataLoader:
        """Sequential test loader (no shuffle, no GLIM packing).

        Returns:
            Test DataLoader.
        """
        assert self.test_set is not None
        return DataLoader(
            self.test_set, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
        )


In [ ]:
# =============================================================
# SemKey EEGEncoder (raw 1280 samples, no conv downsample)
# Adapted from SemKey-main/model/modules.py (GLIM backbone).
# =============================================================

def get_1d_sincos_pos_embed_from_grid(embed_dim: int, pos: np.ndarray) -> np.ndarray:
    """Sine-cosine 1-D positional encoding for a regularly spaced grid.

    Args:
        embed_dim: Even embedding width.
        pos: 1-D positions (typically ``0 .. in_len-1``).

    Returns:
        Array of shape ``(len(pos), embed_dim)``, float32.
    """
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=np.float64)
    omega /= embed_dim / 2.0
    omega = 1.0 / 10000 ** omega
    pos = pos.reshape(-1)
    out = np.einsum('m,d->md', pos, omega)
    emb = np.concatenate([np.sin(out), np.cos(out)], axis=1)
    return emb.astype(np.float32)


class Mlp(nn.Module):
    """Two-layer GELU MLP used inside encoder / decoder blocks."""

    def __init__(self, in_features: int, hidden_features: int, drop: float = 0.0) -> None:
        """Build ``Linear → GELU → Dropout → Linear → Dropout``.

        Args:
            in_features: Input and residual width.
            hidden_features: Expansion width.
            drop: Dropout probability after each linear.
        """
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU(approximate='tanh')
        self.drop = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_features, in_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the MLP to the last dimension.

        Args:
            x: ``(..., in_features)``.

        Returns:
            Same shape as ``x``.
        """
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class SelfAttention(nn.MultiheadAttention):
    """Batch-first self-attention with optional causal or padding mask."""

    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.0,
                 is_causal: bool = False) -> None:
        """Configure multi-head self-attention.

        Args:
            hidden_dim: Model width (must be divisible by ``num_heads``).
            num_heads: Attention heads.
            dropout: Attention dropout.
            is_causal: If True and no padding mask is given, apply a causal mask.
        """
        super().__init__(hidden_dim, num_heads, dropout, batch_first=True)
        self.num_heads = num_heads
        self.is_causal = is_causal

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None,
                need_weights: bool = False) -> torch.Tensor:
        """Self-attend over ``x``.

        Args:
            x: ``(B, L, D)``.
            mask: Optional ``(B, L)`` keep-mask (True = valid).
            need_weights: Unused by callers; forwarded to MHA.

        Returns:
            Attended ``(B, L, D)``.
        """
        if self.is_causal and mask is None:
            B, L, _ = x.shape
            attn_mask = torch.triu(
                torch.full((L, L), float('-inf'), dtype=x.dtype, device=x.device), diagonal=1
            ).unsqueeze(0).expand(B * self.num_heads, -1, -1)
            return super().forward(x, x, x, attn_mask=attn_mask, need_weights=need_weights)[0]
        kpm = (~mask.bool()) if mask is not None else None
        return super().forward(x, x, x, key_padding_mask=kpm, need_weights=need_weights)[0]


class CrossAttention(nn.MultiheadAttention):
    """Batch-first cross-attention from queries onto a key/value memory."""

    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        """Configure multi-head cross-attention.

        Args:
            hidden_dim: Shared query / memory width.
            num_heads: Attention heads.
            dropout: Attention dropout.
        """
        super().__init__(hidden_dim, num_heads, dropout, batch_first=True)

    def forward(self, q: torch.Tensor, x: torch.Tensor, mask: Optional[torch.Tensor] = None,
                need_weights: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Attend queries ``q`` to memory ``x``.

        Args:
            q: ``(B, Q, D)`` queries.
            x: ``(B, L, D)`` memory.
            mask: Optional ``(B, L)`` keep-mask over memory.
            need_weights: If True, also return attention weights.

        Returns:
            ``(attended_queries, weights_or_None)``.
        """
        kpm = (~mask.bool()) if mask is not None else None
        return super().forward(q, x, x, key_padding_mask=kpm, need_weights=need_weights)


class EncoderBlock(nn.Module):
    """Prompt-conditioned transformer block with optional temporal adaLN."""

    def __init__(self, hidden_dim: int, hidden_len: int, inject_prompt: bool = True,
                 temporal_modulate: bool = True, is_causal: bool = False,
                 num_heads: int = 8, mlp_ratio: int = 4, dropout: float = 0.0) -> None:
        """Build one in-stream SemKey encoder block.

        Args:
            hidden_dim: Token width.
            hidden_len: Sequence length (for temporal adaLN).
            inject_prompt: If True, modulate norms from the prompt embedding.
            temporal_modulate: If True (and prompt injection is on), also
                apply per-timestep scale/shift from the prompt.
            is_causal: Causal self-attention (unused for the in-stream).
            num_heads: Attention heads.
            mlp_ratio: MLP expansion.
            dropout: Residual dropout.
        """
        super().__init__()
        self.inject_prompt = inject_prompt
        self.temporal_modulate = temporal_modulate
        self.norm1 = nn.LayerNorm(hidden_dim, eps=1e-6, elementwise_affine=not inject_prompt)
        self.attn = SelfAttention(hidden_dim, num_heads, dropout, is_causal=is_causal)
        self.norm2 = nn.LayerNorm(hidden_dim, eps=1e-6, elementwise_affine=not inject_prompt)
        self.mlp = Mlp(hidden_dim, hidden_dim * mlp_ratio, drop=dropout)
        if inject_prompt:
            self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, 6 * hidden_dim, bias=True))
            nn.init.zeros_(self.adaLN[1].weight)
            nn.init.zeros_(self.adaLN[1].bias)
            if temporal_modulate:
                self.t_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, 2 * hidden_len, bias=True))
                nn.init.zeros_(self.t_adaLN[1].weight)
                nn.init.zeros_(self.t_adaLN[1].bias)

    def modulate(self, x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        """Channel-wise affine: ``x * (1 + scale) + shift``.

        Args:
            x: ``(B, L, D)``.
            shift: ``(B, D)``.
            scale: ``(B, D)``.

        Returns:
            Modulated ``x``.
        """
        return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

    def t_modulate(self, x: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
        """Time-wise affine from the prompt embedding.

        Args:
            x: ``(B, L, D)``.
            p: ``(B, D)`` prompt embedding.

        Returns:
            Temporally modulated ``x``.
        """
        scale, shift = self.t_adaLN(p).chunk(2, dim=1)
        return x * (1 + scale.unsqueeze(2)) + shift.unsqueeze(2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor, p: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Run attention + MLP with optional prompt modulation.

        Args:
            x: ``(B, L, D)`` tokens.
            mask: ``(B, L)`` keep-mask.
            p: ``(B, D)`` prompt embedding when ``inject_prompt`` is True.

        Returns:
            Updated tokens, same shape as ``x``.
        """
        if not self.inject_prompt:
            x = x + self.attn(self.norm1(x), mask)
            return x + self.mlp(self.norm2(x))
        if self.temporal_modulate:
            x = self.t_modulate(x, p)
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN(p).chunk(6, dim=1)
        x = x + gate_msa.unsqueeze(1) * self.attn(self.modulate(self.norm1(x), shift_msa, scale_msa), mask)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(self.modulate(self.norm2(x), shift_mlp, scale_mlp))
        return x


class DecoderBlock(nn.Module):
    """Causal self-attention + cross-attention Q-Merger block."""

    def __init__(self, dim: int, num_heads: int = 8, mlp_ratio: int = 4,
                 dropout: float = 0.0, is_causal: bool = True) -> None:
        """Build one out-stream (query) block.

        Args:
            dim: Query / memory width.
            num_heads: Attention heads.
            mlp_ratio: MLP expansion.
            dropout: Residual dropout.
            is_causal: Causal mask on query self-attention.
        """
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.self_attn = SelfAttention(dim, num_heads, dropout, is_causal=is_causal)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.cross_attn = CrossAttention(dim, num_heads, dropout)
        self.norm3 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, dim * mlp_ratio, drop=dropout)

    def forward(self, q: torch.Tensor, x: torch.Tensor, x_mask: torch.Tensor,
                need_weights: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Update queries by attending to the EEG memory.

        Args:
            q: ``(B, Q, D)`` learned queries.
            x: ``(B, L, D)`` encoder memory.
            x_mask: ``(B, L)`` keep-mask over memory.
            need_weights: If True, return cross-attention weights.

        Returns:
            ``(updated_queries, cross_attn_weights_or_None)``.
        """
        q = q + self.self_attn(self.norm1(q), need_weights=need_weights)
        attn_out, attn_w = self.cross_attn(self.norm2(q), x, x_mask, need_weights=need_weights)
        q = q + attn_out
        q = q + self.mlp(self.norm3(q))
        return q, attn_w


class PromptEmbedder(nn.Module):
    """Sum of task / dataset / subject embeddings, weighted by learned σ."""

    def __init__(self, dim: int = 128, prompt_keys: Optional[Dict[str, List[str]]] = None,
                 drop_probs: Tuple[float, float, float] = (0.0, 0.0, 0.0)) -> None:
        """Create one embedding table per prompt field.

        Args:
            dim: Embedding width (must match encoder ``in_dim``).
            prompt_keys: Token inventories; defaults to ``PROMPT_KEYS``.
            drop_probs: Per-field dropout to the ``<UNK>`` id during training.
        """
        super().__init__()
        self.prompt_keys = prompt_keys or PROMPT_KEYS
        prompt_nums = tuple(len(v) for v in self.prompt_keys.values())
        self.dim = dim
        self.drop_probs = drop_probs
        self.embedders = nn.ModuleList([nn.Embedding(n, dim) for n in prompt_nums])
        for emb in self.embedders:
            nn.init.normal_(emb.weight, std=0.02)
        self.sigma = nn.Parameter(torch.tensor([n / sum(prompt_nums) for n in prompt_nums], dtype=torch.float32))

    @torch.no_grad()
    def p_drop(self, src_pids: torch.Tensor, drop_prob: float) -> torch.Tensor:
        """Randomly replace prompt ids with 0 (``<UNK>``).

        Args:
            src_pids: Integer ids ``(B,)``.
            drop_prob: Independent drop probability.

        Returns:
            Possibly dropped ids, same shape.
        """
        if drop_prob <= 0.0:
            return src_pids
        drop_mask = torch.rand(src_pids.shape, device=src_pids.device) < drop_prob
        return torch.where(drop_mask, torch.zeros_like(src_pids), src_pids)

    def forward(self, prompt_ids: torch.Tensor,
                eval_pembed: Literal['zero', 'sum', 'mean', 'src'] = 'src') -> torch.Tensor:
        """Embed discrete prompt ids and sum σ-weighted fields.

        Args:
            prompt_ids: ``(B, 3)`` integer ids for task, dataset, subject.
            eval_pembed: Unused (kept for SemKey API compatibility).

        Returns:
            Prompt vector ``(B, dim)``.
        """
        p_embed = torch.zeros(prompt_ids.shape[0], self.dim, device=prompt_ids.device)
        for k, (embedder, prob) in enumerate(zip(self.embedders, self.drop_probs)):
            pids = prompt_ids[:, k]
            if self.training and prob > 0.0:
                pids = self.p_drop(pids, prob)
            p = embedder(pids)
            p_embed = p_embed + p * self.sigma[k]
        return p_embed

    def encode(self, prompts: List[List[str]], device: Optional[torch.device] = None) -> torch.Tensor:
        """Map string prompt columns to integer ids.

        Args:
            prompts: Three lists ``[tasks, datasets, subjects]``, each length B.
            device: Device for the returned LongTensor.

        Returns:
            ``(B, 3)`` ids; unknown strings map to 0.
        """
        bsz = len(prompts[0])
        ids = []
        for k, keys in enumerate(self.prompt_keys.values()):
            row = [keys.index(prompts[k][i]) if prompts[k][i] in keys else 0 for i in range(bsz)]
            ids.append(torch.tensor(row, dtype=torch.long, device=device))
        return torch.stack(ids, dim=-1)


class EEGEncoder(nn.Module):
    """Prompt-modulated encoder + learned-query Q-Merger on raw 1280 samples."""

    def __init__(
        self,
        in_len: int = 1280,
        out_len: int = 96,
        in_dim: int = 128,
        out_dim: int = 128,
        n_in_blocks: int = 6,
        n_out_blocks: int = 6,
        num_heads: int = 8,
        mlp_ratio: int = 4,
        dropout: float = 0.0,
        use_channel_weights: bool = True,
    ) -> None:
        """Build the SemKey encoder used by ``ProbeSystem``.

        Args:
            in_len: EEG time samples (1280).
            out_len: Q-Merger tokens (96).
            in_dim: Channel / prompt width (128).
            out_dim: Merger output width (``hidden_dim``).
            n_in_blocks: In-stream encoder depth.
            n_out_blocks: Query-merger depth.
            num_heads: Attention heads.
            mlp_ratio: MLP expansion.
            dropout: Block dropout.
            use_channel_weights: If True, learn a per-channel scale.
        """
        super().__init__()
        self.in_len = in_len
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.out_len = out_len
        self.use_channel_weights = use_channel_weights
        block_kw = dict(num_heads=num_heads, mlp_ratio=mlp_ratio, dropout=dropout)
        self.in_blocks = nn.ModuleList([
            EncoderBlock(in_dim, in_len, inject_prompt=True, temporal_modulate=True,
                         is_causal=False, **block_kw)
            for _ in range(n_in_blocks)
        ])
        self.x_proj = nn.Linear(in_dim, out_dim)
        self.norm1 = nn.LayerNorm(out_dim, eps=1e-6)
        self.shared_queries = nn.Parameter(torch.randn(1, out_len, out_dim) * 0.02)
        self.out_blocks = nn.ModuleList([
            DecoderBlock(out_dim, is_causal=True, **block_kw) for _ in range(n_out_blocks)
        ])
        self.norm2 = nn.LayerNorm(out_dim, eps=1e-6)
        pos = get_1d_sincos_pos_embed_from_grid(in_dim, np.arange(in_len))
        self.register_buffer('pos_embed', torch.from_numpy(pos).unsqueeze(0), persistent=False)
        if use_channel_weights:
            self.channel_weights = nn.Parameter(torch.ones(1, 1, in_dim))

    def forward(self, eeg: torch.Tensor, mask: torch.Tensor, p: torch.Tensor,
                need_weights: bool = False) -> Tuple[torch.Tensor, torch.Tensor, dict]:
        """Encode EEG to 96 merger tokens.

        Args:
            eeg: ``(B, 1280, 128)``.
            mask: ``(B, 1280)`` keep-mask.
            p: ``(B, 128)`` prompt embedding.
            need_weights: If True, collect per-block cross-attention weights.

        Returns:
            ``(Zi, memory, attn_weights)`` where ``Zi`` is ``(B, 96, out_dim)``.
        """
        assert eeg.ndim == 3 and eeg.shape[1:] == (self.in_len, self.in_dim)
        assert mask.shape == eeg.shape[:2]
        assert p.shape == (eeg.shape[0], self.in_dim)
        x = eeg * self.channel_weights if self.use_channel_weights else eeg
        x = x + self.pos_embed
        for block in self.in_blocks:
            x = block(x, mask, p)
        memory = self.norm1(self.x_proj(x))
        q = self.shared_queries.expand(eeg.shape[0], -1, -1)
        attn_weights: dict = {}
        for j, block in enumerate(self.out_blocks):
            q, aw = block(q, memory, mask, need_weights=need_weights)
            attn_weights[j] = aw
        Zi = self.norm2(q)
        assert Zi.shape == (eeg.shape[0], self.out_len, self.out_dim)
        assert not torch.isnan(Zi).any(), 'NaN in Zi'
        return Zi, memory, attn_weights


def build_eeg_encoder(config: Dict[str, Any]) -> EEGEncoder:
    """Construct the SemKey encoder from the notebook CONFIG.

    Args:
        config: Must include ``in_len``, ``out_len``, ``in_dim``, ``hidden_dim``,
            block counts, ``num_heads``, ``mlp_ratio``, ``dropout``, and
            ``use_channel_weights``.

    Returns:
        Uninitialized ``EEGEncoder`` (weights are random until training).
    """
    return EEGEncoder(
        in_len=int(config['in_len']),
        out_len=int(config['out_len']),
        in_dim=int(config['in_dim']),
        out_dim=int(config['hidden_dim']),
        n_in_blocks=int(config['n_in_blocks']),
        n_out_blocks=int(config['n_out_blocks']),
        num_heads=int(config['num_heads']),
        mlp_ratio=int(config['mlp_ratio']),
        dropout=float(config['dropout']),
        use_channel_weights=bool(config['use_channel_weights']),
    )


In [ ]:
# =============================================================
# ProbeSystem: CLIP + AR + EncDec KV (EEG faithfulness)
# =============================================================

def set_seed(seed: int = 2026) -> None:
    """Seed Python/NumPy/PyTorch RNGs (CPU and CUDA).

    Args:
        seed: Integer seed written to all available generators.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def move_to_device(batch: Dict[str, Any], device: torch.device) -> Dict[str, Any]:
    """Copy tensor values in a batch dict onto ``device``; leave other fields as-is.

    Args:
        batch: Collated sample dict.
        device: Destination torch device.

    Returns:
        Shallow-copied batch with tensors moved via ``non_blocking=True``.
    """
    out: Dict[str, Any] = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out


def pool_seq(x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Pool (B, L, D) -> (B, D); optional mask (B, L)."""
    assert x.ndim == 3
    if mask is None:
        return x.mean(dim=1)
    m = mask.bool().unsqueeze(-1).float()
    return (x * m).sum(dim=1) / m.sum(dim=1).clamp_min(1.0)


def clip_info_nce(
    ei: torch.Tensor,
    yi: torch.Tensor,
    sample_weights: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """SemKey symmetric CLIP InfoNCE on pooled embeddings."""
    assert ei.ndim == 2 and yi.ndim == 2 and ei.shape == yi.shape
    bsz = int(ei.shape[0])
    assert bsz >= 1
    x_normed = ei / ei.norm(dim=1, keepdim=True).clamp_min(1e-8)
    y_normed = yi / yi.norm(dim=1, keepdim=True).clamp_min(1e-8)
    x_logits = x_normed @ y_normed.T
    y_logits = x_logits.T
    target = torch.arange(bsz, device=ei.device)
    x_logits = F.normalize(x_logits, p=2, dim=-1)
    y_logits = F.normalize(y_logits, p=2, dim=-1)
    per = (
        F.cross_entropy(x_logits, target, reduction='none')
        + F.cross_entropy(y_logits, target, reduction='none')
    ) / 2.0
    if sample_weights is None:
        loss = per.mean()
    else:
        w = sample_weights.to(device=ei.device, dtype=per.dtype).view(-1)
        assert w.shape[0] == bsz
        denom = w.sum().clamp_min(1e-8)
        loss = (per * w).sum() / denom
    assert not torch.isnan(loss).any()
    return loss


def token_commitment_mse(
    z_src: torch.Tensor,
    h_tgt: torch.Tensor,
    text_mask: torch.Tensor,
) -> torch.Tensor:
    """Normalized token MSE reduced over valid `(B,L,D)` elements only."""
    assert z_src.ndim == 3 and h_tgt.ndim == 3
    assert z_src.shape == h_tgt.shape
    assert text_mask.shape == z_src.shape[:2]
    assert bool(text_mask.bool().any(dim=1).all()), 'every row needs a valid text token'
    x = F.normalize(z_src, p=2, dim=-1)
    y = F.normalize(h_tgt, p=2, dim=-1)
    valid = text_mask.bool().unsqueeze(-1).expand_as(x)
    squared = (x - y).square()
    loss = squared.masked_select(valid).mean()
    assert torch.isfinite(loss), 'non-finite commitment loss'
    return loss


def setup_scheduler(
    optimizer: torch.optim.Optimizer, config: Dict[str, Any],
) -> CosineAnnealingWarmRestarts:
    """Epoch-level cosine annealing with warm restarts.

    Args:
        optimizer: Optimizer whose LR is scheduled.
        config: Uses ``lr_restart_epochs`` (T_0) and ``min_lr``.

    Returns:
        ``CosineAnnealingWarmRestarts`` with ``T_mult=1``.
    """
    return CosineAnnealingWarmRestarts(
        optimizer,
        T_0=int(config.get('lr_restart_epochs', 15)),
        T_mult=1,
        eta_min=float(config.get('min_lr', 1e-9)),
    )


def _print_disk_usage(path: Optional[str]) -> None:
    """Print free/total disk space (GB) for path or its nearest existing parent."""
    if not path:
        print('  [disk] (no cache path)')
        return
    p = Path(path).resolve()
    probe = p if p.exists() else p.parent
    while not probe.exists() and probe != probe.parent:
        probe = probe.parent
    try:
        usage = shutil.disk_usage(str(probe))
        free_gb = usage.free / (1024 ** 3)
        total_gb = usage.total / (1024 ** 3)
        print(f'  [disk] {probe}: free={free_gb:.1f} GB / total={total_gb:.1f} GB')
    except Exception as exc:
        print(f'  [disk] unavailable for {probe}: {exc}')


def _hub_snapshot_dir(repo_id: str, cache_dir: Optional[str]) -> Optional[Path]:
    """Return a complete local Hub snapshot, or None if missing/incomplete."""
    name = 'models--' + repo_id.replace('/', '--')
    tok_markers = (
        'tokenizer.json', 'vocab.json', 'spiece.model',
        'tokenizer.model', 'merges.txt',
    )
    roots: List[Path] = []
    if cache_dir:
        roots.append(Path(cache_dir))
    for env_key in ('HUGGINGFACE_HUB_CACHE', 'HF_HUB_CACHE'):
        v = os.environ.get(env_key)
        if v:
            roots.append(Path(v))
    roots.append(Path.home() / '.cache' / 'huggingface' / 'hub')
    roots.append(Path.home() / '.cache' / 'huggingface')

    seen: set[str] = set()
    for root in roots:
        key = str(root.resolve()) if root.exists() else str(root)
        if key in seen:
            continue
        seen.add(key)
        for base in (root / name, root / 'hub' / name):
            snaps = base / 'snapshots'
            if not snaps.is_dir():
                continue
            for snap in snaps.iterdir():
                if not snap.is_dir():
                    continue
                if not (snap / 'config.json').is_file():
                    continue
                if list(snap.glob('*.incomplete')):
                    continue
                if not any((snap / m).is_file() for m in tok_markers):
                    continue
                has_weights = (
                    any(snap.glob('*.safetensors'))
                    or any(snap.glob('pytorch_model*.bin'))
                    or any(snap.glob('model.safetensors.index.json'))
                )
                if not has_weights:
                    continue
                return snap
    return None


def download_bart_backbone_safetensors(
    model_id: str = 'AI-ModelScope/bart-large',
    cache_dir: Optional[str] = None,
    max_retries: int = 5,
    backoff_s: float = 30.0,
) -> str:
    """Download BART via ModelScope and ensure model.safetensors exists (one-time convert)."""
    from modelscope import snapshot_download  # lazy import keeps encoder smoke offline

    n_retry = max(1, int(max_retries))
    backoff = float(backoff_s)
    last_exc: Optional[BaseException] = None
    model_dir: Optional[str] = None
    for attempt in range(1, n_retry + 1):
        try:
            print(f'  [ModelScope] snapshot_download({model_id}) (attempt {attempt}/{n_retry})')
            model_dir = snapshot_download(
                model_id,
                cache_dir=cache_dir,
                allow_file_pattern=['*.json', '*.txt', 'pytorch_model.bin', '*.safetensors'],
            )
            break
        except Exception as exc:
            last_exc = exc
            print(f'  [retry {attempt}/{n_retry}] ModelScope failed for {model_id}: {exc}')
            if attempt < n_retry:
                sleep_s = backoff * float(attempt)
                print(f'  [retry] sleeping {sleep_s:.1f}s before next attempt ...')
                time.sleep(sleep_s)
    if model_dir is None:
        raise RuntimeError(
            f'Failed to download {model_id} from ModelScope after {n_retry} attempts'
        ) from last_exc

    safetensors_path = Path(model_dir) / 'model.safetensors'
    if not safetensors_path.exists():
        import safetensors.torch  # lazy import

        bin_path = Path(model_dir) / 'pytorch_model.bin'
        assert bin_path.is_file(), f'Missing checkpoint: {bin_path}'
        try:
            state_dict = torch.load(bin_path, map_location='cpu', weights_only=True)
        except TypeError:
            state_dict = torch.load(bin_path, map_location='cpu')
        tensors = {key: value for key, value in state_dict.items() if isinstance(value, torch.Tensor)}
        safetensors.torch.save_file(tensors, str(safetensors_path))
        print(f'  [ModelScope] wrote {safetensors_path}')
    return str(model_dir)


def load_frozen_llm(
    repo_id: str,
    family: str,
    cache_dir: Optional[str],
    device: torch.device,
    max_retries: int = 5,
    backoff_s: float = 30.0,
    source: str = 'hf_mirror',
    modelscope_id: Optional[str] = None,
) -> Tuple[Any, Any, int, str]:
    """
    Load tokenizer + frozen EncDec LLM.

    Flan-T5: Hub/HF-mirror retries. BART: ModelScope snapshot + local safetensors.

    Returns:
        tokenizer, model, hidden_dim, family
    """
    _print_disk_usage(cache_dir)
    use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    n_retry = max(1, int(max_retries))
    backoff = float(backoff_s)

    def _load_t5(load_id: Union[str, Path], local_only: bool) -> Tuple[Any, Any, int]:
        """Load Flan-T5 tokenizer + model from a Hub id or local snapshot.

        Args:
            load_id: Hugging Face repo id or local directory.
            local_only: If True, do not hit the network.

        Returns:
            ``(tokenizer, model, hidden_dim)``.
        """
        kwargs: Dict[str, Any] = {'use_safetensors': True}
        if cache_dir:
            kwargs['cache_dir'] = cache_dir
        if local_only:
            kwargs['local_files_only'] = True
        tok = AutoTokenizer.from_pretrained(load_id, **kwargs)
        model = T5ForConditionalGeneration.from_pretrained(load_id, dtype=dtype, **kwargs)
        hidden = int(model.config.d_model)
        return tok, model, hidden

    def _load_hub_t5_with_retry() -> Tuple[Any, Any, int]:
        """Download Flan-T5 from the Hub/mirror with exponential backoff.

        Returns:
            ``(tokenizer, model, hidden_dim)``.
        """
        last_exc: Optional[BaseException] = None
        for attempt in range(1, n_retry + 1):
            try:
                print(f'  Loading LLM from Hub/mirror: {repo_id} (attempt {attempt}/{n_retry})')
                return _load_t5(repo_id, local_only=False)
            except Exception as exc:
                last_exc = exc
                print(f'  [retry {attempt}/{n_retry}] Hub load failed for {repo_id}: {exc}')
                if attempt < n_retry:
                    sleep_s = backoff * float(attempt)
                    print(f'  [retry] sleeping {sleep_s:.1f}s before next attempt ...')
                    time.sleep(sleep_s)
        raise RuntimeError(
            f'Failed to load {repo_id} after {n_retry} Hub attempts'
        ) from last_exc

    if family == 'encdec_bart' or source == 'modelscope':
        ms_id = modelscope_id or 'AI-ModelScope/bart-large'
        model_dir = download_bart_backbone_safetensors(
            ms_id, cache_dir=cache_dir, max_retries=n_retry, backoff_s=backoff,
        )
        print(f'  Loading BART from ModelScope dir: {model_dir}')
        tok = BartTokenizer.from_pretrained(model_dir)
        model = BartForConditionalGeneration.from_pretrained(
            model_dir, dtype=dtype, use_safetensors=True,
        )
        hidden = int(model.config.d_model)
        family = 'encdec_bart'
    elif family == 'encdec_t5':
        local = _hub_snapshot_dir(repo_id, cache_dir)
        if local is not None:
            print(f'  Loading LLM from local snapshot: {local}')
            try:
                tok, model, hidden = _load_t5(local, local_only=True)
            except Exception as exc:
                print(f'  [WARN] local snapshot load failed ({exc}); falling back to Hub id')
                tok, model, hidden = _load_hub_t5_with_retry()
        else:
            tok, model, hidden = _load_hub_t5_with_retry()
    else:
        raise ValueError(f'Unknown family/source: family={family} source={source}')

    model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    print(f'  Frozen LLM {repo_id} family={family} hidden={hidden} dtype={dtype}')
    return tok, model, hidden, family


def _patch_tuned_lens_final_norm() -> None:
    """Teach official Unembed EncDec decoder LN (T5) or identity (BART-large)."""
    ms = tuned_lens_model_surgery
    original = getattr(ms, '_eeg_faith_original_get_final_norm', None)
    if original is None:
        original = ms.get_final_norm
        ms._eeg_faith_original_get_final_norm = original  # type: ignore[attr-defined]

    def get_final_norm(model: Any) -> nn.Module:
        """Return the decoder final LayerNorm, or Identity for BART-large.

        Args:
            model: Frozen EncDec Hugging Face model.

        Returns:
            ``nn.Module`` used by TunedLens Unembed as the last norm.
        """
        decoder = None
        if hasattr(model, 'get_decoder'):
            try:
                decoder = model.get_decoder()
            except Exception:
                decoder = None
        if decoder is None and hasattr(model, 'model') and hasattr(model.model, 'decoder'):
            decoder = model.model.decoder
        if decoder is None and hasattr(model, 'decoder'):
            decoder = model.decoder
        if decoder is not None:
            ln = getattr(decoder, 'final_layer_norm', None)
            if ln is not None:
                return ln
            ln = getattr(decoder, 'layer_norm', None)
            if ln is not None:
                return ln
            return nn.Identity()
        return original(model)

    get_final_norm._eeg_faith_encdec = True  # type: ignore[attr-defined]
    ms.get_final_norm = get_final_norm


def build_encdec_unembed(model: 'ProbeSystem') -> Unembed:
    """
    Official Unembed with EncDec extras baked in.

    T5: tied-embedding scale d_model^{-1/2} on the copied lm_head.
    BART: add final_logits_bias onto the copied unembed bias.
    """
    _patch_tuned_lens_final_norm()
    unembed = Unembed(model.llm)
    if model.family == 'encdec_t5':
        scale = float(model._t5_tied_scale())
        if abs(scale - 1.0) > 1e-12:
            unembed.unembedding.weight.data.mul_(scale)
            if unembed.unembedding.bias is not None:
                unembed.unembedding.bias.data.mul_(scale)
    elif model.family == 'encdec_bart':
        bias = getattr(model.llm, 'final_logits_bias', None)
        if bias is not None:
            lin = unembed.unembedding
            b = bias.detach().reshape(-1).to(
                device=lin.weight.device, dtype=lin.weight.dtype,
            )
            new_lin = nn.Linear(lin.in_features, lin.out_features, bias=True)
            new_lin = new_lin.to(device=lin.weight.device, dtype=lin.weight.dtype)
            with torch.no_grad():
                new_lin.weight.copy_(lin.weight)
                if lin.bias is not None:
                    new_lin.bias.copy_(lin.bias + b)
                else:
                    new_lin.bias.copy_(b)
            unembed.unembedding = new_lin
    unembed.requires_grad_(False)
    unembed.eval()
    return unembed


def build_tuned_lens(model: 'ProbeSystem', n_translators: int) -> TunedLens:
    """TunedLens with decoder residual depth (not T5 config.num_hidden_layers)."""
    n_tr = int(n_translators)
    assert n_tr >= 1, n_tr
    unembed = build_encdec_unembed(model)
    cfg = TunedLensConfig(
        base_model_name_or_path=str(model.repo_id),
        d_model=int(model.embed_dim),
        num_hidden_layers=n_tr,
        bias=True,
    )
    lens = TunedLens(unembed, cfg)
    assert len(lens.layer_translators) == n_tr
    return lens


def hidden_for_lens(lens: TunedLens, h: torch.Tensor) -> torch.Tensor:
    """Cast decoder hidden to Unembed/translator dtype (bf16 when the LLM is bf16)."""
    w = lens.unembed.unembedding.weight
    out = h.to(device=w.device, dtype=w.dtype)
    assert out.shape == h.shape, (tuple(out.shape), tuple(h.shape))
    return out


def kl_tuned_lens_layer(
    lens: TunedLens,
    h: torch.Tensor,
    idx: int,
    log_p: torch.Tensor,
) -> torch.Tensor:
    """
    KL(p_final || q_lens) at decoder position -1 (Belrose et al.).

    h: (B, D), log_p: (B, V) log-softmax of last-token logits.
    """
    assert h.ndim == 2 and log_p.ndim == 2, (tuple(h.shape), tuple(log_p.shape))
    assert int(h.shape[0]) == int(log_p.shape[0]), (tuple(h.shape), tuple(log_p.shape))
    assert 0 <= int(idx) < len(lens.layer_translators)
    log_q = lens.forward(hidden_for_lens(lens, h), int(idx)).float().log_softmax(dim=-1)
    assert log_q.shape == log_p.shape
    kl = (log_p.exp() * (log_p - log_q)).sum(dim=-1)
    loss = kl.mean()
    assert torch.isfinite(loss), f'non-finite tuned-lens KL at idx={idx}'
    return loss


def canonical_decoder_prefix_ids(
    family: str,
    decoder_start_id: int,
    bart_bos_id: Optional[int],
    instruction_ids: Sequence[int],
) -> List[int]:
    """Canonical instruction prefix shared by teacher, rollout, and analyses."""
    ids = [int(decoder_start_id)]
    if family == 'encdec_bart':
        assert bart_bos_id is not None, 'BART requires an explicit BOS after decoder_start'
        ids.append(int(bart_bos_id))
    else:
        assert family == 'encdec_t5' and bart_bos_id is None
    ids.extend(int(token_id) for token_id in instruction_ids)
    return ids


def lexical_text_after_instruction(text: str, family: str) -> str:
    """Preserve a word boundary for independently tokenized BART lexical text."""
    value = str(text)
    if family == 'encdec_bart' and value:
        return ' ' + value.lstrip()
    assert family in {'encdec_bart', 'encdec_t5'}
    return value


def full_target_token_count(target_ids: Sequence[Sequence[int]]) -> int:
    """Count every lexical/EOS target token, independent of predicted EOS."""
    count = sum(len(row) for row in target_ids)
    assert count >= len(target_ids)
    return int(count)


class ProbeSystem(nn.Module):
    """Independent EEG path plus frozen LLM and analysis-only oracle memory."""

    def __init__(
        self,
        config: Dict[str, Any],
        llm_key: str,
        repo_id: str,
        family: str,
        device: torch.device,
        source: str = 'hf_mirror',
        modelscope_id: Optional[str] = None,
    ) -> None:
        """Load a frozen EncDec LLM and randomly init the trainable EEG path.

        Args:
            config: Notebook CONFIG (encoder sizes, loss weights, cache).
            llm_key: Short id such as ``flan_t5_large``.
            repo_id: Hugging Face repo id.
            family: ``encdec_bart`` or ``encdec_t5``.
            device: Compute device.
            source: ``hf_mirror`` or ``modelscope``.
            modelscope_id: ModelScope snapshot id when ``source='modelscope'``.
        """
        super().__init__()
        self.config = config
        self.llm_key = llm_key
        self.repo_id = repo_id
        self.family = family
        self.source = str(source)
        self.modelscope_id = None if modelscope_id is None else str(modelscope_id)
        self.device = device
        self.data_key = 'eeg'
        cache_dir = config.get('model_cache_dir')
        self.tokenizer, self.llm, embed_dim, self.family = load_frozen_llm(
            repo_id,
            family,
            cache_dir,
            device,
            max_retries=int(config.get('llm_load_max_retries', 5)),
            backoff_s=float(config.get('llm_load_retry_backoff_s', 30.0)),
            source=source,
            modelscope_id=modelscope_id,
        )
        self.embed_dim = int(embed_dim)
        self.model_dtype = next(self.llm.parameters()).dtype

        self.max_target_tokens = int(config.get('max_target_tokens', 64))
        self.out_len = int(config.get('out_len', 96))
        self.input_text_len = int(config.get('input_text_len', self.out_len))
        assert self.input_text_len == self.out_len
        self.commitment_weight = float(config.get('commitment_weight', 0.7))
        self.w_clip = float(config.get('w_clip', 0.5))
        self.w_ar = float(config.get('w_ar', 0.5))
        self.use_ei = bool(config.get('use_ei', True))
        self.decoder_prompt = str(config.get(
            'decoder_prompt',
            'Based on the following signals, translate the sentence',
        ))
        self._build_trainable()

    def _build_trainable(self) -> None:
        """(Re)initialize prompt embedder, SemKey encoder, in_proj, identity projector."""
        cfg = self.config
        self.prompt_embedder = PromptEmbedder(
            dim=int(cfg['prompt_dim']),
            prompt_keys=PROMPT_KEYS,
            drop_probs=tuple(cfg.get('prompt_drop_probs', (0.0, 0.0, 0.0))),
        )
        self.eeg_encoder = build_eeg_encoder(cfg)
        self.in_proj = nn.Linear(int(cfg['hidden_dim']), self.embed_dim)
        self.projector = nn.Linear(self.embed_dim, self.embed_dim)
        with torch.no_grad():
            self.projector.weight.copy_(torch.eye(self.embed_dim))
            self.projector.bias.zero_()

    def reset_trainable(self) -> None:
        """Fresh encoder/projector weights for the next data_key; LLM stays frozen."""
        set_seed(int(self.config.get('seed', 2026)))
        self._build_trainable()
        self.to(self.device)
        self.llm.eval()
        for p in self.llm.parameters():
            p.requires_grad = False

    def set_data_key(self, data_key: str) -> None:
        """Freeze EEG modules for analysis-only oracle memory; train only `eeg`."""
        assert data_key in DATA_KEY_ALPHA, data_key
        self.data_key = str(data_key)
        train_enc = data_key == 'eeg'
        for mod in (self.prompt_embedder, self.eeg_encoder, self.in_proj, self.projector):
            for p in mod.parameters():
                p.requires_grad = train_enc

    def trainable_parameters(self) -> List[nn.Parameter]:
        """Collect encoder / projector parameters that currently require grad.

        Returns:
            List of ``nn.Parameter`` (empty for analysis-only ``oracle_text``).
        """
        params: List[nn.Parameter] = []
        for mod in (self.prompt_embedder, self.eeg_encoder, self.in_proj, self.projector):
            params.extend([p for p in mod.parameters() if p.requires_grad])
        return params

    def encode_prompts(self, batch: Dict[str, Any]) -> torch.Tensor:
        """Convert batch prompt tuples into integer ids for ``PromptEmbedder``.

        Args:
            batch: Must contain ``prompt`` as a list of ``(task, dataset, subject)``.

        Returns:
            LongTensor ``(B, 3)`` on ``self.device``.
        """
        prompts = batch['prompt']
        tasks, datasets, subjects = unpack_prompt_batch(prompts)
        packed = [tasks, datasets, subjects]
        return self.prompt_embedder.encode(packed, device=self.device)

    def encode_eeg(
        self, batch: Dict[str, Any],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            z_src: (B, out_len, D)
            ei_eeg: (B, D)
        """
        eeg = batch['eeg']
        mask = batch['mask']
        assert eeg.ndim == 3 and tuple(eeg.shape[1:]) == (1280, 128), tuple(eeg.shape)
        assert mask.shape == eeg.shape[:2], (tuple(mask.shape), tuple(eeg.shape))
        assert torch.isfinite(eeg).all() and torch.isfinite(mask.float()).all()
        assert bool(((mask == 0) | (mask == 1)).all()), 'EEG mask must be binary'
        assert bool(mask.bool().any(dim=1).all()), 'every EEG row needs a valid sample'
        p_ids = self.encode_prompts(batch)
        p = self.prompt_embedder(p_ids)
        Zi, _, _ = self.eeg_encoder(eeg, mask, p)
        assert Zi.shape[:2] == (eeg.shape[0], self.out_len)
        z_src = self.projector(self.in_proj(Zi))
        ei_eeg = z_src.mean(dim=1)
        assert z_src.shape[-1] == self.embed_dim
        assert ei_eeg.shape == (z_src.shape[0], self.embed_dim)
        return z_src, ei_eeg

    @torch.no_grad()
    def encode_text(
        self, texts: List[str],
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
            h_tgt: (B, input_text_len, D)
            text_mask: (B, input_text_len)
            ei_text: (B, D)
        """
        prev_side = getattr(self.tokenizer, 'padding_side', 'right')
        self.tokenizer.padding_side = 'right'
        try:
            tok = self.tokenizer(
                texts,
                padding='max_length',
                truncation=True,
                max_length=self.input_text_len,
                return_tensors='pt',
            )
        finally:
            self.tokenizer.padding_side = prev_side
        input_ids = tok['input_ids'].to(self.device)
        attn = tok['attention_mask'].to(self.device)
        assert input_ids.shape[1] == self.input_text_len
        enc = self.llm.get_encoder()
        out = enc(input_ids=input_ids, attention_mask=attn, return_dict=True)
        h = out.last_hidden_state.to(dtype=torch.float32)
        assert h.shape[:2] == input_ids.shape
        ei = pool_seq(h, attn)
        return h, attn, ei

    def encode_kv(
        self,
        batch: Dict[str, Any],
        data_key: str,
        mix_seed: Optional[int] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Build EEG or analysis-only oracle full-target memory and masks.

        Args:
            batch: Collated sample dict.
            data_key: ``oracle_text`` (frozen encoder states) or ``eeg``.
            mix_seed: Unused; kept so callers that pass a seed stay compatible.
                This notebook never mixes EEG with Gaussian noise.

        Returns:
            For ``oracle_text``: ``(h_tgt, ei_text, text_mask, h_tgt, text_mask, ei_text)``.
            For ``eeg``: ``(z_src, ei_kv, kv_mask, h_tgt, text_mask, ei_text)``.
        """
        del mix_seed
        assert data_key in DATA_KEY_ALPHA, data_key
        texts = [str(t) for t in batch.get('target text', batch['input text'])]
        h_tgt, text_mask, ei_text = self.encode_text(texts)
        if data_key == 'oracle_text':
            return h_tgt, ei_text, text_mask, h_tgt, text_mask, ei_text
        z_src, _ei = self.encode_eeg(batch)
        alpha = DATA_KEY_ALPHA[data_key]
        assert alpha is not None and float(alpha) == 0.0
        ei_kv = z_src.mean(dim=1)
        kv_mask = torch.ones(z_src.shape[:2], dtype=torch.long, device=z_src.device)
        assert z_src.shape[1] == h_tgt.shape[1] == self.out_len
        return z_src, ei_kv, kv_mask, h_tgt, text_mask, ei_text

    def build_decoder_memory(
        self,
        z_src: torch.Tensor,
        ei: Optional[torch.Tensor] = None,
        source_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Build EncDec memory while preserving oracle text padding."""
        assert z_src.ndim == 3
        if source_mask is None:
            enc_mask = torch.ones(z_src.shape[:2], device=z_src.device, dtype=torch.long)
        else:
            assert source_mask.shape == z_src.shape[:2]
            enc_mask = source_mask.to(device=z_src.device, dtype=torch.long)
            assert bool(((enc_mask == 0) | (enc_mask == 1)).all())
            assert bool(enc_mask.bool().any(dim=1).all())
        raw = z_src
        if self.use_ei:
            assert ei is not None and ei.shape == (z_src.shape[0], z_src.shape[2])
            raw = torch.cat([ei.unsqueeze(1), raw], dim=1)
            pooled_mask = torch.ones((z_src.shape[0], 1), device=z_src.device, dtype=torch.long)
            enc_mask = torch.cat([pooled_mask, enc_mask], dim=1)
        assert enc_mask.shape == raw.shape[:2]
        return raw.to(self.model_dtype), enc_mask

    def _instruction_ids(self) -> List[int]:
        """Token ids for CONFIG decoder_prompt (no BOS/EOS)."""
        text = str(self.decoder_prompt).strip()
        if not text:
            return []
        start = self._decoder_start_id()
        eos = self.tokenizer.eos_token_id
        ids = list(self.tokenizer(
            text, add_special_tokens=False, padding=False, truncation=True,
            max_length=self.max_target_tokens,
        )['input_ids'])
        if ids and eos is not None and ids[-1] == int(eos):
            ids = ids[:-1]
        if ids and start is not None and ids[0] == int(start):
            ids = ids[1:]
        bart_bos = self._bart_bos_id()
        if ids and bart_bos is not None and ids[0] == int(bart_bos):
            ids = ids[1:]
        return ids

    def _teacher_forced_ar(
        self,
        batch: Dict[str, Any],
        z_src: torch.Tensor,
        ei_kv: torch.Tensor,
        source_mask: torch.Tensor,
    ) -> torch.Tensor:
        """Teacher-forced target CE under the canonical manual decoder order."""
        enc_hidden, enc_mask = self.build_decoder_memory(z_src, ei_kv, source_mask)
        B = int(z_src.shape[0])
        pad = int(self.tokenizer.pad_token_id)
        prompt_ids_list, prompt_lens, target_ids_list = self._tokenize_prompt_target_ids(batch)
        merged: List[List[int]] = [p + t for p, t in zip(prompt_ids_list, target_ids_list)]
        max_len = max(len(ids) for ids in merged)
        labels = torch.full((B, max_len), -100, dtype=torch.long, device=self.device)
        full = torch.full((B, max_len), pad, dtype=torch.long, device=self.device)
        valid_full = torch.zeros((B, max_len), dtype=torch.long, device=self.device)
        for i, ids in enumerate(merged):
            row = torch.tensor(ids, dtype=torch.long, device=self.device)
            labels[i, :len(ids)] = row
            full[i, :len(ids)] = row
            valid_full[i, :len(ids)] = 1
        for i, plen in enumerate(prompt_lens):
            labels[i, :plen] = -100
        start_id = self._decoder_start_id()
        assert start_id is not None
        start = torch.full((B, 1), int(start_id), dtype=torch.long, device=self.device)
        decoder_input_ids = torch.cat([start, full[:, :-1]], dim=1)
        decoder_attention_mask = torch.cat([
            torch.ones((B, 1), dtype=torch.long, device=self.device), valid_full[:, :-1],
        ], dim=1)
        assert decoder_input_ids.shape == decoder_attention_mask.shape == labels.shape
        if self.family == 'encdec_t5':
            outputs = self.llm(
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                encoder_outputs=(enc_hidden,),
                attention_mask=enc_mask,
                labels=labels,
                return_dict=True,
            )
        else:
            outputs = self.llm(
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                encoder_outputs=BaseModelOutput(last_hidden_state=enc_hidden),
                attention_mask=enc_mask,
                labels=labels,
                return_dict=True,
            )
        assert outputs.loss is not None
        loss_ar = outputs.loss
        assert loss_ar.ndim == 0
        assert torch.isfinite(loss_ar), f'non-finite L_ar: {loss_ar}'
        return loss_ar

    @torch.no_grad()
    def _free_run_ar_ce(
        self,
        batch: Dict[str, Any],
        z_src: torch.Tensor,
        ei_kv: torch.Tensor,
        source_mask: torch.Tensor,
        return_trace: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
        """Display-only greedy rollout CE over every gold lexical/EOS token."""
        enc_hidden, enc_mask = self.build_decoder_memory(z_src, ei_kv, source_mask)
        B = int(z_src.shape[0])
        pad = int(self.tokenizer.pad_token_id)
        start_id = self._decoder_start_id()
        assert start_id is not None
        prompt_ids_list, _prompt_lens, target_ids_list = self._tokenize_prompt_target_ids(batch)
        max_tgt = max((len(t) for t in target_ids_list), default=0)
        if max_tgt <= 0:
            zero = torch.zeros((), device=self.device, dtype=torch.float32)
            if return_trace:
                z1 = torch.full((B,), pad, dtype=torch.long, device=self.device)
                return zero, z1, z1
            return zero
        labels = torch.full((B, max_tgt), -100, dtype=torch.long, device=self.device)
        for i, t_ids in enumerate(target_ids_list):
            labels[i, :len(t_ids)] = torch.tensor(t_ids, dtype=torch.long, device=self.device)
        prefixes: List[List[int]] = [[int(start_id)] + list(p) for p in prompt_ids_list]
        max_pref = max(len(p) for p in prefixes)
        dec = torch.full((B, max_pref), pad, dtype=torch.long, device=self.device)
        dec_attn = torch.zeros((B, max_pref), dtype=torch.long, device=self.device)
        for i, p in enumerate(prefixes):
            dec[i, max_pref - len(p):] = torch.tensor(p, dtype=torch.long, device=self.device)
            dec_attn[i, max_pref - len(p):] = 1
        enc_out = BaseModelOutput(last_hidden_state=enc_hidden)
        ce_sum = torch.zeros((), device=self.device, dtype=torch.float32)
        n_tok = torch.zeros((), device=self.device, dtype=torch.float32)
        first_argmax: Optional[torch.Tensor] = None
        first_gold = labels[:, 0]

        for t in range(max_tgt):
            if self.family == 'encdec_t5':
                outputs = self.llm(
                    decoder_input_ids=dec,
                    decoder_attention_mask=dec_attn,
                    encoder_outputs=(enc_hidden,),
                    attention_mask=enc_mask,
                    return_dict=True,
                    use_cache=False,
                )
            else:
                outputs = self.llm(
                    decoder_input_ids=dec,
                    decoder_attention_mask=dec_attn,
                    encoder_outputs=enc_out,
                    attention_mask=enc_mask,
                    return_dict=True,
                    use_cache=False,
                )
            logits = outputs.logits[:, -1, :]
            assert torch.isfinite(logits.float()).all(), 'non-finite rollout logits'
            gold = labels[:, t]
            valid = gold.ne(-100)
            if bool(valid.any()):
                logp = F.log_softmax(logits.float(), dim=-1)
                ce_sum = ce_sum + F.nll_loss(logp[valid], gold[valid], reduction='sum')
                n_tok = n_tok + valid.to(dtype=torch.float32).sum()
            next_tok = logits.argmax(dim=-1)
            if first_argmax is None:
                first_argmax = next_tok.detach()
            dec = torch.cat([dec, next_tok.unsqueeze(1)], dim=1)
            dec_attn = torch.cat([
                dec_attn, torch.ones((B, 1), dtype=torch.long, device=self.device),
            ], dim=1)

        assert first_argmax is not None
        assert int(n_tok.item()) == full_target_token_count(target_ids_list)
        if float(n_tok.detach().cpu()) <= 0:
            loss_ar = torch.zeros((), device=self.device, dtype=torch.float32)
        else:
            loss_ar = ce_sum / n_tok.clamp_min(1.0)
        assert loss_ar.ndim == 0
        assert torch.isfinite(loss_ar), f'non-finite rollout diagnostic CE: {loss_ar}'
        if return_trace:
            return loss_ar, first_argmax, first_gold
        return loss_ar

    def forward_e2e(
        self,
        batch: Dict[str, Any],
        data_key: str,
        mix_seed: Optional[int] = None,
    ) -> Dict[str, torch.Tensor]:
        """Teacher-forced SemKey-style composite used identically for train and val."""
        z_src, ei_kv, kv_mask, h_tgt, text_mask, ei_text = self.encode_kv(
            batch, data_key, mix_seed=mix_seed,
        )
        loss_clip = clip_info_nce(ei_kv, ei_text.detach())
        loss_commit = token_commitment_mse(z_src, h_tgt.detach(), text_mask)
        loss_ar = self._teacher_forced_ar(batch, z_src, ei_kv, kv_mask)
        weighted_clip = self.w_clip * loss_clip
        weighted_ar = self.w_ar * loss_ar
        weighted_commit = self.commitment_weight * loss_commit
        total = weighted_clip + weighted_ar + weighted_commit
        _prompts, _prompt_lens, targets = self._tokenize_prompt_target_ids(batch)
        n_ar_tokens = full_target_token_count(targets)
        n_commit_elements = int(text_mask.bool().sum().item()) * int(z_src.shape[-1])
        with torch.no_grad():
            cos = F.cosine_similarity(
                F.normalize(ei_kv, dim=-1), F.normalize(ei_text, dim=-1), dim=-1,
            ).mean()
        return {
            'loss': total,
            'loss_clip': loss_clip.detach(),
            'loss_commit': loss_commit.detach(),
            'loss_ar': loss_ar.detach(),
            'weighted_clip': weighted_clip.detach(),
            'weighted_ar': weighted_ar.detach(),
            'weighted_commit': weighted_commit.detach(),
            'cos': cos,
            'n_samples': torch.tensor(int(z_src.shape[0]), device=z_src.device),
            'n_ar_tokens': torch.tensor(n_ar_tokens, device=z_src.device),
            'n_commit_elements': torch.tensor(n_commit_elements, device=z_src.device),
            'z_src': z_src,
            'ei_kv': ei_kv,
            'ei_text': ei_text.detach(),
        }

    @torch.no_grad()
    def rollout_ar_diagnostic(
        self,
        batch: Dict[str, Any],
        data_key: str,
        mix_seed: Optional[int] = None,
    ) -> Tuple[torch.Tensor, int]:
        """Return full-target greedy rollout CE and its gold-token denominator."""
        z_src, ei_kv, kv_mask, _h_tgt, _text_mask, _ei_text = self.encode_kv(
            batch, data_key, mix_seed=mix_seed,
        )
        loss = self._free_run_ar_ce(batch, z_src, ei_kv, kv_mask, return_trace=False)
        assert isinstance(loss, torch.Tensor)
        _prompts, _prompt_lens, targets = self._tokenize_prompt_target_ids(batch)
        return loss, full_target_token_count(targets)

    def _tokenize_prompt_target_ids(
        self, batch: Dict[str, Any],
    ) -> Tuple[List[List[int]], List[int], List[List[int]]]:
        """Masked prompt plus lexical IDs (`add_special_tokens=False`) and one EOS."""
        pad = self.tokenizer.pad_token_id
        eos = self.tokenizer.eos_token_id
        start = self._decoder_start_id()
        assert pad is not None and eos is not None and start is not None
        assert self.max_target_tokens >= 2
        texts = [str(t) for t in batch.get('target text', batch['input text'])]
        inst = self._instruction_ids()
        canonical_prefix = canonical_decoder_prefix_ids(
            self.family, int(start), self._bart_bos_id(), inst,
        )
        masked_prompt = canonical_prefix[1:]
        prompt_ids_list: List[List[int]] = []
        prompt_lens: List[int] = []
        target_ids_list: List[List[int]] = []
        for text in texts:
            lexical_text = lexical_text_after_instruction(text, self.family)
            lexical = [int(token_id) for token_id in self.tokenizer(
                lexical_text,
                add_special_tokens=False,
                padding=False,
                truncation=True,
                max_length=self.max_target_tokens - 1,
            )['input_ids']]
            lexical = [token_id for token_id in lexical if token_id != int(eos)]
            target = lexical + [int(eos)]
            assert target.count(int(eos)) == 1 and target[-1] == int(eos)
            assert len(target) <= self.max_target_tokens
            prompt_ids_list.append(list(masked_prompt))
            prompt_lens.append(len(masked_prompt))
            target_ids_list.append(target)
        return prompt_ids_list, prompt_lens, target_ids_list

    def _decoder_start_id(self) -> Optional[int]:
        """EncDec decoder_start / pad fallback."""
        pad = self.tokenizer.pad_token_id
        start = self.llm.config.decoder_start_token_id
        if start is None:
            return int(pad) if pad is not None else None
        return int(start)

    def _bart_bos_id(self) -> Optional[int]:
        """Return the explicit BART BOS; T5 has no BOS in this manual prefix."""
        if self.family != 'encdec_bart':
            return None
        bos = getattr(self.tokenizer, 'bos_token_id', None)
        assert bos is not None, 'BART tokenizer missing bos_token_id'
        return int(bos)

    def _special_token_ids(self) -> Set[int]:
        """Pad / BOS / EOS / decoder_start ids to skip when choosing t*."""
        ids: Set[int] = set()
        for tok_id in (
            self.tokenizer.pad_token_id,
            getattr(self.tokenizer, 'bos_token_id', None),
            self.tokenizer.eos_token_id,
            self._decoder_start_id(),
        ):
            if tok_id is not None:
                ids.add(int(tok_id))
        return ids

    def num_decoder_layers(self) -> int:
        """Number of transformer decoder blocks (not counting embeddings)."""
        cfg = self.llm.config
        n_dec = getattr(cfg, 'num_decoder_layers', None)
        if n_dec is not None:
            return int(n_dec)
        n_bart = getattr(cfg, 'decoder_layers', None)
        if n_bart is not None:
            return int(n_bart)
        return int(cfg.num_layers)

    def vocab_size(self) -> int:
        """LM head output dimension."""
        v = getattr(self.llm.config, 'vocab_size', None)
        if v is not None:
            return int(v)
        return int(self.llm.lm_head.out_features)

    def first_target_ids(self, batch: Dict[str, Any]) -> torch.Tensor:
        """First GT content token id t* per row (skip BOS/EOS/pad). Shape (B,)."""
        pad = int(self.tokenizer.pad_token_id)
        skip = self._special_token_ids()
        _p, _pl, target_ids_list = self._tokenize_prompt_target_ids(batch)
        ids: List[int] = []
        for t in target_ids_list:
            content = [int(x) for x in t if int(x) not in skip]
            ids.append(content[0] if content else pad)
        out = torch.tensor(ids, dtype=torch.long, device=self.device)
        assert out.ndim == 1 and int(out.shape[0]) == len(target_ids_list)
        return out

    def instruction_decoder_inputs(self, batch_size: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return the identical canonical instruction-only prefix for both memories."""
        start_id = self._decoder_start_id()
        assert start_id is not None
        ids = canonical_decoder_prefix_ids(
            self.family, int(start_id), self._bart_bos_id(), self._instruction_ids(),
        )
        B = int(batch_size)
        assert B >= 1 and len(ids) >= 1
        dec = torch.tensor([ids] * B, dtype=torch.long, device=self.device)
        attn = torch.ones((B, len(ids)), dtype=torch.long, device=self.device)
        assert dec.shape == attn.shape == (B, len(ids))
        return dec, attn

    def _t5_tied_scale(self) -> float:
        """T5 rescales hidden states before lm_head when embeddings are tied."""
        if self.family != 'encdec_t5':
            return 1.0
        if not bool(getattr(self.llm.config, 'tie_word_embeddings', False)):
            return 1.0
        dim = float(getattr(self.llm, 'model_dim', self.embed_dim))
        return dim ** -0.5

    def _llm_decoder_forward(
        self,
        decoder_input_ids: torch.Tensor,
        decoder_attention_mask: torch.Tensor,
        enc_hidden: torch.Tensor,
        enc_mask: torch.Tensor,
    ) -> Any:
        """One EncDec decoder pass with hidden states."""
        if self.family == 'encdec_t5':
            return self.llm(
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                encoder_outputs=(enc_hidden,),
                attention_mask=enc_mask,
                return_dict=True,
                use_cache=False,
                output_hidden_states=True,
            )
        enc_out = BaseModelOutput(last_hidden_state=enc_hidden)
        return self.llm(
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            encoder_outputs=enc_out,
            attention_mask=enc_mask,
            return_dict=True,
            use_cache=False,
            output_hidden_states=True,
        )

    @torch.no_grad()
    def decoder_states_at_minus1(
        self,
        batch: Dict[str, Any],
        data_key: str,
        mix_seed: Optional[int] = None,
    ) -> Dict[str, Any]:
        """
        Oracle-text or EEG memory -> decoder hidden states at position -1.

        Returns:
            hidden_m1: list of (B, D); index 0 is embeddings
            logits_hf: (B, V) official last-layer logits at -1
            t_star: (B,) first lexical target token ids
            n_blocks: decoder depth
        """
        self.eval()
        z_src, ei_kv, kv_mask, _h_tgt, _text_mask, ei_text = self.encode_kv(
            batch, data_key, mix_seed=mix_seed,
        )
        enc_hidden, enc_mask = self.build_decoder_memory(z_src, ei_kv, kv_mask)
        B = int(z_src.shape[0])
        dec, dec_attn = self.instruction_decoder_inputs(B)
        outputs = self._llm_decoder_forward(dec, dec_attn, enc_hidden, enc_mask)
        hs = outputs.decoder_hidden_states
        n_blocks = self.num_decoder_layers()
        assert hs is not None, 'decoder_hidden_states missing'
        assert len(hs) == n_blocks + 1, (
            f'expected embedding + {n_blocks} decoder block states, got {len(hs)}'
        )
        hidden_m1: List[torch.Tensor] = []
        for state_idx, h in enumerate(hs):
            assert h.ndim == 3 and h.shape[:2] == dec.shape, (state_idx, tuple(h.shape), tuple(dec.shape))
            hidden_m1.append(h[:, -1, :].float())
        logits_hf = outputs.logits[:, -1, :].float()
        t_star = self.first_target_ids(batch)
        assert logits_hf.shape == (B, self.vocab_size()) and t_star.shape == (B,)
        assert torch.isfinite(logits_hf).all(), 'non-finite official HF logits'
        return {
            'hidden_m1': hidden_m1,
            'logits_hf': logits_hf,
            't_star': t_star,
            'n_blocks': n_blocks,
            'z_src': z_src.detach(),
            'ei_kv': ei_kv.detach(),
            'ei_text': ei_text.detach(),
            'memory_mask': enc_mask.detach(),
            'decoder_input_ids': dec.detach(),
        }

    def unload_llm(self) -> None:
        """Drop the frozen LLM and tokenizer and free CUDA cache."""
        del self.llm
        del self.tokenizer
        self.llm = None  # type: ignore[assignment]
        self.tokenizer = None  # type: ignore[assignment]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def _logit_error_stats(prediction: torch.Tensor, reference: torch.Tensor) -> Tuple[float, float]:
    """Maximum absolute and elementwise relative logit error."""
    assert prediction.shape == reference.shape
    error = (prediction.float() - reference.float()).abs()
    max_abs = float(error.max().item())
    max_rel = float((error / reference.float().abs().clamp_min(1e-6)).max().item())
    return max_abs, max_rel


def select_matching_unembed_form(
    raw_logits: torch.Tensor,
    normalized_logits: torch.Tensor,
    official_logits: torch.Tensor,
    atol: float = 1e-1,
    rtol: float = 1e-3,
) -> Tuple[str, Dict[str, float]]:
    """Select raw or final-normalized Unembed form, failing if neither matches HF."""
    raw_abs, raw_rel = _logit_error_stats(raw_logits, official_logits)
    norm_abs, norm_rel = _logit_error_stats(normalized_logits, official_logits)
    raw_ok = bool(torch.allclose(raw_logits.float(), official_logits.float(), atol=atol, rtol=rtol))
    norm_ok = bool(torch.allclose(normalized_logits.float(), official_logits.float(), atol=atol, rtol=rtol))
    metrics = {
        'raw_max_abs': raw_abs,
        'raw_max_rel': raw_rel,
        'normalized_max_abs': norm_abs,
        'normalized_max_rel': norm_rel,
    }
    if raw_ok:
        return 'raw', metrics
    if norm_ok:
        return 'final_normalized', metrics
    raise AssertionError(f'Unembed matches neither official-logit form: {metrics}')


@torch.no_grad()
def verify_unembed_against_official(
    model: ProbeSystem,
    lens: TunedLens,
    pack: Dict[str, Any],
    stage: str,
) -> str:
    """Verify intermediate-stream Unembed against official final logits."""
    assert len(pack['hidden_m1']) == int(pack['n_blocks']) + 1
    h_last = hidden_for_lens(lens, pack['hidden_m1'][-1])
    raw = lens.unembed.unembedding(h_last).float()
    normalized = lens.unembed(h_last).float()
    official = pack['logits_hf'].float()
    form, metrics = select_matching_unembed_form(raw, normalized, official)
    print(
        f'  Unembed verification [{stage}] form={form} '
        f'raw abs/rel={metrics["raw_max_abs"]:.3e}/{metrics["raw_max_rel"]:.3e} '
        f'norm abs/rel={metrics["normalized_max_abs"]:.3e}/{metrics["normalized_max_rel"]:.3e}'
    )
    return form


def _assert_exact_state_keys(actual: Set[str], expected: Set[str], label: str) -> None:
    """Reject missing or unexpected keys instead of permissive checkpoint loading."""
    missing = sorted(expected - actual)
    unexpected = sorted(actual - expected)
    assert not missing and not unexpected, f'{label}: missing={missing[:5]} unexpected={unexpected[:5]}'


_EFFECTIVE_CHECKPOINT_CONFIG_KEYS: Tuple[str, ...] = (
    'in_len', 'in_dim', 'use_channel_weights', 'out_len', 'hidden_dim', 'prompt_dim',
    'n_in_blocks', 'n_out_blocks', 'num_heads', 'mlp_ratio', 'dropout',
    'prompt_drop_probs', 'use_ei', 'max_target_tokens', 'input_text_len',
    'w_clip', 'commitment_weight', 'w_ar', 'decoder_prompt',
)
_LOCAL_WEIGHT_HASH_CACHE: Dict[Tuple[str, Tuple[Tuple[str, int, int], ...]], str] = {}


def _sha256_json(value: Any) -> str:
    """Hash a deterministic UTF-8 JSON serialization."""
    serialized = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(',', ':'), default=str,
    ).encode('utf-8')
    return hashlib.sha256(serialized).hexdigest()


def _immutable_revision_digest(value: Any) -> Optional[str]:
    """Accept only immutable 40/64-character hexadecimal model revisions."""
    if not isinstance(value, str) or len(value) not in {40, 64}:
        return None
    if not all(char in '0123456789abcdefABCDEF' for char in value):
        return None
    return value.lower()


def _local_weight_sha256(resolved_path: str) -> str:
    """Hash local HF/ModelScope weight shards once per immutable file signature."""
    root = Path(resolved_path).resolve()
    assert root.is_dir(), f'no immutable commit and model path is not local: {root}'
    files = sorted({
        path for pattern in ('*.safetensors', 'pytorch_model*.bin')
        for path in root.rglob(pattern) if path.is_file()
    })
    assert files, f'no local safetensors/bin weights under {root}'
    signature = tuple(
        (path.relative_to(root).as_posix(), int(path.stat().st_size), int(path.stat().st_mtime_ns))
        for path in files
    )
    cache_key = (str(root), signature)
    if cache_key not in _LOCAL_WEIGHT_HASH_CACHE:
        digest = hashlib.sha256()
        for path in files:
            relative = path.relative_to(root).as_posix().encode('utf-8')
            digest.update(len(relative).to_bytes(8, 'big'))
            digest.update(relative)
            with path.open('rb') as handle:
                while True:
                    chunk = handle.read(8 * 1024 * 1024)
                    if not chunk:
                        break
                    digest.update(chunk)
        _LOCAL_WEIGHT_HASH_CACHE[cache_key] = digest.hexdigest()
    return _LOCAL_WEIGHT_HASH_CACHE[cache_key]


def _tokenizer_behavior_fingerprint(tokenizer: Any, family: str) -> Dict[str, Any]:
    """Fingerprint executable tokenization behavior and tokenizer revision metadata."""
    assert family in {'encdec_bart', 'encdec_t5'}
    representation_kind: Optional[str] = None
    representation_bytes: Optional[bytes] = None
    backend = getattr(tokenizer, 'backend_tokenizer', None)
    if backend is not None and callable(getattr(backend, 'to_str', None)):
        representation_kind = 'fast_backend_json'
        representation_bytes = backend.to_str().encode('utf-8')
    if representation_bytes is None:
        sp_model = getattr(tokenizer, 'sp_model', None)
        if sp_model is not None and callable(getattr(sp_model, 'serialized_model_proto', None)):
            representation_kind = 'sentencepiece_proto'
            representation_bytes = bytes(sp_model.serialized_model_proto())
    if representation_bytes is None:
        bpe_ranks = getattr(tokenizer, 'bpe_ranks', None)
        if isinstance(bpe_ranks, dict) and bpe_ranks:
            ordered_merges = sorted(
                (int(rank), str(pair[0]), str(pair[1]))
                for pair, rank in bpe_ranks.items()
            )
            representation_kind = 'bart_ordered_bpe_ranks'
            representation_bytes = json.dumps(
                ordered_merges, ensure_ascii=False, separators=(',', ':'),
            ).encode('utf-8')
    assert representation_kind is not None and representation_bytes is not None, (
        f'no tokenization model representation for supported family={family}'
    )
    init_kwargs = getattr(tokenizer, 'init_kwargs', {})
    tokenizer_config = dict(init_kwargs) if isinstance(init_kwargs, dict) else {}
    tokenizer_commit = getattr(tokenizer, '_commit_hash', None) or tokenizer_config.get('_commit_hash')
    tokenizer_revision = getattr(tokenizer, 'revision', None) or tokenizer_config.get('revision')
    behavior_payload = {
        'family': family,
        'tokenizer_class': type(tokenizer).__name__,
        'representation_kind': representation_kind,
        'representation_sha256': hashlib.sha256(representation_bytes).hexdigest(),
        'added_tokens': {
            str(token): int(token_id) for token, token_id in tokenizer.get_added_vocab().items()
        },
        'special_tokens': getattr(tokenizer, 'special_tokens_map_extended', {}),
        'tokenizer_config': tokenizer_config,
    }
    return {
        'behavior_representation': representation_kind,
        'behavior_sha256': _sha256_json(behavior_payload),
        '_commit_hash': None if not tokenizer_commit else str(tokenizer_commit),
        'revision': None if tokenizer_revision is None else str(tokenizer_revision),
    }


def _model_content_fingerprints(model: ProbeSystem) -> Dict[str, Any]:
    """Cache deterministic tokenizer/config and immutable model-content identities."""
    cached = getattr(model, '_eeg_faith_fingerprints', None)
    if cached is not None:
        return dict(cached)
    model_config = model.llm.config.to_dict()
    tokenizer_vocab = {
        str(token): int(token_id) for token, token_id in model.tokenizer.get_vocab().items()
    }
    tokenizer_behavior = _tokenizer_behavior_fingerprint(model.tokenizer, model.family)
    raw_resolved_path = str(getattr(model.llm.config, '_name_or_path', ''))
    resolved_candidate = Path(raw_resolved_path)
    resolved_path = (
        str(resolved_candidate.resolve()) if resolved_candidate.exists() else raw_resolved_path
    )
    commit_hash = getattr(model.llm.config, '_commit_hash', None)
    immutable_commit = _immutable_revision_digest(commit_hash)
    local_weights_sha256 = (
        None if immutable_commit is not None else _local_weight_sha256(resolved_path)
    )
    fingerprints: Dict[str, Any] = {
        'resolved_path': resolved_path,
        '_commit_hash': immutable_commit,
        'model_config_sha256': _sha256_json(model_config),
        'tokenizer_vocab_sha256': _sha256_json(tokenizer_vocab),
        'tokenizer_behavior_representation': tokenizer_behavior['behavior_representation'],
        'tokenizer_behavior_sha256': tokenizer_behavior['behavior_sha256'],
        'tokenizer_commit_hash': tokenizer_behavior['_commit_hash'],
        'tokenizer_revision': tokenizer_behavior['revision'],
        'local_weights_sha256': local_weights_sha256,
    }
    assert immutable_commit is not None or local_weights_sha256 is not None
    model._eeg_faith_fingerprints = dict(fingerprints)
    return fingerprints


def _checkpoint_static_metadata(
    model: ProbeSystem,
    seed: int,
    data_provenance: Dict[str, Any],
) -> Dict[str, Any]:
    """Build strict model, tokenizer, effective-config, and corpus provenance."""
    token_ids = {
        'pad_token_id': model.tokenizer.pad_token_id,
        'bos_token_id': getattr(model.tokenizer, 'bos_token_id', None),
        'eos_token_id': model.tokenizer.eos_token_id,
        'unk_token_id': getattr(model.tokenizer, 'unk_token_id', None),
        'decoder_start_token_id': model._decoder_start_id(),
    }
    effective_config = {
        key: model.config[key] for key in _EFFECTIVE_CHECKPOINT_CONFIG_KEYS
    }
    fingerprints = _model_content_fingerprints(model)
    provenance = dict(data_provenance)
    required_data_keys = {
        'resolved_path', 'file_size_bytes', 'sha256', 'rows', 'phase_counts',
        'required_columns', 'integrity_checks',
    }
    _assert_exact_state_keys(set(provenance), required_data_keys, 'corpus provenance')
    assert len(str(provenance['sha256'])) == 64
    assert int(provenance['rows']) == sum(int(v) for v in provenance['phase_counts'].values())
    assert all(bool(v) for v in provenance['integrity_checks'].values())
    return {
        'format': 'eeg_faith_independent_v5',
        'data_key': 'eeg',
        'seed': int(seed),
        'llm_frozen': True,
        'independent_training': True,
        'probe_checkpoint_loaded': False,
        'validation_objective': 'teacher_forced_composite',
        'model': {
            'llm_key': str(model.llm_key),
            'requested_repo_id': str(model.repo_id),
            'source': str(model.source),
            'modelscope_id': model.modelscope_id,
            'resolved_name_or_path': fingerprints['resolved_path'],
            '_commit_hash': fingerprints['_commit_hash'],
            'model_config_sha256': fingerprints['model_config_sha256'],
            'local_weights_sha256': fingerprints['local_weights_sha256'],
            'family': str(model.family),
            'model_class': type(model.llm).__name__,
            'config_class': type(model.llm.config).__name__,
            'embed_dim': int(model.embed_dim),
            'decoder_layers': int(model.num_decoder_layers()),
            'vocab_size': int(model.vocab_size()),
        },
        'tokenizer': {
            'class': type(model.tokenizer).__name__,
            'resolved_name_or_path': str(getattr(model.tokenizer, 'name_or_path', '')),
            '_commit_hash': fingerprints['tokenizer_commit_hash'],
            'revision': fingerprints['tokenizer_revision'],
            'vocab_size': int(len(model.tokenizer)),
            'vocab_sha256': fingerprints['tokenizer_vocab_sha256'],
            'behavior_representation': fingerprints['tokenizer_behavior_representation'],
            'behavior_sha256': fingerprints['tokenizer_behavior_sha256'],
            'special_token_ids': token_ids,
        },
        'effective_config': effective_config,
        'data': provenance,
    }


def build_e2e_checkpoint_payload(
    model: ProbeSystem,
    epoch: int,
    val_loss: float,
    data_key: str,
    seed: int,
    data_provenance: Dict[str, Any],
) -> Dict[str, Any]:
    """Create an independently trained non-LLM checkpoint with strict provenance."""
    assert data_key == 'eeg', 'oracle memory is never trained or checkpointed'
    state = {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
        if not key.startswith('llm.')
    }
    metadata = _checkpoint_static_metadata(model, seed, data_provenance)
    metadata.update({'epoch': int(epoch), 'val_loss': float(val_loss)})
    return {'model': state, 'metadata': metadata}


def validate_and_load_e2e_checkpoint(
    model: ProbeSystem,
    payload: Dict[str, Any],
    seed: int,
    data_provenance: Dict[str, Any],
) -> None:
    """Strictly validate provenance, keys, and shapes before restoring non-LLM state."""
    assert set(payload.keys()) == {'model', 'metadata'}
    metadata = payload['metadata']
    assert isinstance(metadata, dict)
    expected = _checkpoint_static_metadata(model, seed, data_provenance)
    _assert_exact_state_keys(
        set(metadata), set(expected) | {'epoch', 'val_loss'}, 'checkpoint metadata',
    )
    for key, expected_value in expected.items():
        assert metadata[key] == expected_value, (
            f'checkpoint metadata {key}: {metadata[key]!r} != {expected_value!r}'
        )
    assert isinstance(metadata['epoch'], int) and int(metadata['epoch']) >= 1
    assert math.isfinite(float(metadata['val_loss']))
    state = payload['model']
    assert isinstance(state, dict)
    current = model.state_dict()
    expected_keys = {key for key in current if not key.startswith('llm.')}
    _assert_exact_state_keys(set(state), expected_keys, 'non-LLM checkpoint')
    for key in expected_keys:
        assert torch.is_tensor(state[key]) and state[key].shape == current[key].shape, key
        assert torch.isfinite(state[key].float()).all(), f'non-finite checkpoint tensor: {key}'
    incompatible = model.load_state_dict(state, strict=False)
    expected_missing = {key for key in current if key.startswith('llm.')}
    _assert_exact_state_keys(set(incompatible.missing_keys), expected_missing, 'frozen LLM omissions')
    assert not incompatible.unexpected_keys


In [ ]:
# =============================================================
# One-stage E2E train (eeg only) + qualitative NN
# =============================================================

def sequential_dataloader(
    dataset: Dataset,
    batch_size: int,
    num_workers: int = 0,
) -> DataLoader:
    """Full-split sequential loader (all trials; not unique-UID GLIM batches)."""
    assert batch_size >= 1
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=int(num_workers),
        pin_memory=True,
        collate_fn=probe_collate_fn,
    )


def train_e2e(
    model: ProbeSystem,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Dict[str, Any],
    output_dir: Path,
    data_key: str,
    data_provenance: Dict[str, Any],
) -> Dict[str, List[float]]:
    """Independently train EEG; early-stop on the teacher-forced val composite."""
    assert data_key == 'eeg', 'eeg_faith never trains oracle_text or loads probe weights'
    assert int(data_provenance.get('rows', -1)) >= 3
    device = model.device
    epochs = int(config.get('epochs', 100))
    assert epochs >= 1, 'epochs must be >= 1'
    patience = int(config.get('patience', 10))
    lr = float(config.get('lr', 1e-4))
    grad_clip = float(config.get('grad_clip', 1.0))
    seed = int(config.get('seed', 2026))
    model.set_data_key('eeg')
    params = model.trainable_parameters()
    assert len(params) >= 1
    opt = AdamW(params, lr=lr, weight_decay=float(config.get('weight_decay', 0.05)))
    scheduler = setup_scheduler(opt, config)
    history: Dict[str, List[float]] = {
        'train_loss': [], 'val_loss': [],
        'train_loss_clip': [], 'val_loss_clip': [],
        'train_loss_commit': [], 'val_loss_commit': [],
        'train_loss_ar': [], 'val_loss_ar': [], 'val_rollout_ar': [],
        'train_cos': [], 'val_cos': [],
        'train_weighted_clip': [], 'val_weighted_clip': [],
        'train_weighted_ar': [], 'val_weighted_ar': [],
        'train_weighted_commit': [], 'val_weighted_commit': [],
    }
    best_val = float('inf')
    best_payload: Optional[Dict[str, Any]] = None
    bad = 0
    model.to(device)

    def _run_epoch(loader: DataLoader, training: bool) -> Dict[str, float]:
        """Run one teacher-forced E2E epoch (train or val).

        Args:
            loader: Phase DataLoader.
            training: If True, step the optimizer; else eval-only.

        Returns:
            Weighted-average loss dict (clip / AR / commit / total).
        """
        model.train(mode=training)
        model.llm.eval()
        sample_count = 0
        ar_count = 0
        commit_count = 0
        clip_sum = 0.0
        ar_sum = 0.0
        commit_sum = 0.0
        cos_sum = 0.0
        rollout_sum = 0.0
        rollout_count = 0
        for bi, raw_batch in enumerate(loader):
            batch = move_to_device(raw_batch, device)
            if training:
                opt.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(training):
                out = model.forward_e2e(
                    batch, 'eeg', mix_seed=None if training else seed + int(bi),
                )
                assert torch.isfinite(out['loss'])
                if training:
                    out['loss'].backward()
                    assert all(
                        p.grad is None or bool(torch.isfinite(p.grad).all()) for p in params
                    ), 'non-finite E2E gradient'
                    torch.nn.utils.clip_grad_norm_(params, grad_clip)
                    opt.step()
            n_sample = int(out['n_samples'].item())
            n_ar = int(out['n_ar_tokens'].item())
            n_commit = int(out['n_commit_elements'].item())
            assert n_sample == int(batch['eeg'].shape[0]) and n_ar >= n_sample and n_commit >= n_sample
            sample_count += n_sample
            ar_count += n_ar
            commit_count += n_commit
            clip_sum += float(out['loss_clip'].cpu()) * n_sample
            ar_sum += float(out['loss_ar'].cpu()) * n_ar
            commit_sum += float(out['loss_commit'].cpu()) * n_commit
            cos_sum += float(out['cos'].cpu()) * n_sample
            if not training:
                rollout_ce, rollout_tokens = model.rollout_ar_diagnostic(
                    batch, 'eeg', mix_seed=seed + int(bi),
                )
                rollout_sum += float(rollout_ce.cpu()) * int(rollout_tokens)
                rollout_count += int(rollout_tokens)
        assert sample_count >= 1 and ar_count >= 1 and commit_count >= 1
        clip_mean = clip_sum / float(sample_count)
        ar_mean = ar_sum / float(ar_count)
        commit_mean = commit_sum / float(commit_count)
        weighted_clip = model.w_clip * clip_mean
        weighted_ar = model.w_ar * ar_mean
        weighted_commit = model.commitment_weight * commit_mean
        metrics = {
            'loss': weighted_clip + weighted_ar + weighted_commit,
            'loss_clip': clip_mean,
            'loss_ar': ar_mean,
            'loss_commit': commit_mean,
            'cos': cos_sum / float(sample_count),
            'weighted_clip': weighted_clip,
            'weighted_ar': weighted_ar,
            'weighted_commit': weighted_commit,
            'samples': float(sample_count),
            'ar_tokens': float(ar_count),
            'commit_elements': float(commit_count),
            'rollout_ar': rollout_sum / float(rollout_count) if rollout_count else float('nan'),
        }
        assert all(math.isfinite(value) for key, value in metrics.items() if key != 'rollout_ar' or not training)
        return metrics

    for epoch in range(1, epochs + 1):
        if hasattr(train_loader, 'batch_sampler') and hasattr(train_loader.batch_sampler, 'set_epoch'):
            train_loader.batch_sampler.set_epoch(epoch)
        train_metrics = _run_epoch(train_loader, training=True)
        with torch.no_grad():
            val_metrics = _run_epoch(val_loader, training=False)
        scheduler.step()
        for prefix, metrics in (('train', train_metrics), ('val', val_metrics)):
            history[f'{prefix}_loss'].append(metrics['loss'])
            history[f'{prefix}_loss_clip'].append(metrics['loss_clip'])
            history[f'{prefix}_loss_ar'].append(metrics['loss_ar'])
            history[f'{prefix}_loss_commit'].append(metrics['loss_commit'])
            history[f'{prefix}_cos'].append(metrics['cos'])
            history[f'{prefix}_weighted_clip'].append(metrics['weighted_clip'])
            history[f'{prefix}_weighted_ar'].append(metrics['weighted_ar'])
            history[f'{prefix}_weighted_commit'].append(metrics['weighted_commit'])
        history['val_rollout_ar'].append(val_metrics['rollout_ar'])
        lr_now = float(opt.param_groups[0]['lr'])
        print(
            f'  [E2E eeg] epoch {epoch:03d}/{epochs} '
            f'TF train/val={train_metrics["loss"]:.4f}/{val_metrics["loss"]:.4f} '
            f'rollout-display={val_metrics["rollout_ar"]:.4f} lr={lr_now:.2e}\n'
            f'    weighted CLIP={train_metrics["weighted_clip"]:.4f}/{val_metrics["weighted_clip"]:.4f} '
            f'AR={train_metrics["weighted_ar"]:.4f}/{val_metrics["weighted_ar"]:.4f} '
            f'commit={train_metrics["weighted_commit"]:.4f}/{val_metrics["weighted_commit"]:.4f}\n'
            f'    samples={int(train_metrics["samples"])}/{int(val_metrics["samples"])} '
            f'AR tokens={int(train_metrics["ar_tokens"])}/{int(val_metrics["ar_tokens"])} '
            f'valid commitment elements={int(train_metrics["commit_elements"])}/{int(val_metrics["commit_elements"])}'
        )

        val_total = val_metrics['loss']
        if val_total < best_val - 1e-6:
            best_val = val_total
            bad = 0
            best_payload = build_e2e_checkpoint_payload(
                model, epoch, best_val, 'eeg', seed, data_provenance,
            )
            validate_and_load_e2e_checkpoint(model, best_payload, seed, data_provenance)
            torch.save(best_payload, output_dir / 'best.pt')
        else:
            bad += 1
            if bad >= patience:
                print(f'  Early stop at epoch {epoch} on TF val composite (patience={patience})')
                break

    assert best_payload is not None, 'no finite E2E checkpoint was produced'
    validate_and_load_e2e_checkpoint(model, best_payload, seed, data_provenance)
    print(f'  Restored strictly validated independent EEG checkpoint (TF val={best_val:.4f})')

    fig, ax = plt.subplots(1, 4, figsize=(16.5, 3.8))
    ax[0].plot(history['train_loss'], color=COLOR_PRIMARY, label='train TF')
    ax[0].plot(history['val_loss'], color=COLOR_SECONDARY, label='val TF')
    ax[0].set_title('Teacher-forced composite'); ax[0].legend(); ax[0].set_xlabel('epoch')
    ax[1].plot(history['train_loss_ar'], color=COLOR_ACC, label='train AR (TF)')
    ax[1].plot(history['val_loss_ar'], color=COLOR_PURPLE, label='val AR (TF)')
    ax[1].plot(history['val_rollout_ar'], color=COLOR_SECONDARY, linestyle=':', label='val rollout (display)')
    ax[1].set_title('AR diagnostics'); ax[1].legend(); ax[1].set_xlabel('epoch')
    ax[2].plot(history['train_loss_commit'], color=COLOR_PRIMARY, label='train')
    ax[2].plot(history['val_loss_commit'], color=COLOR_SECONDARY, label='val')
    ax[2].set_title('Masked normalized MSE'); ax[2].legend(); ax[2].set_xlabel('epoch')
    ax[3].plot(history['train_cos'], color=COLOR_PRIMARY, label='train')
    ax[3].plot(history['val_cos'], color=COLOR_SECONDARY, label='val')
    ax[3].set_title('Cosine(ei_kv, ei_text)'); ax[3].legend(); ax[3].set_xlabel('epoch')
    fig.suptitle('Independent EEG E2E curves', fontsize=13)
    fig.tight_layout()
    fig.savefig(output_dir / 'e2e_curves.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return history


def train_tuned_lens(
    model: ProbeSystem,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Dict[str, Any],
    output_dir: Path,
) -> TunedLens:
    """
    Train official TunedLens translators on oracle full-target memory at position -1.

    The frozen LLM and independently trained EEG modules remain frozen. The
    decoder prefix is instruction-only and identical to the EEG path.
    """
    device = model.device
    seed = int(config.get('seed', 2026))
    epochs = int(config.get('lens_epochs', 100))
    patience = int(config.get('lens_patience', 10))
    lr = float(config.get('lens_lr', 1e-3))
    set_seed(seed)
    model.eval()
    model.set_data_key('oracle_text')
    for p in model.parameters():
        p.requires_grad = False

    probe_batch: Optional[Dict[str, Any]] = None
    for batch0 in train_loader:
        probe_batch = move_to_device(batch0, device)
        break
    assert probe_batch is not None, 'empty train loader for tuned lens'
    with torch.no_grad():
        probe = model.decoder_states_at_minus1(
            probe_batch, 'oracle_text', mix_seed=seed,
        )
    n_states = len(probe['hidden_m1'])
    assert n_states == model.num_decoder_layers() + 1
    n_translators = n_states - 1
    lens = build_tuned_lens(model, n_translators)
    lens.to(device)
    lens.unembed.eval()
    lens.unembed.requires_grad_(False)
    _ = verify_unembed_against_official(model, lens, probe, stage='before lens training')
    params = [p for p in lens.layer_translators.parameters() if p.requires_grad]
    assert len(params) >= 1
    opt = AdamW(params, lr=lr, weight_decay=float(config.get('weight_decay', 0.05)))
    print(
        f'  Tuned lens translators={n_translators} hidden_states={n_states} '
        f'd_model={model.embed_dim} lr={lr:.2e}'
    )

    history: Dict[str, List[float]] = {'train_kl': [], 'val_kl': []}
    best_val = float('inf')
    best_state: Optional[Dict[str, torch.Tensor]] = None
    bad = 0
    save_dir = output_dir / 'tuned_lens'

    def _epoch_kl(loader: DataLoader, train: bool) -> float:
        """Mean tuned-lens KL over one loader.

        Args:
            loader: Phase DataLoader (oracle memory).
            train: If True, update lens translators.

        Returns:
            Token-weighted mean KL.
        """
        lens.train(mode=train)
        lens.unembed.eval()
        weighted_sum = 0.0
        sample_count = 0
        for bi, batch in enumerate(loader):
            batch = move_to_device(batch, device)
            with torch.no_grad():
                pack = model.decoder_states_at_minus1(
                    batch, 'oracle_text', mix_seed=None if train else (seed + int(bi)),
                )
            hs = pack['hidden_m1'][:-1]
            assert len(hs) == n_translators
            log_p = pack['logits_hf'].float().log_softmax(dim=-1).detach()
            if train:
                opt.zero_grad(set_to_none=True)
            layer_losses: List[torch.Tensor] = []
            for i, h in enumerate(hs):
                loss_i = kl_tuned_lens_layer(lens, h.detach(), i, log_p)
                if train:
                    loss_i.backward()
                layer_losses.append(loss_i.detach())
            if train:
                assert all(p.grad is None or bool(torch.isfinite(p.grad).all()) for p in params)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step()
            mean_kl = torch.stack(layer_losses).mean()
            batch_size = int(log_p.shape[0])
            weighted_sum += float(mean_kl.cpu()) * batch_size
            sample_count += batch_size
        assert sample_count >= 1
        return weighted_sum / float(sample_count)

    for epoch in range(1, epochs + 1):
        if hasattr(train_loader, 'batch_sampler') and hasattr(train_loader.batch_sampler, 'set_epoch'):
            train_loader.batch_sampler.set_epoch(epoch)
        tr_m = _epoch_kl(train_loader, train=True)
        with torch.no_grad():
            va_m = _epoch_kl(val_loader, train=False)
        history['train_kl'].append(tr_m)
        history['val_kl'].append(va_m)
        print(
            f'  [tuned lens] epoch {epoch:03d}/{epochs} '
            f'train_kl={tr_m:.4f} val_kl={va_m:.4f}'
        )
        if va_m < best_val - 1e-6:
            best_val = va_m
            bad = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in lens.layer_translators.state_dict().items()
            }
            lens.save(save_dir)
        else:
            bad += 1
            if bad >= patience:
                print(f'  Tuned lens early stop at epoch {epoch} (patience={patience})')
                break

    assert best_state is not None, 'no finite tuned-lens checkpoint was produced'
    lens.layer_translators.load_state_dict(best_state, strict=True)
    print(f'  Restored best tuned lens (val_kl={best_val:.4f})')
    lens.eval()
    with torch.no_grad():
        post_pack = model.decoder_states_at_minus1(
            probe_batch, 'oracle_text', mix_seed=seed,
        )
    _ = verify_unembed_against_official(model, lens, post_pack, stage='after lens training')
    lens.save(save_dir)
    print(f'  Saved tuned lens checkpoint: {save_dir}')

    fig, ax = plt.subplots(figsize=(6.8, 3.8))
    ax.plot(history['train_kl'], color=COLOR_PRIMARY, label='train KL')
    ax.plot(history['val_kl'], color=COLOR_SECONDARY, label='val KL')
    ax.set_xlabel('epoch')
    ax.set_ylabel('mean KL(final || lens)')
    ax.set_title('Tuned lens translators')
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / 'tuned_lens_kl.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    with torch.no_grad():
        shown = 0
        n_show = int(config.get('qualitative_n', 5))
        tok = model.tokenizer
        for bi, batch in enumerate(val_loader):
            batch = move_to_device(batch, device)
            pack = model.decoder_states_at_minus1(
                batch, 'oracle_text', mix_seed=seed + int(bi),
            )
            t_star = pack['t_star']
            last_logits = pack['logits_hf']
            pred_final = last_logits.argmax(dim=-1)
            h1 = pack['hidden_m1'][1] if n_translators >= 2 else pack['hidden_m1'][0]
            idx1 = 1 if n_translators >= 2 else 0
            pred_l1 = lens.forward(hidden_for_lens(lens, h1), idx1).argmax(dim=-1)
            gts = [str(t) for t in batch['input text']]
            B = int(t_star.shape[0])
            print('  Qualitative tuned lens (val, pos -1):')
            for i in range(B):
                if shown >= n_show:
                    break
                tid = int(t_star[i].item())
                print(
                    f'  --- example {shown + 1} ---\n'
                    f'    GT         : {gts[i]}\n'
                    f'    t*         : {tok.decode([tid], skip_special_tokens=False)!r}\n'
                    f'    final argmax: {tok.decode([int(pred_final[i].item())], skip_special_tokens=False)!r}\n'
                    f'    lens[{idx1}] argmax: {tok.decode([int(pred_l1[i].item())], skip_special_tokens=False)!r}'
                )
                shown += 1
            if shown >= n_show:
                break
    return lens


@torch.no_grad()
def show_qualitative_nn(
    model: ProbeSystem,
    loader: DataLoader,
    data_key: str,
    n: int = 5,
    seed: int = 2026,
) -> None:
    """Nearest-neighbor retrieval: KV -> closest in-batch texts via cosine."""
    model.eval()
    shown = 0
    for bi, batch in enumerate(loader):
        batch = move_to_device(batch, model.device)
        _z, ei_kv, _kv_mask, _h, _text_mask, ei_t = model.encode_kv(
            batch, data_key, mix_seed=seed + int(bi),
        )
        texts = [str(t) for t in batch['input text']]
        ei_n = F.normalize(ei_kv.float(), dim=-1)
        et_n = F.normalize(ei_t.float(), dim=-1)
        sim = ei_n @ et_n.T
        for i in range(ei_kv.shape[0]):
            if shown >= n:
                return
            j = int(sim[i].argmax().item())
            print(f'--- example {shown + 1} ---')
            print(f'  GT     : {texts[i]}')
            print(f'  NN text: {texts[j]}  (cos={float(sim[i, j]):.3f})')
            shown += 1


In [ ]:
# =============================================================
# Analyses: tuned lens, Spearman, overlap+Frechet, linear probes
# =============================================================

def probe_layer_ids(n_blocks: int, every: int = 3) -> List[int]:
    """1-based decoder blocks {every, 2*every, ..., n_blocks} including the last if divisible."""
    assert n_blocks >= 1 and every >= 1
    layers = list(range(int(every), int(n_blocks) + 1, int(every)))
    assert len(layers) >= 1, f'no probe layers for n_blocks={n_blocks}'
    return layers


def hidden_index_for_block(n_states: int, n_blocks: int, block: int) -> int:
    """Map block `1..L` to HF state index `1..L`; index 0 is embeddings."""
    assert int(n_states) == int(n_blocks) + 1, (n_states, n_blocks)
    assert 1 <= int(block) <= int(n_blocks), (block, n_blocks)
    return int(block)


def mean_sem(x: np.ndarray) -> Tuple[Any, Any]:
    """Finite-only mean and SEM over rows."""
    arr = np.asarray(x, dtype=np.float64)
    assert arr.ndim in (1, 2)
    finite_count = np.isfinite(arr).sum(axis=0)
    assert bool(np.all(finite_count >= 1)), finite_count
    mean = np.nanmean(arr, axis=0)
    sem = np.zeros_like(mean, dtype=np.float64)
    enough = finite_count > 1
    sem = np.where(
        enough,
        np.nanstd(arr, axis=0, ddof=1) / np.sqrt(np.maximum(finite_count, 1)),
        0.0,
    )
    if arr.ndim == 1:
        return float(mean), float(sem)
    return mean.astype(np.float64), sem.astype(np.float64)


def aggregate_rows_by_uid(values: np.ndarray, uids: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Average repeated trial rows within text UID."""
    arr = np.asarray(values, dtype=np.float64)
    uid_arr = np.asarray(uids).reshape(-1)
    assert arr.ndim == 2 and arr.shape[0] == uid_arr.size
    unique_uids = np.asarray(sorted(set(uid_arr.tolist())))
    aggregated = np.stack([
        np.nanmean(arr[uid_arr == uid], axis=0) for uid in unique_uids
    ], axis=0)
    assert aggregated.shape == (unique_uids.size, arr.shape[1])
    return aggregated, unique_uids


def bootstrap_layer_summary(
    values: np.ndarray,
    bootstrap_samples: int,
    seed: int,
) -> Dict[str, np.ndarray]:
    """Finite-count means and UID-level bootstrap 95% intervals by layer."""
    arr = np.asarray(values, dtype=np.float64)
    assert arr.ndim == 2 and arr.shape[0] >= 2
    n_boot = int(bootstrap_samples)
    assert n_boot >= 100
    rng = np.random.default_rng(int(seed))
    means = np.nanmean(arr, axis=0)
    counts = np.isfinite(arr).sum(axis=0).astype(np.int64)
    assert bool(np.all(counts >= 2)), counts
    boot = np.empty((n_boot, arr.shape[1]), dtype=np.float64)
    for i in range(n_boot):
        sampled = rng.integers(0, arr.shape[0], size=arr.shape[0])
        boot[i] = np.nanmean(arr[sampled], axis=0)
    return {
        'mean': means,
        'low': np.nanpercentile(boot, 2.5, axis=0),
        'high': np.nanpercentile(boot, 97.5, axis=0),
        'finite_n': counts,
    }


def spearman_rowwise(a: torch.Tensor, b: torch.Tensor) -> np.ndarray:
    """Per-row Spearman rho across D dimensions. a,b: (B, D) -> (B,)."""
    assert a.shape == b.shape and a.ndim == 2
    a_np = a.detach().float().cpu().numpy()
    b_np = b.detach().float().cpu().numpy()
    rhos: List[float] = []
    for i in range(a_np.shape[0]):
        r, _p = stats.spearmanr(a_np[i], b_np[i])
        rhos.append(float(r) if r == r else float('nan'))
    out = np.asarray(rhos, dtype=np.float64)
    assert out.shape == (a_np.shape[0],)
    return out


def _real_sqrtm(matrix: np.ndarray, label: str, complex_tol: float = 1e-6) -> np.ndarray:
    """Matrix square root that rejects material imaginary numerical error."""
    root = scipy.linalg.sqrtm(matrix)
    if np.iscomplexobj(root):
        imag_max = float(np.max(np.abs(root.imag)))
        real_scale = max(1.0, float(np.max(np.abs(root.real))))
        if imag_max > float(complex_tol) * real_scale:
            raise FloatingPointError(f'{label} sqrtm complex error {imag_max:.3e} exceeds tolerance')
        root = root.real
    root = np.asarray(root, dtype=np.float64)
    assert np.isfinite(root).all(), f'non-finite {label} sqrtm'
    return root


def _frechet_distance(act1: np.ndarray, act2: np.ndarray) -> float:
    """Shrinkage-covariance Gaussian Fréchet distance between `(N,D)` clouds."""
    x = np.asarray(act1, dtype=np.float64)
    y = np.asarray(act2, dtype=np.float64)
    assert x.ndim == 2 and y.ndim == 2 and x.shape[1] == y.shape[1]
    assert x.shape[0] >= 2 and y.shape[0] >= 2
    assert np.isfinite(x).all() and np.isfinite(y).all()
    mu1, mu2 = x.mean(axis=0), y.mean(axis=0)
    sig1 = LedoitWolf().fit(x).covariance_
    sig2 = LedoitWolf().fit(y).covariance_
    sig1 = (sig1 + sig1.T) * 0.5
    sig2 = (sig2 + sig2.T) * 0.5
    sqrt1 = _real_sqrtm(sig1, 'covariance-1')
    middle = sqrt1 @ sig2 @ sqrt1
    middle = (middle + middle.T) * 0.5
    covmean = _real_sqrtm(middle, 'covariance product')
    diff = mu1 - mu2
    distance = float(diff @ diff + np.trace(sig1) + np.trace(sig2) - 2.0 * np.trace(covmean))
    if distance < -1e-6:
        raise FloatingPointError(f'materially negative Frechet distance: {distance}')
    return max(0.0, distance)


def paired_cohens_dz(differences: np.ndarray) -> float:
    """Paired Cohen dz from UID-level within-pair differences."""
    diff = np.asarray(differences, dtype=np.float64).reshape(-1)
    diff = diff[np.isfinite(diff)]
    assert diff.size >= 2
    sd = float(np.std(diff, ddof=1))
    return 0.0 if sd <= 1e-12 else float(np.mean(diff) / sd)


def _derangement(n: int, rng: np.random.Generator) -> np.ndarray:
    """Sample a permutation with no fixed point."""
    assert n >= 2
    base = np.arange(n)
    for _ in range(1000):
        perm = rng.permutation(n)
        if bool(np.all(perm != base)):
            return perm
    shift = int(rng.integers(1, n))
    perm = np.roll(base, shift)
    assert bool(np.all(perm != base))
    return perm


def semantic_overlap_vs_random(
    ei_kv: torch.Tensor,
    ei_text: torch.Tensor,
    uids: np.ndarray,
    n_derangements: int,
    seed: int = 2026,
) -> Tuple[np.ndarray, np.ndarray]:
    """UID-paired cosine and repeated no-self-UID derangement baselines."""
    assert ei_kv.ndim == 2 and ei_text.shape == ei_kv.shape
    uid_arr = np.asarray(uids).reshape(-1)
    n = int(ei_kv.shape[0])
    assert uid_arr.size == n and len(set(uid_arr.tolist())) == n and n >= 2
    trials = int(n_derangements)
    assert trials >= 2
    x = F.normalize(ei_kv.float(), dim=-1).cpu()
    y = F.normalize(ei_text.float(), dim=-1).cpu()
    paired = (x * y).sum(dim=-1).numpy().astype(np.float64)
    rng = np.random.default_rng(int(seed))
    random_trials = np.empty((trials, n), dtype=np.float64)
    for trial in range(trials):
        perm = _derangement(n, rng)
        assert bool(np.all(uid_arr[perm] != uid_arr))
        random_trials[trial] = (x * y[torch.from_numpy(perm)]).sum(dim=-1).numpy()
    return paired, random_trials


def paired_permutation_inference(
    paired: np.ndarray,
    random_trials: np.ndarray,
    permutation_samples: int,
    bootstrap_samples: int,
    seed: int,
) -> Dict[str, float]:
    """UID-level paired sign-flip inference, Cohen dz, and bootstrap intervals."""
    paired_arr = np.asarray(paired, dtype=np.float64).reshape(-1)
    random_arr = np.asarray(random_trials, dtype=np.float64)
    assert random_arr.ndim == 2 and random_arr.shape[1] == paired_arr.size
    random_uid = random_arr.mean(axis=0)
    diff = paired_arr - random_uid
    assert diff.size >= 2 and np.isfinite(diff).all()
    observed = float(np.mean(diff))
    rng = np.random.default_rng(int(seed))
    extreme = 0
    for _ in range(int(permutation_samples)):
        signs = rng.choice(np.asarray([-1.0, 1.0]), size=diff.size)
        extreme += int(abs(float(np.mean(diff * signs))) >= abs(observed))
    p_value = float((extreme + 1) / (int(permutation_samples) + 1))
    n_boot = int(bootstrap_samples)
    boot_paired = np.empty(n_boot, dtype=np.float64)
    boot_random = np.empty(n_boot, dtype=np.float64)
    boot_diff = np.empty(n_boot, dtype=np.float64)
    boot_dz = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        idx = rng.integers(0, diff.size, size=diff.size)
        b_diff = diff[idx]
        boot_paired[i] = float(np.mean(paired_arr[idx]))
        boot_random[i] = float(np.mean(random_uid[idx]))
        boot_diff[i] = float(np.mean(b_diff))
        boot_dz[i] = paired_cohens_dz(b_diff) if float(np.std(b_diff, ddof=1)) > 1e-12 else 0.0
    def _ci(values: np.ndarray) -> Tuple[float, float]:
        """Percentile bootstrap interval.

        Args:
            values: 1-D bootstrap replicates.

        Returns:
            ``(2.5th, 97.5th)`` percentiles.
        """
        return float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))
    paired_ci = _ci(boot_paired)
    random_ci = _ci(boot_random)
    diff_ci = _ci(boot_diff)
    dz_ci = _ci(boot_dz)
    return {
        'paired_mean': float(np.mean(paired_arr)),
        'paired_ci_low': paired_ci[0], 'paired_ci_high': paired_ci[1],
        'random_mean': float(np.mean(random_uid)),
        'random_ci_low': random_ci[0], 'random_ci_high': random_ci[1],
        'mean_difference': observed,
        'difference_ci_low': diff_ci[0], 'difference_ci_high': diff_ci[1],
        'permutation_p': p_value,
        'cohens_dz': paired_cohens_dz(diff),
        'cohens_dz_ci_low': dz_ci[0], 'cohens_dz_ci_high': dz_ci[1],
    }


def reduce_embeddings_for_frechet(
    x: np.ndarray,
    y: np.ndarray,
    frechet_dim: int = 64,
    seed: int = 2026,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    """Fit one deterministic PCA on combined UID-level test embeddings."""
    x_arr = np.asarray(x, dtype=np.float64)
    y_arr = np.asarray(y, dtype=np.float64)
    assert x_arr.shape == y_arr.shape and x_arr.ndim == 2 and x_arr.shape[0] >= 2
    assert np.isfinite(x_arr).all() and np.isfinite(y_arr).all()
    combined = np.concatenate([x_arr, y_arr], axis=0)
    retained_dim = min(int(frechet_dim), int(combined.shape[0] - 1), int(combined.shape[1]))
    assert retained_dim >= 1
    pca = PCA(n_components=retained_dim, svd_solver='full', random_state=int(seed))
    reduced = pca.fit_transform(combined)
    x_reduced = reduced[:x_arr.shape[0]]
    y_reduced = reduced[x_arr.shape[0]:]
    explained = float(np.sum(pca.explained_variance_ratio_))
    assert x_reduced.shape == y_reduced.shape == (x_arr.shape[0], retained_dim)
    assert np.isfinite(x_reduced).all() and np.isfinite(y_reduced).all()
    assert math.isfinite(explained) and -1e-12 <= explained <= 1.0 + 1e-9
    return x_reduced, y_reduced, {
        'frechet_dim': float(retained_dim),
        'frechet_explained_variance': explained,
    }


def bootstrap_frechet_ci(
    x: np.ndarray,
    y: np.ndarray,
    bootstrap_samples: int,
    seed: int,
) -> Tuple[float, float]:
    """Paired UID bootstrap in one already-fitted common PCA space."""
    x_arr = np.asarray(x, dtype=np.float64)
    y_arr = np.asarray(y, dtype=np.float64)
    assert x_arr.shape == y_arr.shape and x_arr.shape[0] >= 2
    rng = np.random.default_rng(int(seed))
    values = np.empty(int(bootstrap_samples), dtype=np.float64)
    for i in range(values.size):
        idx = rng.integers(0, x_arr.shape[0], size=x_arr.shape[0])
        values[i] = _frechet_distance(x_arr[idx], y_arr[idx])
    return float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))


def _kde_curve(values: np.ndarray, xs: np.ndarray) -> np.ndarray:
    """Gaussian KDE density on xs; uniform fallback if variance is 0."""
    v = np.asarray(values, dtype=np.float64).reshape(-1)
    v = v[np.isfinite(v)]
    if v.size < 2 or float(np.std(v)) < 1e-12:
        dens = np.zeros_like(xs, dtype=np.float64)
        mid = float(np.mean(v)) if v.size else 0.0
        dens[int(np.argmin(np.abs(xs - mid)))] = 1.0
        return dens
    kde = stats.gaussian_kde(v)
    return kde(xs)


def plot_overlap_kde(
    paired: np.ndarray,
    random_s: np.ndarray,
    title: str,
    save_path: Optional[Path] = None,
) -> None:
    """Overlay KDE of paired semantic overlap vs random-shuffle baseline."""
    xs = np.linspace(-1.05, 1.05, 256)
    dens_p = _kde_curve(paired, xs)
    dens_r = _kde_curve(random_s, xs)
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    ax.fill_between(xs, dens_p, alpha=0.28, color=COLOR_PRIMARY, label='Paired (semantic overlap)')
    ax.plot(xs, dens_p, color=COLOR_PRIMARY, linewidth=2.2)
    ax.plot(xs, dens_r, color=COLOR_SECONDARY, linestyle='--', linewidth=1.8, label='Random shuffle (baseline)')
    ax.set_xlabel('cosine')
    ax.set_ylabel('density')
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=PLOT_DPI, bbox_inches='tight')
        print(f'  Saved overlap KDE: {save_path}')
    plt.show()
    plt.close(fig)


def unembed_block(
    model: ProbeSystem,
    hidden_m1: Sequence[torch.Tensor],
    official_logits: torch.Tensor,
    n_blocks: int,
    block: int,
    lens: TunedLens,
) -> torch.Tensor:
    """Use tuned lens for intermediate blocks and official HF logits for final."""
    assert len(hidden_m1) == int(n_blocks) + 1
    assert official_logits.shape == (hidden_m1[0].shape[0], model.vocab_size())
    if int(block) == int(n_blocks):
        return official_logits.float()
    idx = hidden_index_for_block(len(hidden_m1), n_blocks, block)
    assert 0 <= idx < len(lens.layer_translators), (idx, len(lens.layer_translators))
    h = hidden_m1[idx]
    out = lens.forward(hidden_for_lens(lens, h), idx).float()
    assert out.shape == official_logits.shape
    assert torch.isfinite(out).all(), 'non-finite tuned-lens logits'
    return out


@torch.no_grad()
def collect_two_path_layer_stats(
    model: ProbeSystem,
    loader: DataLoader,
    config: Dict[str, Any],
    lens: TunedLens,
    max_batches: Optional[int] = None,
) -> Dict[str, Any]:
    """
    Stream test batches: oracle-vs-EEG Delta(t*) and Spearman at every block.

    Returns trial arrays `(N,L)`, text UIDs, and qualitative official-logit rows.
    """
    model.eval()
    lens.eval()
    seed = int(config.get('seed', 2026))
    n_show = int(config.get('qualitative_n', 5))
    delta_rows: List[np.ndarray] = []
    rho_rows: List[np.ndarray] = []
    uid_rows: List[np.ndarray] = []
    qual: List[Dict[str, Any]] = []
    n_blocks: Optional[int] = None
    tok = model.tokenizer

    for bi, batch in enumerate(loader):
        if max_batches is not None and int(max_batches) > 0 and bi >= int(max_batches):
            break
        batch = move_to_device(batch, model.device)
        oracle = model.decoder_states_at_minus1(batch, 'oracle_text', mix_seed=seed + int(bi))
        eeg = model.decoder_states_at_minus1(batch, 'eeg', mix_seed=seed + int(bi))
        t_star = oracle['t_star']
        assert torch.equal(t_star, eeg['t_star'])
        assert torch.equal(oracle['decoder_input_ids'], eeg['decoder_input_ids'])
        if n_blocks is None:
            n_blocks = int(oracle['n_blocks'])
            assert len(oracle['hidden_m1']) == n_blocks + 1
            print(
                f'  decoder blocks={n_blocks} hidden_states={len(oracle["hidden_m1"])} '
                f'translators={len(lens.layer_translators)} final=official HF logits'
            )
        assert n_blocks is not None
        B = int(t_star.shape[0])
        d_row = np.zeros((B, n_blocks), dtype=np.float64)
        r_row = np.zeros((B, n_blocks), dtype=np.float64)
        t_col = t_star.unsqueeze(-1)
        for block in range(1, n_blocks + 1):
            logits_oracle = unembed_block(
                model, oracle['hidden_m1'], oracle['logits_hf'], n_blocks, block, lens,
            )
            logits_e = unembed_block(
                model, eeg['hidden_m1'], eeg['logits_hf'], n_blocks, block, lens,
            )
            logit_oracle_t = logits_oracle.gather(-1, t_col).squeeze(-1)
            logit_e_t = logits_e.gather(-1, t_col).squeeze(-1)
            d_row[:, block - 1] = (logit_oracle_t - logit_e_t).detach().cpu().numpy()
            idx = hidden_index_for_block(len(oracle['hidden_m1']), n_blocks, block)
            r_row[:, block - 1] = spearman_rowwise(oracle['hidden_m1'][idx], eeg['hidden_m1'][idx])
        delta_rows.append(d_row)
        rho_rows.append(r_row)
        uid_rows.append(batch['text uid'].detach().cpu().numpy().reshape(-1))

        if len(qual) < n_show:
            logits_l_last = oracle['logits_hf'].float()
            logits_e_last = eeg['logits_hf'].float()
            pred_l = logits_l_last.argmax(dim=-1)
            pred_e = logits_e_last.argmax(dim=-1)
            gts = [str(t) for t in batch['input text']]
            for i in range(B):
                if len(qual) >= n_show:
                    break
                tid = int(t_star[i].item())
                qual.append({
                    'gt': gts[i],
                    't_star': tok.decode([tid], skip_special_tokens=False),
                    'oracle_argmax': tok.decode([int(pred_l[i].item())], skip_special_tokens=False),
                    'eeg_argmax': tok.decode([int(pred_e[i].item())], skip_special_tokens=False),
                    'delta_last': float(d_row[i, n_blocks - 1]),
                })

    assert n_blocks is not None
    delta = np.concatenate(delta_rows, axis=0) if delta_rows else np.zeros((0, 1))
    rho = np.concatenate(rho_rows, axis=0) if rho_rows else np.zeros((0, 1))
    uids = np.concatenate(uid_rows, axis=0) if uid_rows else np.zeros((0,), dtype=np.int64)
    assert delta.shape == rho.shape and delta.shape[0] == uids.size
    return {
        'n_blocks': n_blocks,
        'delta': delta,
        'rho': rho,
        'uids': uids,
        'qualitative': qual,
    }


def plot_layer_mean_ci(
    summary: Dict[str, np.ndarray],
    ylabel: str,
    title: str,
    save_path: Optional[Path] = None,
) -> None:
    """Plot UID means with UID-bootstrap 95% confidence bands."""
    mean = summary['mean']
    low = summary['low']
    high = summary['high']
    assert mean.ndim == low.ndim == high.ndim == 1
    layers = np.arange(1, mean.size + 1)
    fig, ax = plt.subplots(figsize=(8.2, 4.2))
    ax.plot(layers, mean, marker='o', color=COLOR_PRIMARY, label=ylabel)
    ax.fill_between(layers, low, high, color=COLOR_PRIMARY, alpha=0.22, label='UID bootstrap 95% CI')
    ax.set_xlabel('decoder layer')
    ax.set_ylabel(ylabel)
    ax.set_xticks(list(layers))
    ax.legend(loc='best')
    ax.set_title(title)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=PLOT_DPI, bbox_inches='tight')
        print(f'  Saved figure: {save_path}')
    plt.show()
    plt.close(fig)


@torch.no_grad()
def collect_pooled_embeddings(
    model: ProbeSystem,
    loader: DataLoader,
    data_key: str,
    max_batches: Optional[int] = None,
    seed: int = 2026,
) -> Tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    """Collect pooled vectors and text UIDs from an untouched sequential split."""
    model.eval()
    kv_l: List[torch.Tensor] = []
    tx_l: List[torch.Tensor] = []
    uid_l: List[np.ndarray] = []
    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= int(max_batches):
            break
        batch = move_to_device(batch, model.device)
        _z, ei_kv, _kv_mask, _h, _text_mask, ei_text = model.encode_kv(
            batch, data_key, mix_seed=int(seed) + int(bi),
        )
        kv_l.append(ei_kv.detach().float().cpu())
        tx_l.append(ei_text.detach().float().cpu())
        uid_l.append(batch['text uid'].detach().cpu().numpy().reshape(-1))
    assert kv_l and tx_l and uid_l
    ei_kv = torch.cat(kv_l, dim=0)
    ei_text = torch.cat(tx_l, dim=0)
    uids = np.concatenate(uid_l, axis=0)
    assert ei_kv.shape == ei_text.shape and ei_kv.shape[0] == uids.size
    return ei_kv, ei_text, uids


def aggregate_embeddings_by_uid(
    ei_kv: torch.Tensor,
    ei_text: torch.Tensor,
    uids: np.ndarray,
) -> Tuple[torch.Tensor, torch.Tensor, np.ndarray]:
    """Average repeated EEG/text trials within UID before inference."""
    uid_arr = np.asarray(uids).reshape(-1)
    unique_uids = np.asarray(sorted(set(uid_arr.tolist())))
    kv_uid = torch.stack([
        ei_kv[torch.from_numpy(np.flatnonzero(uid_arr == uid))].mean(dim=0) for uid in unique_uids
    ])
    text_uid = torch.stack([
        ei_text[torch.from_numpy(np.flatnonzero(uid_arr == uid))].mean(dim=0) for uid in unique_uids
    ])
    assert kv_uid.shape == text_uid.shape and kv_uid.shape[0] == unique_uids.size
    return kv_uid, text_uid, unique_uids


def run_overlap_and_frechet(
    model: ProbeSystem,
    loader: DataLoader,
    config: Dict[str, Any],
    output_dir: Path,
    llm_key: str,
) -> Dict[str, Any]:
    """Full-test UID-level paired overlap inference and shrinkage Fréchet."""
    seed = int(config.get('seed', 2026))
    raw_kv, raw_text, raw_uids = collect_pooled_embeddings(
        model, loader, 'eeg', max_batches=None, seed=seed,
    )
    ei_kv, ei_text, uids = aggregate_embeddings_by_uid(raw_kv, raw_text, raw_uids)
    paired, random_trials = semantic_overlap_vs_random(
        ei_kv,
        ei_text,
        uids,
        n_derangements=int(config.get('derangement_trials', 256)),
        seed=seed,
    )
    plot_overlap_kde(
        paired,
        random_trials.reshape(-1),
        title=f'UID semantic vs deranged overlap — {llm_key}',
        save_path=output_dir / 'overlap_kde.png',
    )
    inference = paired_permutation_inference(
        paired,
        random_trials,
        permutation_samples=int(config.get('permutation_samples', 10000)),
        bootstrap_samples=int(config.get('bootstrap_samples', 1000)),
        seed=seed,
    )
    x = F.normalize(ei_kv, dim=-1).numpy()
    y = F.normalize(ei_text, dim=-1).numpy()
    x_pca, y_pca, pca_info = reduce_embeddings_for_frechet(
        x, y, frechet_dim=int(config.get('frechet_dim', 64)), seed=seed,
    )
    fd = _frechet_distance(x_pca, y_pca)
    fd_low, fd_high = bootstrap_frechet_ci(
        x_pca,
        y_pca,
        bootstrap_samples=int(config.get('frechet_bootstrap_samples', 100)),
        seed=seed + 17,
    )
    result: Dict[str, Any] = {
        'llm_key': llm_key,
        'n_trials': int(raw_kv.shape[0]),
        'n_uids': int(ei_kv.shape[0]),
        **inference,
        'frechet_pca': float(fd),
        'frechet_pca_ci_low': float(fd_low),
        'frechet_pca_ci_high': float(fd_high),
        'frechet_dim': int(pca_info['frechet_dim']),
        'frechet_explained_variance': float(pca_info['frechet_explained_variance']),
    }
    print(
        f'  Full test: trials={result["n_trials"]} unique_uids={result["n_uids"]} D={int(ei_kv.shape[1])}\n'
        f'  Paired={inference["paired_mean"]:.4f} '
        f'CI[{inference["paired_ci_low"]:.4f}, {inference["paired_ci_high"]:.4f}]\n'
        f'  Deranged={inference["random_mean"]:.4f} '
        f'CI[{inference["random_ci_low"]:.4f}, {inference["random_ci_high"]:.4f}]\n'
        f'  Paired difference={inference["mean_difference"]:.4f} '
        f'CI[{inference["difference_ci_low"]:.4f}, {inference["difference_ci_high"]:.4f}] '
        f'p_perm={inference["permutation_p"]:.4g} dz={inference["cohens_dz"]:.4f}\n'
        f'  PCA-space shrinkage Frechet={fd:.4f} CI[{fd_low:.4f}, {fd_high:.4f}] '
        f'dim={result["frechet_dim"]} explained={result["frechet_explained_variance"]:.2%}'
    )
    return result


class LinearTokenProbe(nn.Module):
    """Linear readout: frozen H_{-1} (B, D) -> vocab logits (B, V)."""

    def __init__(self, hidden_dim: int, vocab_size: int) -> None:
        """Build a single linear layer ``hidden_dim → vocab_size``.

        Args:
            hidden_dim: Decoder hidden width.
            vocab_size: Tokenizer vocabulary size.
        """
        super().__init__()
        self.fc = nn.Linear(int(hidden_dim), int(vocab_size))

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """Map pooled hidden states to vocab logits.

        Args:
            h: ``(B, hidden_dim)``.

        Returns:
            Logits ``(B, vocab_size)``.
        """
        assert h.ndim == 2 and int(h.shape[-1]) == int(self.fc.in_features)
        logits = self.fc(h)
        assert logits.shape == (h.shape[0], int(self.fc.out_features))
        return logits


@torch.no_grad()
def cache_probe_features(
    model: ProbeSystem,
    loader: DataLoader,
    data_key: str,
    layer_ids: Sequence[int],
    seed: int = 2026,
) -> Dict[int, Tuple[torch.Tensor, torch.Tensor]]:
    """
    Cache H_ell[:, -1] and t* on CPU for each 1-based decoder block in layer_ids.

    Returns:
        {block: (features (N, D), labels (N,))}
    """
    model.eval()
    feats: Dict[int, List[torch.Tensor]] = {int(k): [] for k in layer_ids}
    labels: List[torch.Tensor] = []
    n_blocks: Optional[int] = None
    for bi, batch in enumerate(loader):
        batch = move_to_device(batch, model.device)
        st = model.decoder_states_at_minus1(batch, data_key, mix_seed=seed + int(bi))
        if n_blocks is None:
            n_blocks = int(st['n_blocks'])
        labels.append(st['t_star'].detach().cpu().long())
        n_states = len(st['hidden_m1'])
        for block in layer_ids:
            idx = hidden_index_for_block(n_states, int(n_blocks), int(block))
            feats[int(block)].append(st['hidden_m1'][idx].detach().cpu())
    y = torch.cat(labels, dim=0)
    out: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}
    for block, chunks in feats.items():
        x = torch.cat(chunks, dim=0)
        assert x.shape[0] == y.shape[0]
        out[int(block)] = (x, y)
    return out


def _probe_loader(
    x: torch.Tensor,
    y: torch.Tensor,
    batch_size: int,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    """Wrap feature/label tensors in a DataLoader.

    Args:
        x: Features ``(N, D)``.
        y: Integer labels ``(N,)``.
        batch_size: Loader batch size.
        shuffle: If True, shuffle with ``seed``.
        seed: Generator seed when shuffling.

    Returns:
        ``DataLoader`` over ``TensorDataset(x, y)``.
    """
    ds = TensorDataset(x, y)
    gen = torch.Generator()
    gen.manual_seed(int(seed))
    return DataLoader(ds, batch_size=int(batch_size), shuffle=shuffle, generator=gen if shuffle else None)


def standardize_probe_splits(
    train_x: torch.Tensor,
    val_x: torch.Tensor,
    test_x: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Standardize every split with training-feature statistics only."""
    assert train_x.ndim == val_x.ndim == test_x.ndim == 2
    assert train_x.shape[1] == val_x.shape[1] == test_x.shape[1]
    mean = train_x.float().mean(dim=0, keepdim=True)
    std = train_x.float().std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-6)
    standardized = tuple((x.float() - mean) / std for x in (train_x, val_x, test_x))
    assert all(bool(torch.isfinite(x).all()) for x in standardized)
    return standardized[0], standardized[1], standardized[2], mean, std


def train_linear_probe(
    train_xy: Tuple[torch.Tensor, torch.Tensor],
    val_xy: Tuple[torch.Tensor, torch.Tensor],
    test_xy: Tuple[torch.Tensor, torch.Tensor],
    device: torch.device,
    config: Dict[str, Any],
    title: str,
    save_path: Optional[Path] = None,
) -> Dict[str, Any]:
    """Train a standardized linear token probe with leakage-safe diagnostics."""
    x_tr_raw, y_tr = train_xy
    x_va_raw, y_va = val_xy
    x_te_raw, y_te = test_xy
    assert x_tr_raw.ndim == 2 and y_tr.ndim == 1 and x_tr_raw.shape[0] == y_tr.shape[0]
    assert x_va_raw.shape[0] == y_va.shape[0] and x_te_raw.shape[0] == y_te.shape[0]
    assert '_probe_vocab' in config, 'vocabulary dimension must come from the LLM, not labels'
    vocab = int(config['_probe_vocab'])
    d_model = int(x_tr_raw.shape[1])
    assert vocab >= 2 and d_model >= 1
    for split_name, labels in (('train', y_tr), ('val', y_va), ('test', y_te)):
        assert labels.ndim == 1 and labels.numel() >= 1
        assert int(labels.min()) >= 0 and int(labels.max()) < vocab, split_name
    x_tr, x_va, x_te, train_mean, train_std = standardize_probe_splits(
        x_tr_raw, x_va_raw, x_te_raw,
    )
    epochs = int(config.get('probe_epochs', 100))
    patience = int(config.get('probe_patience', 10))
    bs = int(config.get('probe_batch_size', 256))
    seed = int(config.get('seed', 2026))
    grad_clip = float(config.get('probe_grad_clip', 1.0))
    chance_ce = math.log(float(vocab))
    majority_token = int(torch.bincount(y_tr.long(), minlength=vocab).argmax().item())
    majority_acc = float((y_te.long() == majority_token).float().mean().item())

    def _evaluate(probe: LinearTokenProbe, loader: DataLoader) -> Tuple[float, float]:
        """Mean CE and accuracy of a probe on one split.

        Args:
            probe: Trained or candidate linear probe.
            loader: Feature/label loader.

        Returns:
            ``(mean_ce, accuracy)``.
        """
        probe.eval()
        ce_sum = 0.0
        n_tok = 0
        n_ok = 0
        with torch.no_grad():
            for xb, yb in loader:
                logits = probe(xb.to(device, non_blocking=True))
                assert torch.isfinite(logits).all(), 'non-finite probe logits'
                yb = yb.to(device, non_blocking=True)
                ce = F.cross_entropy(logits, yb, reduction='sum')
                assert torch.isfinite(ce)
                ce_sum += float(ce.cpu())
                n_tok += int(yb.numel())
                n_ok += int((logits.argmax(dim=-1) == yb).sum().item())
        assert n_tok >= 1
        return ce_sum / float(n_tok), float(n_ok) / float(n_tok)

    def _fit(
        train_labels: torch.Tensor,
        validation_labels: torch.Tensor,
        run_seed: int,
        run_title: str,
        verbose: bool,
    ) -> Tuple[LinearTokenProbe, Dict[str, List[float]], float]:
        """Train one linear probe with early stopping on validation CE.

        Args:
            train_labels: Train token ids (true or shuffled).
            validation_labels: Val token ids matching the same scheme.
            run_seed: RNG seed for this fit.
            run_title: Log prefix.
            verbose: If True, print the selected epoch.

        Returns:
            ``(probe, history, best_val_ce)``.
        """
        assert train_labels.shape == y_tr.shape and validation_labels.shape == y_va.shape
        set_seed(run_seed)
        probe = LinearTokenProbe(d_model, vocab).to(device)
        opt = AdamW(
            probe.parameters(),
            lr=float(config.get('probe_lr', 1e-3)),
            weight_decay=float(config.get('weight_decay', 0.05)),
        )
        train_loader = _probe_loader(x_tr, train_labels, bs, True, run_seed)
        val_loader = _probe_loader(x_va, validation_labels, bs, False, run_seed)
        history: Dict[str, List[float]] = {'train_ce': [], 'val_ce': [], 'val_acc': []}
        best_val = float('inf')
        best_state: Optional[Dict[str, torch.Tensor]] = None
        bad = 0
        for epoch in range(1, epochs + 1):
            probe.train()
            train_sum = 0.0
            train_count = 0
            for xb, yb in train_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = probe(xb)
                assert torch.isfinite(logits).all(), 'non-finite probe logits'
                loss = F.cross_entropy(logits, yb)
                assert torch.isfinite(loss)
                loss.backward()
                for parameter in probe.parameters():
                    assert parameter.grad is not None and torch.isfinite(parameter.grad).all()
                torch.nn.utils.clip_grad_norm_(probe.parameters(), grad_clip)
                opt.step()
                train_sum += float(loss.detach().cpu()) * int(yb.shape[0])
                train_count += int(yb.shape[0])
            train_ce = train_sum / float(train_count)
            val_ce, val_acc = _evaluate(probe, val_loader)
            history['train_ce'].append(train_ce)
            history['val_ce'].append(val_ce)
            history['val_acc'].append(val_acc)
            if verbose and (epoch == 1 or epoch % 10 == 0):
                print(
                    f'    [{run_title}] epoch {epoch:03d} train_ce={train_ce:.4f} '
                    f'val_ce={val_ce:.4f} val_acc={val_acc:.3f}'
                )
            if val_ce < best_val - 1e-6:
                best_val = val_ce
                bad = 0
                best_state = {key: value.detach().cpu().clone() for key, value in probe.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    if verbose:
                        print(f'    [{run_title}] early stop at epoch {epoch} (patience={patience})')
                    break
        assert best_state is not None and math.isfinite(best_val)
        probe.load_state_dict(best_state, strict=True)
        return probe, history, best_val

    probe, history, best_val = _fit(y_tr, y_va, seed, title, verbose=True)
    test_loader = _probe_loader(x_te, y_te, bs, False, seed)
    te_ce, te_acc = _evaluate(probe, test_loader)
    diagnostic_limit = chance_ce + float(config.get('probe_ce_margin', 5.0))
    if te_ce > diagnostic_limit:
        raise RuntimeError(
            f'{title}: test CE={te_ce:.4f} exceeds diagnostic limit={diagnostic_limit:.4f} '
            f'(chance log(V)={chance_ce:.4f}); refusing a pathological probe'
        )

    fig, ax = plt.subplots(1, 2, figsize=(10.8, 3.6))
    ax[0].plot(history['train_ce'], color=COLOR_PRIMARY, label='train CE')
    ax[0].plot(history['val_ce'], color=COLOR_SECONDARY, label='val CE')
    ax[0].axhline(chance_ce, color='black', linestyle=':', label='chance log(V)')
    ax[0].set_xlabel('epoch'); ax[0].set_title('CE'); ax[0].legend()
    ax[1].plot(history['val_acc'], color=COLOR_ACC, label='val acc')
    ax[1].set_xlabel('epoch'); ax[1].set_title('Val accuracy'); ax[1].legend()
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    probe.eval()
    with torch.no_grad():
        n_show = min(5, int(x_te.shape[0]))
        pred_ids = probe(x_te[:n_show].to(device)).argmax(dim=-1).cpu().tolist()
        gold_ids = y_te[:n_show].cpu().tolist()
    del probe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_shuffle_generator = torch.Generator().manual_seed(seed + 1)
    val_shuffle_generator = torch.Generator().manual_seed(seed + 2)
    shuffled_train_labels = y_tr[
        torch.randperm(y_tr.numel(), generator=train_shuffle_generator)
    ]
    shuffled_val_labels = y_va[
        torch.randperm(y_va.numel(), generator=val_shuffle_generator)
    ]
    shuffled_probe, _shuffled_history, _shuffled_val = _fit(
        shuffled_train_labels,
        shuffled_val_labels,
        seed + 1,
        f'{title} shuffled-label',
        verbose=False,
    )
    shuffled_ce, shuffled_acc = _evaluate(shuffled_probe, test_loader)
    del shuffled_probe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(
        f'    baselines: chance log(V)={chance_ce:.4f}; train-majority test acc={majority_acc:.3f}; '
        f'shuffled-label test CE/acc={shuffled_ce:.4f}/{shuffled_acc:.3f}'
    )
    return {
        'val_ce': float(best_val),
        'test_ce': float(te_ce),
        'test_acc': float(te_acc),
        'chance_log_vocab': float(chance_ce),
        'majority_test_acc': float(majority_acc),
        'shuffled_test_ce': float(shuffled_ce),
        'shuffled_test_acc': float(shuffled_acc),
        'train_standardized_abs_mean_max': float(x_tr.mean(dim=0).abs().max().item()),
        'train_scale_min': float(train_std.min().item()),
        'pred_ids': pred_ids,
        'gold_ids': gold_ids,
    }


def run_linear_probes(
    model: ProbeSystem,
    dm: 'PhaseDataModule',
    config: Dict[str, Any],
    output_dir: Path,
    llm_key: str,
) -> List[Dict[str, Any]]:
    """Leakage-safe linear probes for independent EEG and oracle text paths."""
    seed = int(config.get('seed', 2026))
    every = int(config.get('probe_every', 3))
    bs_feat = int(config.get('batch_size', 72))
    n_workers = int(config.get('num_workers', 0))
    assert dm.train_set is not None and dm.val_set is not None and dm.test_set is not None
    loaders = {
        'train': sequential_dataloader(dm.train_set, bs_feat, n_workers),
        'val': sequential_dataloader(dm.val_set, bs_feat, n_workers),
        'test': sequential_dataloader(dm.test_set, bs_feat, n_workers),
    }
    n_blocks = model.num_decoder_layers()
    layer_ids = probe_layer_ids(n_blocks, every)
    print(f'  Linear probe layers (1-based): {layer_ids}  (n_blocks={n_blocks})')
    cfg_probe = dict(config)
    cfg_probe['_probe_vocab'] = model.vocab_size()
    rows: List[Dict[str, Any]] = []
    tok = model.tokenizer

    for data_key in ('eeg', 'oracle_text'):
        print(f'\n  Caching {llm_key} / {data_key} probe features ...')
        model.set_data_key(data_key)
        cache = {
            split: cache_probe_features(model, loaders[split], data_key, layer_ids, seed=seed)
            for split in ('train', 'val', 'test')
        }
        acc_by_layer: List[float] = []
        for block in layer_ids:
            title = f'{llm_key} / {data_key} / L{block}'
            print(f'  Training linear probe {title}')
            metrics = train_linear_probe(
                cache['train'][block],
                cache['val'][block],
                cache['test'][block],
                device=model.device,
                config=cfg_probe,
                title=title,
                save_path=output_dir / f'probe_{data_key}_L{block}.png',
            )
            acc_by_layer.append(float(metrics['test_acc']))
            print(
                f'    test CE={metrics["test_ce"]:.4f}  test acc={metrics["test_acc"]:.3f}'
            )
            print('    Qualitative first-token (test head):')
            for gold, pred in zip(metrics['gold_ids'], metrics['pred_ids']):
                g_s = tok.decode([int(gold)], skip_special_tokens=False)
                p_s = tok.decode([int(pred)], skip_special_tokens=False)
                print(f'      gold={g_s!r}  pred={p_s!r}')
            rows.append({
                'llm_key': llm_key,
                'data_key': data_key,
                'layer': int(block),
                'test_ce': float(metrics['test_ce']),
                'test_acc': float(metrics['test_acc']),
                'chance_log_vocab': float(metrics['chance_log_vocab']),
                'majority_test_acc': float(metrics['majority_test_acc']),
                'shuffled_test_ce': float(metrics['shuffled_test_ce']),
                'shuffled_test_acc': float(metrics['shuffled_test_acc']),
            })
        fig, ax = plt.subplots(figsize=(7.4, 3.8))
        ax.plot(layer_ids, acc_by_layer, marker='o', color=COLOR_PRIMARY)
        ax.set_xlabel('decoder layer')
        ax.set_ylabel('test top-1 acc')
        ax.set_title(f'Linear probe accuracy — {llm_key} / {data_key}')
        ax.set_xticks(list(layer_ids))
        ax.set_ylim(0.0, 1.05)
        fig.tight_layout()
        fig.savefig(output_dir / f'probe_acc_{data_key}.png', dpi=PLOT_DPI, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        del cache
        gc.collect()

    print('\n  Linear probe summary (display only; no metric file):')
    display(pd.DataFrame(rows))
    return rows


def run_faithfulness_analyses(
    model: ProbeSystem,
    dm: 'PhaseDataModule',
    config: Dict[str, Any],
    output_dir: Path,
    llm_key: str,
    lens: TunedLens,
) -> None:
    """Run UID-aware faithfulness analyses on the untouched full test split."""
    output_dir.mkdir(parents=True, exist_ok=True)
    assert dm.test_set is not None
    test_loader = dm.test_dataloader()
    seed = int(config.get('seed', 2026))
    n_boot = int(config.get('bootstrap_samples', 1000))

    print('\n===== (a)(b) Tuned lens + Spearman (full test, token -1) =====')
    stats_ab = collect_two_path_layer_stats(model, test_loader, config, lens, max_batches=None)
    delta_uid, uids = aggregate_rows_by_uid(stats_ab['delta'], stats_ab['uids'])
    rho_uid, rho_uids = aggregate_rows_by_uid(stats_ab['rho'], stats_ab['uids'])
    assert np.array_equal(uids, rho_uids)
    delta_summary = bootstrap_layer_summary(delta_uid, n_boot, seed)
    rho_summary = bootstrap_layer_summary(rho_uid, n_boot, seed + 1)
    plot_layer_mean_ci(
        delta_summary,
        ylabel=r'$\Delta_\ell(t^*)$ oracle $-$ eeg',
        title=f'Tuned-lens difference at $t^*$ — {llm_key}',
        save_path=output_dir / 'logit_diff.png',
    )
    plot_layer_mean_ci(
        rho_summary,
        ylabel='Spearman rho',
        title=f'Spearman(H_oracle, H_eeg) at pos -1 — {llm_key}',
        save_path=output_dir / 'spearman.png',
    )
    layer_rows: List[Dict[str, Any]] = []
    for layer_idx in range(delta_uid.shape[1]):
        layer_rows.append({
            'llm_key': llm_key,
            'layer': layer_idx + 1,
            'uid_n': int(uids.size),
            'delta_finite_n': int(delta_summary['finite_n'][layer_idx]),
            'delta_mean': float(delta_summary['mean'][layer_idx]),
            'delta_ci_low': float(delta_summary['low'][layer_idx]),
            'delta_ci_high': float(delta_summary['high'][layer_idx]),
            'spearman_finite_n': int(rho_summary['finite_n'][layer_idx]),
            'spearman_mean': float(rho_summary['mean'][layer_idx]),
            'spearman_ci_low': float(rho_summary['low'][layer_idx]),
            'spearman_ci_high': float(rho_summary['high'][layer_idx]),
        })
    print('  UID-aggregated layer statistics (display only):')
    display(pd.DataFrame(layer_rows))
    print('  Qualitative final official-logit first token (oracle vs EEG):')
    for i, row in enumerate(stats_ab['qualitative']):
        print(
            f'  --- example {i + 1} ---\n'
            f'    GT      : {row["gt"]}\n'
            f'    t*      : {row["t_star"]!r}\n'
            f'    oracle argmax: {row["oracle_argmax"]!r}\n'
            f'    eeg    argmax: {row["eeg_argmax"]!r}\n'
            f'    Delta_L(t*): {row["delta_last"]:.4f}'
        )

    print('\n===== (c) Full-test UID overlap + shrinkage Frechet =====')
    overlap = run_overlap_and_frechet(model, test_loader, config, output_dir, llm_key)
    display(pd.DataFrame([overlap]))

    print('\n===== (d) Standardized linear probes every 3 decoder layers =====')
    _ = run_linear_probes(model, dm, config, output_dir, llm_key)


In [ ]:
# =============================================================
# Smokes: encoder-only then per-LLM (fail-fast)
# =============================================================

def _make_dummy_probe_batch(device: torch.device, batch_size: int = 2) -> Dict[str, Any]:
    """Synthetic independent rows for offline and per-LLM smoke tests."""
    B = int(batch_size)
    assert B >= 2
    T, C = 1280, 128
    base_texts = [
        'hello world extra words here',
        'goodbye moon and stars tonight',
        'third unique sentence for batching',
    ]
    texts = [base_texts[i % len(base_texts)] + f' {i}' for i in range(B)]
    subjects = [ALL_SUBJECTS[i % len(ALL_SUBJECTS)] for i in range(B)]
    return {
        'eeg': torch.randn(B, T, C, device=device),
        'mask': torch.ones(B, T, dtype=torch.int32, device=device),
        'input text': texts,
        'target text': list(texts),
        'text uid': torch.arange(1, B + 1, dtype=torch.long, device=device),
        'prompt': [('<NR>', 'ZuCo1', subject) for subject in subjects],
        'dataset_raw': ['ZuCo1'] * B,
        'task_raw': ['task1'] * B,
        'subject_raw': subjects,
        'row_index': torch.arange(B, dtype=torch.long, device=device),
    }


def _smoke_glim_sampler() -> None:
    """Assert exact coverage, unique UIDs, correct length, and a final partial batch."""
    class _Dummy(Dataset):
        """Tiny dataset with repeated UIDs for GLIM packing tests."""

        def __init__(self) -> None:
            """Assign eight rows to five text UIDs."""
            self.text_uid = [0, 0, 0, 1, 1, 2, 3, 4]

        def __len__(self) -> int:
            """Number of dummy rows."""
            return len(self.text_uid)

        def __getitem__(self, idx: int) -> int:
            """Return the index itself (sampler only needs length / UIDs).

            Args:
                idx: Row index.

            Returns:
                The same index as an int.
            """
            return int(idx)

    ds = _Dummy()
    ids = _dataset_text_uids(ds)
    sampler = GLIMSampler(
        ds, ids, phase='val', batch_size=3, seed=2026, shuffle=False,
    )
    batches = list(iter(sampler))
    assert len(batches) == len(sampler) == 3
    assert [len(batch) for batch in batches] == [3, 3, 2], batches
    assert sorted(index for batch in batches for index in batch) == list(range(len(ds)))
    for batch in batches:
        uids = [ids[index] for index in batch]
        assert len(uids) == len(set(uids))


def _smoke_data_reliability() -> None:
    """Exercise B=3 prompts, idempotent patching, split leakage, and EEG masks."""
    prompts = [
        ('<NR>', 'ZuCo1', 'ZAB'),
        ('<TSR>', 'ZuCo2', 'ZDM'),
        ('<NR>', 'ZuCo1', 'ZDN'),
    ]
    tasks, datasets, subjects = unpack_prompt_batch(prompts)
    assert tasks == ['<NR>', '<TSR>', '<NR>']
    assert datasets == ['ZuCo1', 'ZuCo2', 'ZuCo1']
    assert subjects == ['ZAB', 'ZDM', 'ZDN']
    assert canonicalize_text_uid('01') == canonicalize_text_uid('1') == 1
    assert canonicalize_text_uid('+001') == 1
    assert canonicalize_text_uid(np.int64(2)) == 2
    assert canonicalize_text_uid(3.0) == 3
    assert canonicalize_text_uid(_INT64_MIN) == _INT64_MIN
    assert canonicalize_text_uid(str(_INT64_MAX)) == _INT64_MAX
    assert canonicalize_text_uid(float(_FLOAT_SAFE_INTEGER_MAX)) == _FLOAT_SAFE_INTEGER_MAX
    assert normalize_text_for_split_check('Ａ  Strasse') == 'a strasse'
    for invalid_uid in (
        None, True, 1.5, '1.0', 'abc', '',
        _INT64_MAX + 1, _INT64_MIN - 1,
        str(_INT64_MAX + 1), str(_INT64_MIN - 1),
        float(2 ** 53), -float(2 ** 53),
    ):
        try:
            canonicalize_text_uid(invalid_uid)
        except AssertionError:
            pass
        else:
            raise AssertionError(f'invalid UID must fail: {invalid_uid!r}')
    for null_text in (None, pd.NA, float('nan')):
        try:
            normalize_text_for_split_check(null_text)
        except AssertionError:
            pass
        else:
            raise AssertionError(f'null input text must fail: {null_text!r}')
    requested = './explicit/requested.df'
    assert _collect_data_candidates(requested)[0] == Path(requested)
    original_index = _pd_indexes_base._eeg_faith_original_new_index  # type: ignore[attr-defined]
    original_block = _pd_libinternals._eeg_faith_original_unpickle_block  # type: ignore[attr-defined]
    _patch_pandas_pickle_compat()
    _patch_pandas_pickle_compat()
    assert _pd_indexes_base._eeg_faith_original_new_index is original_index  # type: ignore[attr-defined]
    assert _pd_libinternals._eeg_faith_original_unpickle_block is original_block  # type: ignore[attr-defined]
    eeg = np.zeros((1280, 128), dtype=np.float32)
    mask = np.ones((1280,), dtype=np.int32)
    validate_eeg_mask_sample(eeg, mask)
    bad_mask = mask.copy(); bad_mask[0] = 2
    try:
        validate_eeg_mask_sample(eeg, bad_mask)
    except AssertionError:
        pass
    else:
        raise AssertionError('non-binary mask smoke must fail')
    rows: List[Dict[str, Any]] = []
    for phase_idx, phase in enumerate(('train', 'val', 'test')):
        for row_idx in range(2):
            uid = phase_idx * 10 + row_idx
            rows.append({
                'phase': phase,
                'text uid': uid,
                'input text': f'Unique {phase} sentence {row_idx}',
                'eeg': eeg,
                'mask': mask,
            })
    valid_rows = pd.DataFrame(rows)
    validate_corpus_splits(valid_rows)
    assert all(type(uid) is int for uid in valid_rows['text uid'])
    validate_all_eeg_mask_rows(valid_rows)
    uid_conflict = pd.DataFrame(rows)
    uid_conflict['text uid'] = uid_conflict['text uid'].astype(object)
    uid_conflict.at[uid_conflict.index[0], 'text uid'] = '01'
    uid_conflict.at[uid_conflict.index[1], 'text uid'] = '1'
    try:
        validate_corpus_splits(uid_conflict)
    except AssertionError as exc:
        assert 'multiple normalized texts' in str(exc)
    else:
        raise AssertionError('canonical-equivalent UID conflict must fail')
    bad_later = valid_rows.copy()
    late_mask = mask.copy()
    late_mask[-1] = 2
    bad_later.at[bad_later.index[-1], 'mask'] = late_mask
    try:
        validate_all_eeg_mask_rows(bad_later)
    except AssertionError as exc:
        assert 'position=5' in str(exc)
    else:
        raise AssertionError('streaming validator must reject a bad later row')
    leaking = pd.DataFrame(rows)
    leaking.loc[leaking['phase'].eq('val').idxmax(), 'input text'] = rows[0]['input text']
    try:
        validate_corpus_splits(leaking)
    except AssertionError:
        pass
    else:
        raise AssertionError('cross-phase normalized-text leakage smoke must fail')


def run_smoke_tests() -> Dict[str, bool]:
    """Dependency-light correctness smokes without loading data or an LLM."""
    results: Dict[str, bool] = {}
    set_seed(2026)
    _smoke_data_reliability()
    results['data_reliability'] = True
    print('  OK data reliability / canonical UID/text / mask / split checks')
    selected = select_llm_configs(
        LLM_CONFIGS, {**CONFIG, 'llm_keys_to_run': ['bart_large', 'flan_t5_large']},
    )
    assert [item['key'] for item in selected] == ['bart_large', 'flan_t5_large']
    invalid_selections = [
        (LLM_CONFIGS, {**CONFIG, 'llm_keys_to_run': ['bart_large', 'bart_large']}),
        (LLM_CONFIGS, {**CONFIG, 'llm_keys_to_run': ['unknown']}),
        (LLM_CONFIGS, {**CONFIG, 'epochs': 0}),
        ([LLM_CONFIGS[0], dict(LLM_CONFIGS[0])], CONFIG),
    ]
    for bad_configs, bad_config in invalid_selections:
        try:
            select_llm_configs(bad_configs, bad_config)
        except AssertionError:
            pass
        else:
            raise AssertionError('invalid LLM selection/config must fail')
    assert _sha256_json({'b': 2, 'a': 1}) == _sha256_json({'a': 1, 'b': 2})
    assert _immutable_revision_digest('a' * 40) == 'a' * 40
    assert _immutable_revision_digest('B' * 64) == 'b' * 64
    for mutable_or_malformed in (
        None, 'main', 'master', 'a' * 39, 'a' * 41, 'f' * 63, 'f' * 65,
        'g' * 40, ('a' * 40) + ' ', 123,
    ):
        assert _immutable_revision_digest(mutable_or_malformed) is None

    class _FakeBackend:
        """tokenizers-backend stub whose ``to_str`` is hashed for behavior."""

        def __init__(self, payload: str) -> None:
            """Store a fixed JSON payload.

            Args:
                payload: String returned by ``to_str``.
            """
            self.payload = payload

        def to_str(self) -> str:
            """Return the stored backend JSON.

            Returns:
                Behavior-fingerprint payload.
            """
            return self.payload

    class _FakeTokenizer:
        """Tokenizer stub with backend JSON, commit, and added vocab."""

        def __init__(self, backend_payload: str, added_id: int = 7) -> None:
            """Attach a fake backend and a single added token.

            Args:
                backend_payload: JSON string for the backend stub.
                added_id: Id stored under ``<added>``.
            """
            self.backend_tokenizer = _FakeBackend(backend_payload)
            self.init_kwargs = {'_commit_hash': 'tok-commit', 'revision': 'tok-revision'}
            self.special_tokens_map_extended = {'eos_token': '</s>'}
            self.added_id = int(added_id)

        def get_added_vocab(self) -> Dict[str, int]:
            """Return the single added-token map.

            Returns:
                ``{'<added>': added_id}``.
            """
            return {'<added>': self.added_id}

    behavior_a = _tokenizer_behavior_fingerprint(_FakeTokenizer('backend-a'), 'encdec_t5')
    behavior_a_repeat = _tokenizer_behavior_fingerprint(_FakeTokenizer('backend-a'), 'encdec_t5')
    behavior_b = _tokenizer_behavior_fingerprint(_FakeTokenizer('backend-b'), 'encdec_t5')
    behavior_added = _tokenizer_behavior_fingerprint(
        _FakeTokenizer('backend-a', added_id=8), 'encdec_t5',
    )
    assert behavior_a == behavior_a_repeat
    assert behavior_a['behavior_sha256'] != behavior_b['behavior_sha256']
    assert behavior_a['behavior_sha256'] != behavior_added['behavior_sha256']
    assert behavior_a['_commit_hash'] == 'tok-commit'
    assert behavior_a['revision'] == 'tok-revision'
    try:
        _tokenizer_behavior_fingerprint(object(), 'encdec_bart')
    except AssertionError:
        pass
    else:
        raise AssertionError('supported tokenizer must expose a model representation')
    results['llm_selection_and_hashing'] = True
    print('  OK LLM key validation / deterministic behavior fingerprints')
    B, T, C = 2, 1280, 128
    eeg = torch.randn(B, T, C, device=DEVICE)
    mask = torch.ones(B, T, dtype=torch.int32, device=DEVICE)
    batch = {
        'eeg': eeg,
        'mask': mask,
        'input text': ['hello world extra words here', 'goodbye moon and stars tonight'],
        'target text': ['hello world extra words here', 'goodbye moon and stars tonight'],
        'text uid': torch.tensor([1, 2], dtype=torch.long),
        'prompt': [('<NR>', 'ZuCo1', 'ZAB'), ('<NR>', 'ZuCo1', 'ZDM')],
        'dataset_raw': ['ZuCo1', 'ZuCo1'],
        'task_raw': ['task1', 'task1'],
        'subject_raw': ['ZAB', 'ZDM'],
        'row_index': torch.tensor([0, 1], dtype=torch.long),
    }
    pe = PromptEmbedder(dim=CONFIG['prompt_dim'], prompt_keys=PROMPT_KEYS)
    enc = build_eeg_encoder(CONFIG)
    pe.to(DEVICE); enc.to(DEVICE)
    p_ids = pe.encode([
        [batch['prompt'][i][0] for i in range(B)],
        [batch['prompt'][i][1] for i in range(B)],
        [batch['prompt'][i][2] for i in range(B)],
    ], device=DEVICE)
    p = pe(p_ids)
    Zi, memory, _ = enc(eeg, mask, p)
    assert Zi.shape == (B, CONFIG['out_len'], CONFIG['hidden_dim']), tuple(Zi.shape)
    assert memory.shape[1] == CONFIG['in_len'], tuple(memory.shape)
    assert hasattr(enc, 'channel_weights')
    results['eeg_encoder_shapes'] = True
    print('  OK eeg_encoder_shapes', tuple(Zi.shape))

    D = 64
    ei = torch.randn(B, D, device=DEVICE)
    yi = torch.randn(B, D, device=DEVICE)
    loss = clip_info_nce(ei, yi)
    assert loss.ndim == 0
    results['clip_info_nce'] = True
    print('  OK clip_info_nce', float(loss))

    z_eq = torch.randn(B, 96, D, device=DEVICE)
    text_mask = torch.ones(B, 96, dtype=torch.long, device=DEVICE)
    text_mask[:, 48:] = 0
    z_mask_changed = z_eq.clone()
    z_mask_changed[:, 48:] = torch.randn_like(z_mask_changed[:, 48:]) * 100.0
    commit0 = token_commitment_mse(z_mask_changed, z_eq, text_mask)
    assert commit0.ndim == 0 and float(commit0) < 1e-6
    commit_r = token_commitment_mse(
        torch.randn(B, 96, D, device=DEVICE),
        torch.randn(B, 96, D, device=DEVICE),
        text_mask,
    )
    assert torch.isfinite(commit_r) and float(commit_r) > 0.0
    results['token_commitment_mse'] = True
    print('  OK masked token_commitment_mse', float(commit0), float(commit_r))

    w_clip = float(CONFIG['w_clip'])
    w_ar = float(CONFIG['w_ar'])
    w_c = float(CONFIG['commitment_weight'])
    assert abs(w_clip - 0.5) < 1e-12 and abs(w_ar - 0.5) < 1e-12
    assert abs(w_c - 0.7) < 1e-12
    dummy_clip = torch.tensor(1.0)
    dummy_ar = torch.tensor(2.0)
    dummy_commit = torch.tensor(3.0)
    total = w_clip * dummy_clip + w_ar * dummy_ar + w_c * dummy_commit
    assert abs(float(total) - (0.5 * 1.0 + 0.5 * 2.0 + 0.7 * 3.0)) < 1e-6
    results['e2e_loss_weights'] = True
    print('  OK e2e_loss_weights', float(total))

    dummy_lin = nn.Linear(4, 4)
    opt = AdamW(dummy_lin.parameters(), lr=float(CONFIG['lr']), weight_decay=float(CONFIG['weight_decay']))
    sch = setup_scheduler(opt, CONFIG)
    assert isinstance(sch, CosineAnnealingWarmRestarts)
    assert int(sch.T_0) == 15
    results['cosine_scheduler'] = True
    print('  OK cosine_scheduler T_0=15')

    _smoke_glim_sampler()
    results['glim_sampler'] = True
    print('  OK glim_sampler')

    assert CONFIG['decoder_prompt'] == 'Based on the following signals, translate the sentence'
    assert list(CONFIG['data_keys']) == ['eeg'] and 'oracle_text' in DATA_KEY_ALPHA
    bart_prefix = canonical_decoder_prefix_ids('encdec_bart', 2, 0, [11, 12])
    t5_prefix = canonical_decoder_prefix_ids('encdec_t5', 0, None, [11, 12])
    assert bart_prefix == [2, 0, 11, 12]
    assert t5_prefix == [0, 11, 12]
    assert lexical_text_after_instruction('hello', 'encdec_bart') == ' hello'
    assert lexical_text_after_instruction('  hello', 'encdec_bart') == ' hello'
    assert lexical_text_after_instruction('', 'encdec_bart') == ''
    assert lexical_text_after_instruction('hello', 'encdec_t5') == 'hello'
    targets_after_early_eos = [[7, 8, 1], [9, 1]]
    predicted_eos_at_first_step = torch.tensor([True, True])
    assert bool(predicted_eos_at_first_step.all())
    assert full_target_token_count(targets_after_early_eos) == 5
    _assert_exact_state_keys({'a', 'b'}, {'a', 'b'}, 'smoke exact keys')
    try:
        _assert_exact_state_keys({'a', 'extra'}, {'a', 'b'}, 'smoke strict rejection')
    except AssertionError:
        pass
    else:
        raise AssertionError('strict-key smoke must reject mismatch')
    reference = torch.zeros(2, 4)
    try:
        select_matching_unembed_form(torch.ones(2, 4), torch.full((2, 4), 2.0), reference, atol=1e-6)
    except AssertionError:
        pass
    else:
        raise AssertionError('Unembed fail-fast smoke must reject both forms')
    cfg_tl = TunedLensConfig(
        base_model_name_or_path='dummy', d_model=8, num_hidden_layers=2,
    )
    assert int(cfg_tl.num_hidden_layers) == 2
    results['decoder_order'] = True
    results['early_eos_denominator'] = True
    results['strict_keys'] = True
    results['unembed_fail_fast'] = True
    results['tuned_lens_config'] = True
    print('  OK decoder order / early-EOS denominator / strict keys / Unembed fail-fast')

    layers12 = probe_layer_ids(12, 3)
    layers24 = probe_layer_ids(24, 3)
    assert layers12 == [3, 6, 9, 12], layers12
    assert layers24 == [3, 6, 9, 12, 15, 18, 21, 24], layers24
    assert hidden_index_for_block(13, 12, 3) == 3
    assert hidden_index_for_block(13, 12, 12) == 12
    for bad_shape in ((14, 12, 12), (12, 12, 1)):
        try:
            hidden_index_for_block(*bad_shape)
        except AssertionError:
            pass
        else:
            raise AssertionError(f'hidden-state mapping must reject {bad_shape}')
    results['probe_layer_ids'] = True
    print('  OK probe_layer_ids', layers12, layers24)

    a = torch.arange(32, dtype=torch.float32).unsqueeze(0).repeat(4, 1)
    rho_id = spearman_rowwise(a, a)
    assert rho_id.shape == (4,)
    assert np.allclose(rho_id, 1.0, atol=1e-6), rho_id
    rng = torch.Generator().manual_seed(2026)
    noise = torch.randn(4, 32, generator=rng)
    rho_noise = spearman_rowwise(a, noise)
    assert float(np.nanmean(np.abs(rho_noise))) < 0.6
    results['spearman_rowwise'] = True
    print('  OK spearman_rowwise identical=1')

    rng2 = np.random.RandomState(2026)
    e1 = rng2.randn(24, 16).astype(np.float64)
    e2 = e1 + rng2.randn(24, 16) * 0.5
    e1_pca, e2_pca, pca_info = reduce_embeddings_for_frechet(
        e1, e2, frechet_dim=5, seed=2026,
    )
    assert e1_pca.shape == e2_pca.shape == (24, 5)
    assert int(pca_info['frechet_dim']) == 5
    assert 0.0 < float(pca_info['frechet_explained_variance']) <= 1.0
    fd0 = _frechet_distance(e1_pca, e1_pca)
    fd1 = _frechet_distance(e1_pca, e2_pca)
    fd_low, fd_high = bootstrap_frechet_ci(
        e1_pca, e2_pca, bootstrap_samples=12, seed=2026,
    )
    assert fd0 < 1e-6 and fd1 > fd0
    assert 0.0 <= fd_low <= fd_high and math.isfinite(fd_high)
    results['frechet_pca'] = True
    print('  OK common-PCA shrinkage Frechet', fd0, fd1, (fd_low, fd_high))

    kv = torch.randn(32, 8)
    tx = kv + 0.05 * torch.randn(32, 8)
    uids = np.arange(32)
    paired, random_trials = semantic_overlap_vs_random(
        kv, tx, uids, n_derangements=16, seed=2026,
    )
    assert paired.shape == (32,) and random_trials.shape == (16, 32)
    paired_stats = paired_permutation_inference(
        paired, random_trials, permutation_samples=500, bootstrap_samples=200, seed=2026,
    )
    assert float(paired_stats['mean_difference']) > 0.0
    assert math.isfinite(float(paired_stats['cohens_dz']))
    results['paired_statistics'] = True
    print('  OK UID paired/permutation statistics')

    train_x = torch.arange(40, dtype=torch.float32).reshape(10, 4)
    val_x = torch.full((3, 4), 100.0)
    test_x = torch.full((3, 4), -100.0)
    train_z, val_z, test_z, train_mean, train_std = standardize_probe_splits(train_x, val_x, test_x)
    assert torch.allclose(train_z.mean(dim=0), torch.zeros(4), atol=1e-6)
    assert torch.allclose(val_z, (val_x - train_mean) / train_std)
    assert torch.allclose(test_z, (test_x - train_mean) / train_std)
    results['probe_standardization'] = True
    print('  OK train-only probe standardization')

    set_seed(2026)
    d_h, vsz, n = 8, 12, 40
    x_all = torch.randn(n, d_h)
    y_all = torch.randint(0, vsz, (n,))
    cfg_p = {
        'probe_epochs': 3, 'probe_patience': 10, 'probe_lr': 1e-2,
        'probe_batch_size': 16, 'seed': 2026, 'weight_decay': 0.0,
        '_probe_vocab': vsz,
    }
    metrics = train_linear_probe(
        (x_all[:24], y_all[:24]),
        (x_all[24:32], y_all[24:32]),
        (x_all[32:], y_all[32:]),
        device=torch.device('cpu'),
        config=cfg_p,
        title='smoke linear probe',
        save_path=None,
    )
    assert math.isfinite(float(metrics['test_ce']))
    assert 0.0 <= float(metrics['test_acc']) <= 1.0
    assert abs(float(metrics['chance_log_vocab']) - math.log(vsz)) < 1e-8
    assert math.isfinite(float(metrics['shuffled_test_ce']))
    results['linear_probe_step'] = True
    print('  OK linear probe gradients/clipping/baselines', metrics['test_ce'], metrics['test_acc'])

    print('Encoder smoke results:', results)
    assert all(results.values())
    return results


def run_llm_smoke(
    llm_cfg: Dict[str, Any],
    config: Dict[str, Any],
    device: torch.device,
) -> Dict[str, bool]:
    """Load one frozen LLM and smoke E2E + two-path tuned lens on a tiny dummy batch."""
    key = str(llm_cfg['key'])
    print(f'\n[SMOKE LLM] {key} ({llm_cfg["repo_id"]}) family={llm_cfg["family"]}')
    results: Dict[str, bool] = {}
    set_seed(int(config.get('seed', 2026)))
    model: Optional[ProbeSystem] = None
    try:
        model = ProbeSystem(
            config=config,
            llm_key=key,
            repo_id=str(llm_cfg['repo_id']),
            family=str(llm_cfg['family']),
            device=device,
            source=str(llm_cfg.get('source', 'hf_mirror')),
            modelscope_id=llm_cfg.get('modelscope_id'),
        )
        model.to(device)
        model.eval()
        D = int(model.embed_dim)
        n_blocks = model.num_decoder_layers()
        print(f'  OK loaded D={D} n_decoder_blocks={n_blocks}')
        assert n_blocks >= 3

        batch = _make_dummy_probe_batch(device, batch_size=3)
        z_src, ei_eeg = model.encode_eeg(batch)
        assert z_src.shape == (3, int(config['out_len']), D), tuple(z_src.shape)
        assert ei_eeg.shape == (3, D), tuple(ei_eeg.shape)
        assert not torch.isnan(z_src).any()
        results['encode_eeg_b3'] = True
        print(f'  OK encode_eeg B=3 {tuple(z_src.shape)}')

        seed = int(config.get('seed', 2026))
        z_oracle, ei_oracle, oracle_mask, h_tgt, text_mask, ei_text = model.encode_kv(
            batch, 'oracle_text', mix_seed=seed,
        )
        assert torch.equal(z_oracle, h_tgt) and torch.equal(ei_oracle, ei_text)
        assert torch.equal(oracle_mask, text_mask)
        oracle_memory, oracle_memory_mask = model.build_decoder_memory(
            z_oracle, ei_oracle, oracle_mask,
        )
        expected_extra = 1 if model.use_ei else 0
        assert oracle_memory.shape[1] == text_mask.shape[1] + expected_extra
        assert torch.equal(oracle_memory_mask[:, expected_extra:], text_mask)
        if model.use_ei:
            assert bool(oracle_memory_mask[:, 0].eq(1).all())
        results['oracle_mask_preserved'] = True
        print('  OK oracle full-target memory preserves text mask')

        inst = model._instruction_ids()
        assert len(inst) >= 1
        dec, dec_attn = model.instruction_decoder_inputs(3)
        assert dec.shape[0] == 3 and int(dec[0, 0].item()) == int(model._decoder_start_id())
        bart_bos = model._bart_bos_id()
        extra = 1 if model.family == 'encdec_bart' else 0
        assert int(dec.shape[1]) == 1 + extra + len(inst)
        if model.family == 'encdec_bart':
            assert bart_bos is not None and int(dec[0, 1].item()) == int(bart_bos)
        prompt_ids, prompt_lens, target_ids = model._tokenize_prompt_target_ids(batch)
        eos_id = int(model.tokenizer.eos_token_id)
        for row_idx, text in enumerate(batch['target text']):
            lexical_text = lexical_text_after_instruction(str(text), model.family)
            lexical = list(model.tokenizer(
                lexical_text,
                add_special_tokens=False,
                truncation=True,
                max_length=model.max_target_tokens - 1,
            )['input_ids'])
            lexical = [int(token_id) for token_id in lexical if int(token_id) != eos_id]
            assert target_ids[row_idx] == lexical + [eos_id]
            expected_prefix = dec[row_idx].tolist()[1:]
            assert prompt_ids[row_idx] == expected_prefix
            assert prompt_lens[row_idx] == len(expected_prefix)
            if model.family == 'encdec_bart':
                assert prompt_ids[row_idx][0] == int(bart_bos)
                assert lexical, 'BART lexical smoke text must produce a token'
                decoded_instruction = model.tokenizer.decode(
                    inst, skip_special_tokens=True, clean_up_tokenization_spaces=False,
                )
                decoded_boundary = model.tokenizer.decode(
                    inst + lexical[:1], skip_special_tokens=True,
                    clean_up_tokenization_spaces=False,
                )
                assert decoded_boundary.startswith(decoded_instruction + ' '), (
                    decoded_instruction, decoded_boundary,
                )
        t_star = model.first_target_ids(batch)
        bos = getattr(model.tokenizer, 'bos_token_id', None)
        if bos is not None:
            assert not bool((t_star == int(bos)).any()), t_star.tolist()
        results['real_tokenizer_decoder_order'] = True
        print(f'  OK real-tokenizer canonical order prefix={tuple(dec.shape)} t*={t_star.tolist()}')

        for dk in ('oracle_text', 'eeg'):
            model.set_data_key(dk)
            out = model.forward_e2e(batch, dk, mix_seed=seed)
            loss = out['loss']
            assert loss.ndim == 0 and torch.isfinite(loss), dk
            results[f'forward_e2e_{dk}'] = True
            print(f'  OK forward_e2e[{dk}] (TF) loss={float(loss.detach().cpu()):.4f}')
        smoke_provenance: Dict[str, Any] = {
            'resolved_path': '/smoke/corpus.df',
            'file_size_bytes': 123,
            'sha256': '0' * 64,
            'rows': 6,
            'phase_counts': {'train': 2, 'val': 2, 'test': 2},
            'required_columns': ['dataset', 'eeg', 'input text', 'mask', 'phase', 'subject', 'task', 'text uid'],
            'integrity_checks': {
                'canonical_integer_uids': True,
                'non_null_nfkc_normalized_texts': True,
                'exact_phases': True,
                'uid_to_normalized_text': True,
                'cross_phase_uid_disjoint': True,
                'cross_phase_normalized_text_disjoint': True,
                'all_rows_eeg_mask_valid': True,
            },
        }
        checkpoint_smoke = build_e2e_checkpoint_payload(
            model, 1, 1.0, 'eeg', seed, smoke_provenance,
        )
        validate_and_load_e2e_checkpoint(model, checkpoint_smoke, seed, smoke_provenance)
        metadata = checkpoint_smoke['metadata']
        assert metadata['model']['source'] == model.source
        assert metadata['model']['modelscope_id'] == model.modelscope_id
        assert len(metadata['model']['model_config_sha256']) == 64
        assert len(metadata['tokenizer']['vocab_sha256']) == 64
        assert len(metadata['tokenizer']['behavior_sha256']) == 64
        assert metadata['tokenizer']['behavior_representation'] in {
            'fast_backend_json', 'sentencepiece_proto', 'bart_ordered_bpe_ranks',
        }
        assert metadata['tokenizer']['special_token_ids']['eos_token_id'] == model.tokenizer.eos_token_id
        assert metadata['data'] == smoke_provenance
        assert set(metadata['effective_config']) == set(_EFFECTIVE_CHECKPOINT_CONFIG_KEYS)
        fingerprint_mutations = [
            ('data', 'file_size_bytes', 124),
            ('data', 'sha256', '1' * 64),
            ('model', 'model_config_sha256', '2' * 64),
            ('model', 'resolved_name_or_path', '/wrong/model'),
            ('model', '_commit_hash', 'wrong-commit'),
            ('model', 'local_weights_sha256', '3' * 64),
            ('tokenizer', 'vocab_sha256', '4' * 64),
            ('tokenizer', 'behavior_sha256', '5' * 64),
            ('tokenizer', 'behavior_representation', 'wrong-representation'),
            ('tokenizer', '_commit_hash', 'wrong-tokenizer-commit'),
            ('tokenizer', 'revision', 'wrong-tokenizer-revision'),
        ]
        for section, field, bad_value in fingerprint_mutations:
            bad_checkpoint = dict(checkpoint_smoke)
            bad_metadata = dict(metadata)
            bad_section = dict(metadata[section])
            bad_section[field] = bad_value
            bad_metadata[section] = bad_section
            bad_checkpoint['metadata'] = bad_metadata
            try:
                validate_and_load_e2e_checkpoint(
                    model, bad_checkpoint, seed, smoke_provenance,
                )
            except AssertionError:
                pass
            else:
                raise AssertionError(f'checkpoint fingerprint mutation must fail: {section}.{field}')
        results['strict_checkpoint_provenance'] = True

        oracle = model.decoder_states_at_minus1(batch, 'oracle_text', mix_seed=seed)
        eeg = model.decoder_states_at_minus1(batch, 'eeg', mix_seed=seed)
        assert len(oracle['hidden_m1']) == n_blocks + 1
        assert oracle['hidden_m1'][0].shape == (3, D)
        assert oracle['logits_hf'].shape == (3, model.vocab_size())
        assert torch.equal(oracle['decoder_input_ids'], eeg['decoder_input_ids'])
        n_tr = len(oracle['hidden_m1']) - 1
        assert n_tr >= 1
        lens = build_tuned_lens(model, n_tr)
        lens.to(device)
        lens.eval()
        assert not any(p.requires_grad for p in lens.unembed.parameters())
        idx_mid = 1 if n_tr >= 2 else 0
        h_mid = oracle['hidden_m1'][idx_mid]
        h_lens = hidden_for_lens(lens, h_mid)
        with torch.no_grad():
            y_id = lens.forward(h_lens, idx_mid)
            y_u = lens.unembed(h_lens)
        assert y_id.shape == y_u.shape == oracle['logits_hf'].shape
        assert torch.allclose(y_id.float(), y_u.float(), atol=1e-4, rtol=1e-4), (
            'identity-init translator must match Unembed'
        )
        matched_form = verify_unembed_against_official(
            model, lens, oracle, stage=f'{key} smoke before lens step',
        )
        results['tuned_lens_identity_unembed'] = True
        results['unembed_matches_official'] = matched_form in ('raw', 'final_normalized')
        print(f'  OK tuned-lens identity init translators={n_tr} form={matched_form}')

        params = [p for p in lens.layer_translators.parameters() if p.requires_grad]
        opt_l = AdamW(params, lr=1e-3)
        opt_l.zero_grad(set_to_none=True)
        log_p = oracle['logits_hf'].float().log_softmax(dim=-1).detach()
        loss_kl = kl_tuned_lens_layer(
            lens, oracle['hidden_m1'][0].detach(), 0, log_p,
        )
        loss_kl.backward()
        assert all(p.grad is None or bool(torch.isfinite(p.grad).all()) for p in params)
        opt_l.step()
        assert torch.isfinite(loss_kl)
        _ = verify_unembed_against_official(
            model, lens, oracle, stage=f'{key} smoke after lens step',
        )
        results['tuned_lens_kl_step'] = True
        print(f'  OK tuned-lens one KL step loss={float(loss_kl.detach().cpu()):.4f}')

        logits_l = unembed_block(
            model, oracle['hidden_m1'], oracle['logits_hf'], n_blocks, n_blocks, lens,
        )
        logits_e = unembed_block(
            model, eeg['hidden_m1'], eeg['logits_hf'], n_blocks, n_blocks, lens,
        )
        assert torch.equal(logits_l, oracle['logits_hf'])
        assert torch.equal(logits_e, eeg['logits_hf'])
        t_star = oracle['t_star']
        delta = logits_l.gather(-1, t_star.unsqueeze(-1)) - logits_e.gather(-1, t_star.unsqueeze(-1))
        assert delta.shape == (3, 1)
        rho = spearman_rowwise(oracle['hidden_m1'][-1], eeg['hidden_m1'][-1])
        assert rho.shape == (3,)
        results['two_path_hidden'] = True
        print(f'  OK two-path Delta={delta.squeeze(-1).tolist()} rho={rho.tolist()}')

        batch_probe = _make_dummy_probe_batch(device, batch_size=9)
        oracle_probe = model.decoder_states_at_minus1(
            batch_probe, 'oracle_text', mix_seed=seed,
        )
        x_all = oracle_probe['hidden_m1'][-1].detach().cpu()
        y_all = oracle_probe['t_star'].detach().cpu()
        assert x_all.shape[0] == 9 and y_all.shape[0] == 9
        assert x_all.ndim == 2 and y_all.ndim == 1
        cfg_p = dict(config)
        cfg_p['_probe_vocab'] = model.vocab_size()
        cfg_p['probe_epochs'] = 1
        cfg_p['probe_patience'] = 1
        cfg_p['probe_batch_size'] = 3
        m_probe = train_linear_probe(
            (x_all[:3], y_all[:3]),
            (x_all[3:6], y_all[3:6]),
            (x_all[6:], y_all[6:]),
            device=torch.device('cpu'),
            config=cfg_p,
            title=f'smoke probe {key}',
            save_path=None,
        )
        assert math.isfinite(float(m_probe['test_ce']))
        results['linear_probe_dummy'] = True
        print(f'  OK linear_probe_dummy ce={m_probe["test_ce"]:.4f}')

        assert all(results.values()), f'{key} smoke incomplete: {results}'
        print(f'  [SMOKE LLM] {key} PASSED')
        return results
    finally:
        if model is not None:
            model.unload_llm()
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _print_disk_usage(config.get('model_cache_dir'))


def run_all_llm_smokes(
    llm_configs: List[Dict[str, Any]],
    config: Dict[str, Any],
    device: torch.device,
) -> Dict[str, Dict[str, bool]]:
    """Validate selection, then smoke every requested LLM. Fail-fast."""
    configs = select_llm_configs(llm_configs, config)
    assert len(configs) >= 1, 'No LLM configs to smoke'
    print(f'\n========== Per-LLM smoke tests ({len(configs)} models) ==========')
    all_results: Dict[str, Dict[str, bool]] = {}
    for llm_cfg in configs:
        key = str(llm_cfg['key'])
        try:
            all_results[key] = run_llm_smoke(llm_cfg, config, device)
        except Exception as exc:
            print(f'[SMOKE FAIL] {key}: {exc}')
            raise RuntimeError(f'LLM smoke failed for {key}: {exc}') from exc
    print('\nAll per-LLM smokes passed:', list(all_results.keys()))
    return all_results


print('========== Encoder smoke ==========')
_ = run_smoke_tests()


In [ ]:
# =============================================================
# Main: per-LLM smokes -> train eeg E2E -> tuned lens -> analyses (a)-(d)
# =============================================================

def run_one_llm(
    llm_cfg: Dict[str, Any],
    dm: 'PhaseDataModule',
    config: Dict[str, Any],
    device: torch.device,
) -> None:
    """Independently train the EEG path, then run oracle-vs-EEG analyses.

    Args:
        llm_cfg: One entry from ``LLM_CONFIGS``.
        dm: Prepared ``PhaseDataModule``.
        config: Notebook CONFIG (must keep ``data_keys == ['eeg']``).
        device: Compute device for ``ProbeSystem``.
    """
    assert list(config['data_keys']) == ['eeg']
    key = llm_cfg['key']
    print(f'\n========== {key} ({llm_cfg["repo_id"]}) ==========')

    bs = int(llm_cfg.get('batch_size', config['batch_size']))
    dm.batch_size = bs
    train_loader = dm.train_dataloader(seed=int(config['seed']))
    val_loader = dm.val_dataloader(seed=int(config['seed']))
    out_dir = Path(config['output_dir']) / str(key)
    out_dir.mkdir(parents=True, exist_ok=True)

    set_seed(int(config['seed']))
    model = ProbeSystem(
        config=config,
        llm_key=key,
        repo_id=llm_cfg['repo_id'],
        family=llm_cfg['family'],
        device=device,
        source=str(llm_cfg.get('source', 'hf_mirror')),
        modelscope_id=llm_cfg.get('modelscope_id'),
    )
    model.to(device)

    print(f'\n----- {key} / eeg E2E train -----')
    model.reset_trainable()
    model.set_data_key('eeg')
    _ = train_e2e(
        model, train_loader, val_loader, config, out_dir, 'eeg', dm.provenance,
    )
    show_qualitative_nn(
        model, val_loader, data_key='eeg', n=int(config.get('qualitative_n', 5)),
        seed=int(config['seed']),
    )

    print(f'\n----- {key} / tuned lens train (oracle full-target memory, pos -1) -----')
    lens = train_tuned_lens(model, train_loader, val_loader, config, out_dir)
    run_faithfulness_analyses(model, dm, config, out_dir, str(key), lens=lens)
    del lens

    model.unload_llm()
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _print_disk_usage(config.get('model_cache_dir'))


set_seed(int(CONFIG['seed']))

configs = select_llm_configs(LLM_CONFIGS, CONFIG)
LLM_SMOKE_RESULTS = run_all_llm_smokes(LLM_CONFIGS, CONFIG, DEVICE)

if CONFIG.get('run_smoke_only', False):
    print('run_smoke_only=True — skipping full training / analysis loop.')
else:
    merged = load_merged_corpus(CONFIG['data_path'])
    corpus_provenance = merged.attrs['eeg_faith_provenance']
    dm = PhaseDataModule(
        merged,
        provenance=corpus_provenance,
        batch_size=int(CONFIG['batch_size']),
        num_workers=int(CONFIG['num_workers']),
    )
    dm.setup()

    for llm_cfg in configs:
        run_one_llm(llm_cfg, dm, CONFIG, DEVICE)

    print('Done: every requested LLM completed successfully.')
